# El componente visual, enchufado y medido

### Módulo 4 · El tutor recibe la foto de un procedimiento y señala el paso erróneo

**Proyecto:** Tutor inteligente de matemáticas · Comparación de arquitecturas
mediante fine-tuning
**Curso:** SI4006 · Tópicos Especiales y Aplicaciones en IA · Universidad EAFIT
**Notebook base:** `S04_Lab_Fine_tuning_Qwen.ipynb`

---

## 1 · Qué hace este notebook

**La pregunta del usuario que necesita ojos:** *"¿En qué paso me equivoqué?"*

El procedimiento está escrito a mano en el cuaderno. Hoy el tutor solo lee texto,
así que el estudiante tendría que transcribirlo — y al transcribir corrige el
error sin darse cuenta, con lo que el sistema nunca llega a ver el fallo que debía
encontrar.

**El patrón es `input`.** Conviene decir por qué se descartaron los otros dos:

| Patrón | Decisión | Razón |
|---|---|---|
| **input** | **Elegido** | La pregunta necesita el *contenido* de la hoja como pasos de texto, y el sistema de M3 ya consume texto: no hay que reescribirlo |
| tool | Descartado | Un clasificador de imagen devuelve una clase, no una secuencia de pasos: no alcanza para localizar uno |
| output | Descartado | Dibujar el paso corregido no cambia ninguna decisión del estudiante, que lo necesita explicado |

**La decisión que ordena el experimento: separar leer de diagnosticar.** Son dos
etapas con dos modos de fallo distintos, y una sola cifra los confunde. Por eso
cada caso registra además si la transcripción fue fiel: así un fallo se atribuye a
*los ojos* o al *razonamiento*, no a "el sistema".

> **Este notebook no entrena nada.** Es inferencia y evaluación.

## 2 · Los dos conjuntos, que miden cosas distintas

Esto condiciona cómo se lee todo lo que viene después.

| Conjunto | n | Qué mide | Qué significa una cifra alta |
|---|---|---|---|
| **Eval set de M4** | 12 casos **con imagen** | Una tarea **nueva**: localizar el paso erróneo en una hoja fotografiada | El sistema puede hacer algo que antes no podía |
| **Eval set de M2** | 21 casos **sin imagen** | La tarea **de siempre**: resolver un problema planteado en texto | Anteponer la visión no rompió lo que ya funcionaba |

Son **conjuntos distintos, con criterios distintos**. Se reportan los dos, cada
uno etiquetado, pero **no se restan**: no existe "la mejora de M3 a M4" como un
número único, porque M3 no podía recibir imágenes.

### El orden de la corrida, y por qué

| Fase | Qué ocupa la GPU | Qué queda en disco |
|---|---|---|
| **1 · leer** | Qwen2-VL-2B (~4.4 GB en fp16) | `m4/lecturas_m4.json` |
| *liberar* | — | |
| **2 · diagnosticar** | tutor de M3 (~3.1 GB) + embeddings + juez | `m4/scorecard_m4.csv` |

Un VLM de 2B y el tutor de 1.5B con su juez no caben cómodos en una T4 de 16 GB.
Y como la transcripción queda guardada, la fase 2 se puede repetir sin volver a
leer las fotos, que es la parte lenta.

---
## 0 · Preparación del entorno

> **Activen una GPU.** Sirve cualquiera con CUDA —T4, G4, L4, A100, RTX—; el
> notebook imprime la que detecte y no exige ningún modelo concreto. Carga dos
> modelos grandes, pero **en momentos distintos**: la celda de liberación que va
> entre ellos no es opcional, y con menos de 14 GB es imprescindible.

In [1]:
%pip install -q "transformers>=4.45" "peft>=0.12" "accelerate>=0.33" \
    sentence-transformers chromadb rank_bm25 "scikit-learn>=1.3" sympy pandas \
    pillow qwen-vl-utils
%pip uninstall -y -q torchao
print("Listo. Si Colab pide reiniciar la sesión, reinícienla y sigan desde aquí.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
Listo. Si Colab pide reiniciar la sesión, reinícienla y sigan desde aquí.


In [2]:
import os, random, re
import numpy as np
import pandas as pd
import torch
import transformers

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
transformers.set_seed(SEMILLA)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers {transformers.__version__} | torch {torch.__version__} | {DEVICE}")

# Sirve CUALQUIER GPU CUDA: T4, G4, L4, A100, RTX… No se comprueba el modelo,
# solo que haya CUDA, y se imprime el dispositivo detectado. La corrida del 22-10
# usó una RTX PRO 6000 de 102 GB; exigir "Tesla T4" habría sido un falso bloqueo.
if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    VRAM_GB = props.total_memory / 1e9
    print(f"GPU detectada: {props.name} ({VRAM_GB:.0f} GB)")
    if VRAM_GB < 14:
        print(f"  Aviso: con {VRAM_GB:.0f} GB conviene no tener los dos modelos a la vez. "
              "La celda que libera el VLM (5.2) es obligatoria.")
else:
    print("Sin GPU: el notebook corre, pero leer 11 imágenes en CPU tarda mucho. "
          "Activen un entorno de ejecución con GPU (cualquiera con CUDA sirve).")

transformers 5.20.2 | torch 2.11.0+cu130 | cuda
GPU detectada: Tesla T4 (16 GB)


### Persistencia entre notebooks

Los notebooks 3 y 5 **leen carpetas que producen los notebooks 1, 2 y 4**:

```
1 · Qwen    -> adaptadores/qwen-lora/      ─┐
2 · BERT    -> modelos/bert-clasificador/  ─┤-> el notebook 3 las lee
4 · FLAN-T5 -> adaptadores/flan-t5-lora/    │
1,2,3,4     -> resultados/*.json           ─┴-> el notebook 5 los lee
```

En Colab, `/content` se borra al desconectar el runtime, así que ese trabajo se
perdería entre sesiones. Montando Google Drive y trabajando desde una carpeta
suya, los artefactos sobreviven y cada notebook se puede ejecutar el día que se
pueda.

Con `USAR_DRIVE = False` todo queda en `/content`, lo cual es válido si
ejecutan los notebooks 1, 2 y 3 seguidos sin desconectar.

Fuera de Colab la celda no hace nada: el directorio de trabajo se queda como
está.

> **Los checkpoints intermedios nunca van a Drive.** El `Trainer` guarda en
> `output_dir` el modelo *más el estado del optimizador* en cada época. Para el
> notebook 2, que hace fine-tuning completo de BETO, eso son varios GB que
> además se escribirían por red. Como son desechables —lo que importa es el
> modelo final—, se mandan siempre al disco local del runtime mediante
> `DIR_CHECKPOINTS`.

In [3]:
import os
from pathlib import Path

USAR_DRIVE    = True
CARPETA_DRIVE = "/content/drive/MyDrive/ProyectoIA"

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CARPETA_DRIVE).mkdir(parents=True, exist_ok=True)
    os.chdir(CARPETA_DRIVE)

# Checkpoints del Trainer: grandes y desechables -> siempre en disco local.
DIR_CHECKPOINTS = "/content/salidas" if EN_COLAB else "salidas"

print(f"En Colab           : {EN_COLAB}")
print(f"Directorio de trabajo: {Path.cwd()}")
print(f"Checkpoints en     : {DIR_CHECKPOINTS}")

Mounted at /content/drive
En Colab           : True
Directorio de trabajo: /content/drive/MyDrive/ProyectoIA
Checkpoints en     : /content/salidas


---
## 3 · Los datos

Los eval sets viajan dentro del notebook como blob comprimido y se verifican por
SHA-256. Es la salvaguarda que faltó en S08: aquella corrida usó la copia de Drive
con las claves viejas de `adv-02` y hubo que recalcular el criterio después.

### 3.1 · El módulo de visión

`scripts/agente/vision.py`, pegado tal cual, igual que el resto del núcleo. Aquí
está lo que puede equivocarse **en silencio**: el alias de `vis-12`, el inventario
de fotos que faltan, el parser tolerante del JSON, el formato con el que el tutor
ve la transcripción y la atribución del fallo a la lectura o al diagnóstico.

Está en un módulo y no en una celda porque se prueba con
`python -m pytest scripts/tests/test_m4_vision.py`, **sin GPU y sin fotos**. Lo
que no se puede probar así — la pasada del VLM y el redimensionado — se queda en
el notebook.

In [4]:
"""La lógica de M4 que no necesita GPU ni fotos, para poder probarla.

El notebook de M4 pega este archivo tal cual, igual que hace con `registro.py` o
`nucleo.py`. Fuera queda solo lo que no se puede probar sin el modelo cargado: la
pasada del VLM y el redimensionado de la imagen.

Dentro queda lo que puede equivocarse **en silencio**. La corrida del 22-10-2026
(archivada en `m4/corridas/corrida-01_2026-10-22/`) dio 0/11 y las cuatro causas
fueron de medición, no del modelo:

  1. **El criterio contaba menciones de pasos dentro de la explicación.** Todos
     los casos fallaron con "cayó en la trampa: 'paso 1'" porque la respuesta
     empieza repasando el procedimiento. `vis-05` llegó a decir "Paso 2:
     Incorrecto" —la respuesta esperada— y aun así se contó como fallo. Ahora el
     tutor tiene que emitir una línea `Diagnóstico:` y es **esa línea** la que se
     califica; la explicación se evalúa aparte, como señal, nunca como acierto.

  2. **La numeración que veía el tutor no coincidía con la del eval set.** El eval
     set cuenta la línea "Respuesta final" como un paso más; el VLM la devuelve en
     el campo `respuesta_final`. Para `vis-03`, cuyo error está en el paso 3, el
     tutor solo veía dos pasos: la respuesta correcta era inalcanzable. Ahora
     `pasos_numerados` reconstruye la numeración del eval set.

  3. **El inventario emparejaba por nombre exacto.** `vis-10.jpeg` no satisfacía
     `vis-10.jpg` y el caso quedó sin evaluar. Ahora se empareja por nombre base.

  4. **`lectura_fiel` exigía el mismo número de pasos.** `vis-01` y `vis-03` se
     atribuyeron a fallo de lectura cuando la transcripción era fiel y lo que
     cambiaba era dónde había puesto el VLM la respuesta final.

**Lo que no cambia:** ningún valor esperado del eval set. Los pasos, el
`paso_error` y las claves siguen siendo los que se escribieron antes de ver una
sola respuesta del modelo.

Solo biblioteca estándar: las pruebas corren sin instalar nada.
"""

from __future__ import annotations

import json
import re
import unicodedata

# `vis-12` reutiliza a propósito la hoja de `vis-07` con otra pregunta: se pide un
# dato que no está escrito en ella. El eval set guarda el nombre lógico y el alias
# se resuelve aquí, así que hay 12 casos y 11 fotos, y el eval set no se toca.
ALIAS_IMAGENES = {"vis-12.jpg": "vis-07.jpg"}

# Extensiones que cuentan como "la foto de este caso". El eval set escribe `.jpg`,
# pero una cámara o una conversión producen `.jpeg` o `.png` sin que eso cambie
# nada del caso: lo que identifica a la foto es el nombre base.
EXTENSIONES = (".jpg", ".jpeg", ".png")


def _norm(texto: str) -> str:
    """Minúsculas y sin tildes, igual que el criterio de M2 y M3."""
    t = unicodedata.normalize("NFKD", (texto or "").lower())
    return "".join(c for c in t if not unicodedata.combining(c))


# --------------------------------------------------------------------------
# Inventario de imágenes
# --------------------------------------------------------------------------

def archivo_de(caso: dict) -> str:
    """El nombre lógico de la foto de este caso, resuelto el alias."""
    return ALIAS_IMAGENES.get(caso["imagen"], caso["imagen"])


def base_de(nombre: str) -> str:
    """`vis-10.jpeg` -> `vis-10`. El nombre base es lo que identifica la foto."""
    for ext in EXTENSIONES:
        if nombre.lower().endswith(ext):
            return nombre[: -len(ext)]
    return nombre


class ImagenAmbigua(Exception):
    """Dos archivos con el mismo nombre base, por ejemplo `vis-10.jpg` y
    `vis-10.jpeg`.

    No se elige uno por orden alfabético: si las dos fotos son distintas, la
    corrida mediría una hoja y el informe hablaría de la otra, y nada avisaría.
    Hay que borrar la que no corresponda.
    """


def inventario(casos, presentes) -> tuple[dict, dict, set]:
    """-> (pedidas, faltan, sobran).

    `pedidas` va del nombre lógico al par (archivo real en disco, casos que lo
    usan), así que una foto compartida aparece una sola vez: es lo que mantiene
    separados el recuento de fotos y el de casos.

    Empareja por nombre base, de modo que `vis-10.jpeg` satisface `vis-10.jpg`.
    Lanza `ImagenAmbigua` si dos archivos comparten el nombre base.
    """
    por_base: dict = {}
    for archivo in sorted(presentes):
        if not archivo.lower().endswith(EXTENSIONES):
            continue
        por_base.setdefault(base_de(archivo), []).append(archivo)

    ambiguas = {b: v for b, v in por_base.items() if len(v) > 1}
    if ambiguas:
        detalle = "; ".join(f"{b}: {', '.join(v)}" for b, v in sorted(ambiguas.items()))
        raise ImagenAmbigua(
            f"hay más de una imagen con el mismo nombre base ({detalle}). "
            "Dejen una sola por caso y vuelvan a ejecutar: elegir una por orden "
            "mediría una hoja y el informe hablaría de la otra.")

    pedidas: dict = {}
    for c in casos:
        logico = archivo_de(c)
        real = (por_base.get(base_de(logico)) or [None])[0]
        entrada = pedidas.setdefault(logico, {"archivo": real, "casos": []})
        entrada["casos"].append(c["id"])

    faltan = {k: v for k, v in pedidas.items() if v["archivo"] is None}
    usados = {v["archivo"] for v in pedidas.values() if v["archivo"]}
    sobran = {a for lista in por_base.values() for a in lista} - usados
    return pedidas, faltan, sobran


# --------------------------------------------------------------------------
# El JSON que devuelve el VLM
# --------------------------------------------------------------------------

def json_de(texto: str):
    """El primer objeto JSON del texto, o None.

    Un modelo de 2B devuelve el JSON envuelto en prosa o en una valla de código
    (pasó con `vis-06`): `json.loads` del texto completo fallaría y se perdería
    una lectura buena.
    """
    i, j = (texto or "").find("{"), (texto or "").rfind("}")
    if i == -1 or j <= i:
        return None
    try:
        return json.loads(texto[i:j + 1])
    except json.JSONDecodeError:
        return None


_PREFIJO_PASO = re.compile(r"^\s*paso\s*\d+\s*[:.\-]\s*", re.IGNORECASE)
_MENCIONA_FINAL = re.compile(r"respuesta\s*final", re.IGNORECASE)


def pasos_numerados(lectura) -> list:
    """Los pasos de la hoja con la MISMA numeración que usa el eval set.

    El eval set cuenta la línea "Respuesta final: ..." como un paso más —el error
    de `vis-03` está justamente ahí—. El VLM a veces la devuelve dentro de
    `pasos` y a veces en el campo `respuesta_final`. Sin unificarlo, el paso que
    hay que señalar no existe en lo que el tutor ve, y el caso es imposible de
    acertar por construcción.

    También quita el "Paso N:" que el VLM escribe a veces dentro del texto, para
    no numerar dos veces.
    """
    js = (lectura or {}).get("json") or {}
    pasos = [_PREFIJO_PASO.sub("", str(p)).strip() for p in (js.get("pasos") or [])]
    final = js.get("respuesta_final")
    # Solo se añade la respuesta final si HAY pasos. Con `pasos: []` —la lista de
    # mercado de `vis-11`— añadirla fabricaba un "Paso 1: Respuesta final: Arroz"
    # y al tutor se le presentaba un procedimiento de un paso donde la hoja no
    # tenía ninguno, lo que hace casi imposible concluir "fuera de dominio".
    if pasos and final is not None and not any(_MENCIONA_FINAL.search(p) for p in pasos):
        pasos.append(f"Respuesta final: {final}")
    return pasos


# --------------------------------------------------------------------------
# ¿Es fiable esta lectura?
# --------------------------------------------------------------------------

_PAT_NUMERO = re.compile(r"-?\d+(?:[.,]\d+)?(?:/\d+)?")


def numeros_de(texto: str) -> list:
    """Los números de un texto, normalizados. La huella de una transcripción."""
    return [n.replace(",", ".") for n in _PAT_NUMERO.findall(texto or "")]


def huella_lectura(lectura) -> tuple:
    """Lo que tiene que coincidir entre dos lecturas de la MISMA foto: cuántos
    pasos y qué números. La redacción puede variar; los números, no."""
    js = (lectura or {}).get("json") or {}
    if js.get("ilegible"):
        return ("ilegible",)
    pasos = js.get("pasos") or []
    texto = " ".join(map(str, pasos)) + " " + str(js.get("respuesta_final", ""))
    return (len(pasos), tuple(numeros_de(texto)))


def lecturas_consistentes(a, b) -> bool:
    """¿Dos lecturas de la misma foto dicen lo mismo?

    **Necesaria pero NO suficiente, y hay prueba.** En la corrida del 24-10 esta
    señal falló exactamente donde tenía que servir: ante la foto desenfocada de
    `vis-10` el VLM inventó `24 x 13 = 306` (la hoja dice 302) y lo inventó **las
    dos veces**. La invención no era ruido de muestreo: estaba anclada en el
    prior del modelo —la hoja es una versión borrosa de un ejercicio de 24 × 13—
    y por eso es estable.

    Se conserva porque es barata y sí caza las lecturas inestables, pero ya no se
    usa como única defensa.
    """
    return huella_lectura(a) == huella_lectura(b)


def atipicos_por_mad(valores: dict, k: float = 3.0) -> set:
    """Las claves cuyo valor queda muy por debajo del resto del conjunto.

    Mediana y desviación absoluta mediana, el criterio robusto de siempre:
    `valor < mediana - k · MAD`. Es **relativo al propio conjunto**, no un umbral
    absoluto elegido a mano, y eso es deliberado: cualquier constante calibrada
    con estas 11 fotos sería un umbral ajustado al conjunto de prueba.

    `k = 3` es la convención estadística habitual, no un valor afinado para que
    `vis-10` caiga del lado correcto. **Si separa o no esa foto es algo que hay
    que medir, y no debe tocarse para que salga.**
    """
    if len(valores) < 4:          # con tres o menos no hay "resto del conjunto"
        return set()
    import statistics
    xs = sorted(valores.values())
    mediana = statistics.median(xs)
    mad = statistics.median([abs(x - mediana) for x in xs]) * 1.4826
    if mad == 0:
        return set()
    return {clave for clave, v in valores.items() if v < mediana - k * mad}


# Dos ideas tentadoras que NO se usan, y por qué:
#
#   · **Comprobar la aritmética de la transcripción.** Suena bien y es inservible
#     aquí: las hojas llevan cuentas mal A PROPÓSITO. En `vis-10` el invento es
#     `24 × 13 = 306` y lo que la hoja dice es `24 × 13 = 302`; las dos son
#     aritméticamente falsas, así que la comprobación no distingue la lectura de
#     la invención. Peor: marcaría como dudosas justo las hojas con error, que son
#     las que el módulo existe para medir.
#
#   · **Un umbral de nitidez sobre la imagen** (varianza del laplaciano y
#     similares). Habría que calibrarlo, y calibrarlo con estas 11 fotos es
#     ajustarlo al conjunto de prueba. Si algún día hay un conjunto de calibración
#     aparte, vuelve a ser una opción razonable.


def lectura_no_fiable(lectura) -> tuple:
    """-> (no_fiable, motivo). Si es no fiable, el tutor NO debe tratarla como un
    procedimiento legible: se le dice que la hoja no se pudo leer.

    Cuatro señales de naturaleza distinta, porque la del 24-10 demostró que una
    sola no basta. Basta que **una** se dispare: ante la duda no se le entrega al
    tutor un procedimiento que puede ser inventado.

      1. la transcripción no salió estructurada;
      2. el propio VLM declaró la hoja ilegible al transcribir;
      3. preguntado **aparte** si la hoja se lee —sin transcribir, otra pregunta y
         otro encuadre, para no arrastrar el prior de la transcripción—, dijo que
         no (`legible_declarada = False`);
      4. la confianza del modelo en los **números** que transcribió es atípica
         frente a las demás fotos del conjunto (`confianza_atipica`). Un número
         leído de una cifra nítida sale con más confianza que uno rellenado.

    La señal de consistencia entre dos generaciones queda como quinta, degradada:
    sirve, pero no se le confía la decisión.

    Nunca acredita un caso: una lectura no fiable no vuelve correcta ninguna
    respuesta, solo evita juzgar al tutor sobre un invento del VLM.
    """
    if not lectura or not lectura.get("ok"):
        return True, "la transcripción no salió estructurada"
    js = lectura.get("json") or {}
    if js.get("ilegible"):
        return True, js.get("motivo") or "el modelo la declaró ilegible"
    if lectura.get("legible_declarada") is False:
        return True, ("preguntado aparte, el modelo dijo que la hoja no se lee: "
                      + (lectura.get("motivo_legibilidad") or "sin motivo"))
    if lectura.get("confianza_atipica") is True:
        return True, ("la confianza del modelo en los números transcritos es "
                      "atípicamente baja frente a las demás fotos")
    if lectura.get("consistente") is False:
        return True, ("dos lecturas de la misma foto no coinciden: la transcripción "
                      "no es de fiar")
    return False, ""


# --------------------------------------------------------------------------
# El diagnóstico: una conclusión explícita, separada de la explicación
# --------------------------------------------------------------------------

# Las cinco categorías, ajustadas al esquema real del eval set:
#   paso N            -> bloques `localizacion` y `compensado` (tienen `paso_error`)
#   sin error         -> bloque `sin_error`
#   ilegible          -> adversarial: la hoja no se puede leer (vis-10)
#   fuera de dominio  -> adversarial: la hoja no es matemáticas (vis-11)
#   dato no disponible-> adversarial: se pide algo que no está escrito (vis-12)
CATEGORIAS = ("paso", "sin error", "ilegible", "fuera de dominio", "dato no disponible")

# Las tres formas de abstenerse. Cualquiera de ellas es "no inventó nada", que es
# la conducta que el bloque adversarial busca; acertar la categoría exacta es más
# exigente y se reporta por separado.
ABSTENCIONES = ("ilegible", "fuera de dominio", "dato no disponible")

# Qué abstención corresponde a cada caso adversarial. Va por id porque la
# distinción está en el enunciado del caso, no en un campo del eval set; la
# prueba `test_todo_adversarial_tiene_categoria` falla si se añade un caso
# adversarial y se olvida ponerlo aquí.
DIAGNOSTICO_ADVERSARIAL = {
    "vis-10": "ilegible",             # foto movida y desenfocada a propósito
    "vis-11": "fuera de dominio",     # una lista de mercado, no un procedimiento
    "vis-12": "dato no disponible",   # se pide una nota que no está en la hoja
}

# Sinónimos que se aceptan como la misma categoría. Un modelo pequeño no va a
# escribir la etiqueta exacta siempre, y penalizarlo por el sinónimo mediría
# obediencia al formato en vez de criterio.
_SINONIMOS = {
    "sin error": ("sin error", "sin errores", "no hay error", "no hay errores",
                  "correcto", "todo correcto", "ninguno"),
    "ilegible": ("ilegible", "no se puede leer", "no legible", "borrosa", "no se lee"),
    "fuera de dominio": ("fuera de dominio", "no es matematicas", "no es un procedimiento",
                         "no es un ejercicio", "fuera de alcance"),
    "dato no disponible": ("dato no disponible", "dato no escrito", "no aparece",
                           "no esta en la hoja", "no disponible", "no tengo esa informacion"),
}

_PAT_DIAGNOSTICO = re.compile(r"diagn[oó]stico\s*:\s*(.+?)\s*(?:\n|$)", re.IGNORECASE)
# Acepta "Paso 2", "paso n.º 2" (que `_norm` deja en "paso n.o 2") y
# "paso número 2". Un modelo pequeño escribe el ordinal de varias formas y
# penalizarlo por eso mediría obediencia al formato, no criterio.
_PAT_PASO = re.compile(
    r"paso\s*(?:(?:n[.\u00ba\u00b0o]*|numero|num\.?|#)\s*)?(\d+)", re.IGNORECASE)


def diagnostico_esperado(caso: dict) -> str:
    """La conclusión que este caso espera, derivada del eval set tal como está.

    No se toca ningún valor esperado: `paso_error` y `bloque` ya estaban escritos
    antes de ver una respuesta del modelo.
    """
    if caso.get("paso_error") is not None:
        return f"paso {caso['paso_error']}"
    if caso.get("bloque") == "sin_error":
        return "sin error"
    if caso.get("bloque") == "adversarial":
        try:
            return DIAGNOSTICO_ADVERSARIAL[caso["id"]]
        except KeyError:
            raise KeyError(
                f"el caso adversarial {caso['id']!r} no tiene categoría en "
                "DIAGNOSTICO_ADVERSARIAL: añádanla antes de evaluarlo") from None
    raise ValueError(f"no sé qué espera el caso {caso.get('id')!r}")


def extraer_diagnostico(respuesta: str) -> str | None:
    """La conclusión de la línea `Diagnóstico:`, normalizada. None si no la hay.

    **Solo mira esa línea.** Si el tutor no la emite, el caso no se cuenta como
    acierto aunque la explicación acabe diciendo lo correcto: una conclusión que
    hay que adivinar no sirve a un estudiante, y un criterio que la adivina mide
    la suerte del parser.
    """
    m = _PAT_DIAGNOSTICO.search(respuesta or "")
    if not m:
        return None
    crudo = _norm(m.group(1)).strip(" .;")
    paso = _PAT_PASO.search(crudo)
    if paso:
        return f"paso {int(paso.group(1))}"
    for canonica, variantes in _SINONIMOS.items():
        if any(v in crudo for v in variantes):
            return canonica
    return crudo or None


def diagnostico_valido(conclusion) -> bool:
    """¿La conclusión es una de las cinco categorías, y no texto libre?

    `vis-04` emitió "Diagnóstico: equivoco el producto 1248…": una línea que
    señalaba el error sin nombrar el paso. Eso no es una conclusión válida, pero
    tampoco es una invención, y contarlo como alucinación exageraba el fallo.
    """
    if not conclusion:
        return False
    return conclusion.startswith("paso ") or conclusion in ("sin error",) + ABSTENCIONES


def pasos_mencionados(texto: str) -> set:
    """Los números de paso que menciona un texto. Señal, nunca criterio."""
    return {int(n) for n in _PAT_PASO.findall(texto or "")}


def _sin_la_linea_de_diagnostico(respuesta: str) -> str:
    return _PAT_DIAGNOSTICO.sub("", respuesta or "")


def calificar_m4(caso: dict, respuesta: str) -> dict:
    """Califica un caso de M4 leyendo la conclusión, no la explicación.

    Devuelve señales independientes, igual que `evaluar_caso` en M2 y M3:

      acierto       la conclusión coincide con la que el caso espera
      abstuvo       concluyó una de las tres formas de "no puedo / no está"
      abstuvo_bien  se abstuvo **y** era un caso en que había que abstenerse
      alucino       concluyó una categoría válida, concreta y equivocada
      sin_conclusion no emitió la línea `Diagnóstico:`
      conclusion_invalida  la emitió, pero en texto libre que no es ninguna de
                           las cinco categorías

    Y, por separado, sobre la explicación —que se conserva y se evalúa, pero no
    decide el acierto—:

      explicacion_pasos   qué pasos menciona la explicación
      explicacion_justifica  si la explicación habla del paso que concluyó (o no
                             menciona ninguno). No es un juicio de calidad: dice
                             si la conclusión viene argumentada en el texto.
    """
    esperado = diagnostico_esperado(caso)
    obtenido = extraer_diagnostico(respuesta)
    explicacion = _sin_la_linea_de_diagnostico(respuesta)
    mencionados = pasos_mencionados(explicacion)

    valida = diagnostico_valido(obtenido)
    acierto = valida and obtenido == esperado
    abstuvo = obtenido in ABSTENCIONES
    debia_abstenerse = esperado in ABSTENCIONES

    if obtenido is None:
        motivo = "no emitió la línea 'Diagnóstico:'"
    elif not valida:
        motivo = f"conclusión no válida: {obtenido[:48]!r}"
    elif acierto:
        motivo = f"concluyó {obtenido!r}"
    else:
        motivo = f"esperaba {esperado!r} y concluyó {obtenido!r}"

    paso_concluido = (int(obtenido.split()[1]) if obtenido and obtenido.startswith("paso ")
                      else None)
    return {
        "acierto": acierto,
        "conclusion": obtenido,
        "esperado": esperado,
        "abstuvo": abstuvo,
        "abstuvo_bien": abstuvo and debia_abstenerse,
        # Alucinar es concluir algo CONCRETO y equivocado. No emitir conclusión, o
        # emitirla en texto libre, es un fallo de formato y se cuenta aparte: son
        # problemas distintos y se arreglan de forma distinta.
        "alucino": valida and (not acierto) and not abstuvo,
        "sin_conclusion": obtenido is None,
        "conclusion_invalida": (obtenido is not None) and not valida,
        "motivo": motivo,
        "explicacion_pasos": sorted(mencionados),
        "explicacion_justifica": (paso_concluido in mencionados if paso_concluido
                                  else not mencionados),
    }


# --------------------------------------------------------------------------
# La puerta del ensayo de desarrollo
# --------------------------------------------------------------------------

class PuertaNoSuperada(RuntimeError):
    """El ensayo de formato sobre los casos `dev-*` no se superó, o no se ejecutó.

    Por qué es una excepción y no un aviso: en la corrida del 24-10 el ensayo dio
    **0/3** y la celda imprimió "PAREN AQUÍ"… y la ejecución siguió hasta evaluar
    los 12 casos. Un `print` no detiene nada. El resultado de esos 12 estaba
    decidido antes de empezar: 7 de 12 sin conclusión.
    """


def revisar_puerta_dev(conclusiones) -> dict:
    """`conclusiones`: iterable de (id, conclusión emitida). -> dict de la puerta.

    La puerta se supera si **todos** los casos de desarrollo dan una conclusión
    válida. No se exige que acierten: lo que se comprueba es el formato, porque
    es la condición sin la cual el scorecard de los 12 no mide el criterio del
    tutor sino su obediencia al formato de salida.
    """
    filas = [(cid, c, diagnostico_valido(c)) for cid, c in conclusiones]
    validos = sum(1 for _, _, ok in filas if ok)
    return {"filas": filas, "validos": validos, "total": len(filas),
            "ok": len(filas) > 0 and validos == len(filas)}


def exigir_puerta_dev(puerta, permitir_sin_puerta: bool = False) -> None:
    """Lanza `PuertaNoSuperada` si la puerta no se superó o no se ejecutó.

    Va al principio de las celdas de evaluación, así que la parada no depende de
    cómo el entorno trate una excepción en otra celda: si alguien ejecuta la
    celda de los 12 casos directamente, se detiene igual.

    `permitir_sin_puerta=True` es una salida deliberada para recoger datos de
    diagnóstico. No se silencia: quien la usa queda registrado en los artefactos
    (`puerta_dev: "no superada"`), de modo que cualquier cifra que salga de esa
    corrida lleva la marca encima.
    """
    if permitir_sin_puerta:
        return
    if puerta is None:
        raise PuertaNoSuperada(
            "el ensayo de formato sobre los casos de desarrollo no se ha ejecutado. "
            "Ejecuten la celda 7.1 antes de evaluar los 12 casos: sin ella no se "
            "sabe si el tutor emite la línea `Diagnóstico:`, y un scorecard sin "
            "eso mide el formato de salida y no el criterio del tutor.")
    if not puerta["ok"]:
        detalle = ", ".join(f"{cid}->{c!r}" for cid, c, ok in puerta["filas"] if not ok)
        raise PuertaNoSuperada(
            f"el ensayo de formato dio {puerta['validos']}/{puerta['total']} en los "
            f"casos de desarrollo ({detalle}). La evaluación de los 12 casos se "
            "detiene aquí a propósito: con el formato roto su total no mediría el "
            "criterio del tutor.\n"
            "Lo que toca es el prompt o el entrenamiento, y la decisión se toma con "
            "los casos de desarrollo, no con los 12: afinar contra los 12 los "
            "convierte en conjunto de entrenamiento.\n"
            "Si quieren correrlos igualmente para recoger diagnóstico, definan "
            "PERMITIR_SIN_PUERTA = True en una celda nueva; el scorecard quedará "
            "marcado como obtenido sin superar la puerta.")


# --------------------------------------------------------------------------
# Instrucciones
# --------------------------------------------------------------------------

INSTRUCCION_VER = (
    "Transcribe EXACTAMENTE lo que está escrito a mano en esta hoja. No corrijas "
    "ningún error: si una cuenta está mal, cópiala mal, tal como aparece.\n"
    "Devuelve SOLO un JSON con esta forma:\n"
    '{"enunciado": "...", "pasos": ["...", "..."], "respuesta_final": "..."}\n'
    "Un elemento de `pasos` por cada línea escrita.\n"
    "Si la hoja no se puede leer, o no contiene un procedimiento matemático, "
    'devuelve {"ilegible": true, "motivo": "..."}.'
)

# El formato de salida, y la evidencia que lo decidió.
#
# En las once respuestas completas de la corrida del 22-10, OCHO terminan
# copiando la última etiqueta que el prompt les mostró:
#
#     vis-01 -> "Respuesta final escrita: 24 x 13 lápices"
#     vis-03 -> "Respuesta final escrita: 36000"
#     vis-05 -> "Respuesta final escrita: 60 metros cuadrados"
#
# El modelo no desobedecía: estaba **continuando el formato de la transcripción**,
# que es lo que la LoRA de M1 le enseñó a hacer con un texto que empieza por
# "Paso 1:" y acaba en "Respuesta final:". De ahí las tres decisiones:
#
#   1. La exigencia del formato va **al final**, pegada a la generación, para que
#      lo último que el modelo lea sea la conclusión que se le pide y no la
#      respuesta final del estudiante.
#   2. Un ejemplo resuelto, porque un modelo de 1.5B copia formatos mejor de lo
#      que sigue instrucciones abstractas.
#   3. Se le dice explícitamente que NO escriba "Respuesta final", que es la
#      etiqueta que lo descarrila.
#
# Nada de esto se ajustó a los 12 casos: el ejemplo usa un ejercicio inventado
# (pan y panes), no una de las hojas, y las cinco categorías salen del esquema del
# eval set, no de lo que el modelo respondió.

INSTRUCCION_REVISION = (
    "Eres un tutor de matemáticas. Un estudiante te muestra el procedimiento que "
    "escribió a mano, ya transcrito. Tu trabajo NO es resolver el ejercicio ni "
    "reescribirlo: es revisarlo y dar un veredicto.\n"
    "Reglas:\n"
    "- Señala UN solo paso, el primero que esté mal. Si los siguientes arrastran "
    "ese error, no los cuentes aparte.\n"
    "- Si el procedimiento está correcto, dilo. No inventes un error para tener "
    "algo que decir.\n"
    "- No cambies los números del estudiante por los correctos: importa dónde se "
    "equivocó, no cuál era el resultado bueno.\n"
    "- La respuesta final escrita cuenta como un paso más, el último.\n"
    "- NO escribas 'Respuesta final': eso es lo que escribió el estudiante, no tu "
    "veredicto."
)

# NO hay ejemplo resuelto, y la corrida del 24-10 explica por qué.
#
# El ejemplo que había —una panadería, 15 panes por bandeja— apareció COPIADO en
# la respuesta a `vis-11`, la lista de mercado:
#
#     Paso 1: Multiplicamos las bandejas por los panes de cada una: 15 × 4.
#     Paso 2: 15 × 4 = 60.
#     Diagnóstico: Paso 2
#
# Copió el ejercicio entero y el veredicto del ejemplo, y "corrigió" su 54 a 60.
# Un modelo que ante una entrada degenerada se agarra a lo último parecido que vio
# convierte cualquier ejemplo resuelto en una respuesta posible.
#
# Así que del ejemplo queda solo la FORMA de la última línea, sin ningún
# procedimiento que copiar. Enseñar el formato no exige enseñar un ejercicio.
EJEMPLO_REVISION = (
    "La última línea de tu respuesta tiene que verse exactamente así, con una de "
    "las cinco etiquetas:\n"
    "  Diagnóstico: Paso 2"
)

# Esto va LO ÚLTIMO, inmediatamente antes de generar.
RECORDATORIO_FORMATO = (
    "Termina SIEMPRE con una última línea con este formato exacto, elige una:\n"
    "  Diagnóstico: Paso N              (un paso está mal; N es su número)\n"
    "  Diagnóstico: sin error           (todos los pasos están bien)\n"
    "  Diagnóstico: ilegible            (la hoja no se pudo leer)\n"
    "  Diagnóstico: fuera de dominio    (la hoja no es un ejercicio de matemáticas)\n"
    "  Diagnóstico: dato no disponible  (piden algo que no está escrito en la hoja)\n"
    "Esa línea es obligatoria, también cuando el procedimiento esté correcto."
)

# El reintento: si no hay conclusión válida, se pide SOLO la línea. No se le
# permite recalcular ni cambiar de opinión sobre el fondo —se le devuelve su
# propia explicación y se le pide que la resuma en una de las cinco etiquetas—,
# y queda registrado cuáles necesitaron reintento, porque un sistema que necesita
# dos pasadas para contestar no es lo mismo que uno que contesta de una.
PETICION_CONCLUSION = (
    "Esta fue tu revisión:\n"
    "---\n{explicacion}\n---\n"
    "No terminaste con la línea obligatoria. Escribe AHORA solo esa línea, sin "
    "ninguna explicación, coherente con lo que acabas de decir:\n"
    "  Diagnóstico: Paso N  |  Diagnóstico: sin error  |  Diagnóstico: ilegible  |  "
    "Diagnóstico: fuera de dominio  |  Diagnóstico: dato no disponible"
)


def prompt_revision(pregunta: str, lectura) -> str:
    """El prompt completo, con la exigencia del formato al final."""
    return (f"{INSTRUCCION_REVISION}\n\n{EJEMPLO_REVISION}\n\n"
            f"Ahora la hoja de verdad.\n{pasos_como_texto(lectura)}\n\n"
            f"Pregunta del estudiante: {pregunta}\n\n{RECORDATORIO_FORMATO}")


def asegurar_diagnostico(respuesta: str, reintentar) -> tuple:
    """-> (respuesta, reintento_usado).

    `reintentar(prompt) -> texto`. Se llama UNA sola vez y solo si la respuesta no
    trae una conclusión válida. La línea que devuelva se añade al final de la
    explicación original, que se conserva entera.
    """
    if diagnostico_valido(extraer_diagnostico(respuesta)):
        return respuesta, False
    linea = reintentar(PETICION_CONCLUSION.format(explicacion=(respuesta or "").strip()[:1200]))
    return f"{(respuesta or '').rstrip()}\n{(linea or '').strip()}", True


def pasos_como_texto(lectura) -> str:
    """La transcripción, en el formato que ve el tutor.

    La numeración la pone esta función, con la convención del eval set (la
    respuesta final es el último paso), y no la copia del VLM: el criterio es
    "Paso N", así que si la numeración dependiera de cómo el modelo partió las
    líneas, el caso se podría perder por una coma.
    """
    no_fiable, motivo = lectura_no_fiable(lectura)
    if no_fiable:
        return (f"(la hoja NO se pudo leer: {motivo}. No hay un procedimiento "
                "legible que revisar.)")
    js = lectura.get("json") or {}
    pasos = pasos_numerados(lectura)
    if not pasos:
        return (f"Lo único que se alcanza a leer en la hoja: "
                f"{js.get('enunciado') or '(nada)'}\n"
                "(no hay ningún paso de un procedimiento matemático)")
    lineas = [f"Enunciado: {js.get('enunciado') or '(no transcrito)'}"]
    lineas += [f"Paso {i}: {p}" for i, p in enumerate(pasos, 1)]
    return "\n".join(lineas)


# --------------------------------------------------------------------------
# Lectura o diagnóstico: a qué etapa se atribuye el fallo
# --------------------------------------------------------------------------

def lectura_fiel(caso: dict, lectura) -> bool | None:
    """¿La transcripción conserva el procedimiento que la hoja tenía escrito?

    Devuelve None cuando la pregunta no aplica (los adversariales no tienen un
    procedimiento que transcribir). None no es "sí" ni "no": un caso así nunca se
    atribuye a la lectura, porque no habría nada que leer.

    Compara sobre los pasos **ya numerados** con la convención del eval set. En la
    corrida del 22-10, exigir el mismo número de elementos en `pasos` marcó
    `vis-01` y `vis-03` como fallos de lectura cuando la transcripción era fiel y
    lo único distinto era dónde había puesto el VLM la respuesta final.
    """
    # Una lectura no fiable es un fallo de lectura, aunque el caso sea adversarial:
    # es lo que hace que `vis-10` se atribuya a la etapa de ver y no al tutor.
    if lectura_no_fiable(lectura)[0]:
        return False
    if caso.get("bloque") == "adversarial" or not caso.get("pasos"):
        return None
    js = (lectura or {}).get("json") or {}
    pasos = pasos_numerados(lectura)
    if len(pasos) != len(caso["pasos"]):
        return False
    esperado = caso.get("valor_final_escrito")
    if esperado is None:
        return True
    # El valor escrito a mano tiene que seguir ahí: es el error que el tutor debe
    # encontrar. Si el VLM lo "arregló", el caso ya no mide lo que decía medir.
    escrito = " ".join(pasos).replace(" ", "")
    return esperado.replace(" ", "") in escrito


def fuente_del_fallo(acierto: bool, fiel: bool | None) -> str:
    """A qué etapa se le atribuye un fallo. El resultado principal de M4."""
    if acierto:
        return "—"
    return "lectura" if fiel is False else "diagnostico"

## 3.2 · El eval set

El eval set **no es el corpus de entrenamiento**. Es un conjunto nuevo, escrito
a mano, que cumple los cuatro criterios de S05:

| Criterio | Cómo se cumple aquí |
|---|---|
| **Representativo** | Problemas que un estudiante real plantearía, con enunciados en lenguaje natural. |
| **Con salida esperada** | Cada caso trae la respuesta correcta y un `criterio` explícito de qué la hace correcta. |
| **Cubre casos difíciles** | Es el eje del diseño: multipaso, distractores, porcentajes encadenados, proporcionalidad inversa, redondeo contextual. |
| **No contaminado** | Ningún caso proviene de `math_tutor_dataset.jsonl`. El generador lo verifica automáticamente y falla si hay solapamiento. |

**Por qué hizo falta uno nuevo.** En M1 el modelo *sin entrenar* ya resolvía el
90.91% del conjunto de validación. Un eval set con ese nivel de dificultad no
puede discriminar entre sistemas: todo se ve bien. Este está construido para
que el modelo falle, porque un eval set que no reta al sistema no informa nada.

### Los tres bloques

- **A · CÁLCULO (12 casos)** — mide **habilidad**. Cada uno tiene un modo de
  fallo documentado en su `criterio` (por ejemplo, sumar descuentos sucesivos
  en lugar de encadenarlos).
- **B · CONOCIMIENTO (4 casos)** — mide **conocimiento institucional** que
  ningún modelo puede tener en sus pesos. El baseline debería abstenerse aquí;
  si en cambio inventa una cifra con seguridad, es una alucinación y el
  scorecard la registra. **Este bloque es el que motiva el RAG de M3.**
- **C · ADVERSARIAL (5 casos)** — mide **robustez**, siguiendo el red-teaming
  de S06: premisa falsa, indefinición matemática, datos insuficientes, fuera de
  dominio y mantenimiento del rol.

Supera de largo el mínimo de M2 (≥10 gold + ≥2 adversariales): 16 gold y 5
adversariales.

In [5]:
import base64, gzip, hashlib, json
from pathlib import Path

RUTA_EVAL = Path("data/eval_set_m2.jsonl")
SHA_EVAL  = "ba9c4f3b2d80d687b4a56e1d93e260a964e0b375e61de1d27d10c41a43e1f96c"

_BLOB_EVAL = (
    "H4sIAAAAAAAC/61aTW8cyZG9L+D/kCBggIJbrf4k2b0YLLgSZRPQkBpyxoDhNYjsrmQrtdWVPVVdBOmBf8Ue9qyjDjoM5mBg"
    "Lgbcf8zvRWZ9dTU1zbGAGYidlR+RES8jXkTmDwc2Opiqg8jePu/1DzrqYBa773PDtrmO53ns2Jjls7VdObYu8xh/6szdmGSu"
    "I5PoSLrYZJWv2eG7RK11HJtUzd1ylWo1VHP9XmcqMmrt0sTGscuUVv2Tca+nVibDL0ykVZ5o9aAGKnXSA93nehYbdB2Mjsdb"
    "XV1XvdULjTUS1cdExVRd9c9/vMw3H5K1U5FNTOoUpojMXW7iO5P8F0U12cqkFBvSvsVOVH+q3ugsiDnPTbbWCcTe/H8Q8is1"
    "HuPf7v8k0n2A7pAkyFn0H7C/l/QrNZrU+g+n6izG3qEWZTI/l/qd74O+/R7lLzqPKAsFdnE+t5ufEg4JO3we+nKBMOYKmxEB"
    "1K1NdDz1H7wuuNd5atcmtbLXl7leQx+Ou59bl2Di0oY6m0KryqSpS5VNMGZpIuvw552OLVRuYpWaDMaH4rqiRXzIZR7M/cNB"
    "gY8kX2K9uWACPVzKRhHq4G9/+81//FAH3GAfwNnk1qVLWegmstkasgNGuoG5s4TCZzqmvt7pBzUcQG3rPLIaW8HW+icqA1KW"
    "+XuDXQBm/ZE0vHPLGRq6NFAYvrYmMWqkgBaYVWeCSeg4XWv0u7YegdXsKjFzk9m1AD2HNtPEZR3AcO5hmFWtKqt6Q9eJx8Sj"
    "mIRMkQaOUxND2VyKFotVsvl5SWB7REWmudXhoIbTr+W0xnaul1h+BdvGDYHkkPkDxZGE8BDomhztgtbkqBq5jS3IapI8mVvs"
    "AaCZx/mDEekzZdNyA5k6LNTaKXTawTm9s5lAnQJm5t49E4MsXWRiHGMzM8ouEhg9hfRPBt/kqI284T7Ig2+5M2lG4OUJzkBk"
    "sranSzBFwAwd2FyvIBb6UrWD7lgtDc4cVL75eQbBiCA9Q29oiQP7/nvxWZnvcwvB6fT6PMex5egOMMjesH1Y8EE8n192UutX"
    "Q51vUrc6JthWGp4YXjmhDh9F3EvZ8doSLHF9MzpMN5U90TF6NzQYl4vXUHdF1Mgcfpc1aad+yHORGr7vaLcXY3uYt4W0e7sw"
    "ah4kTVVhGuUBBrWLg0u7XgwAyia17gJ2pWdZnv4KR0a52mga7YOm1EQuiYy7oSrM/TrX8Raa4FUQME2mPeTh6ZIMJ4J76ENr"
    "Pj511UseWBxoOTExTxYjrVPLzYd7WA6+ruxaxsJMBojLr/ugR4HwCkcyIgymfunN32Xar9RRdzJSepU6LAU5ljjNpmb5C0D4"
    "Hk7aH4Zb+Gofw8TPiMQdpbPNJ4FF0Igg5R0MoJVOUzvT06afPC5l34WU6usOlxRUjZg3iy3Ygi7XTNUfTl+en6rTq6vz/z4V"
    "rKz4IVVHyvldQldwZA7eCwHnyUg5bsNkvA9MvDNkrDNzqHgbIzhSa/IZdYxz+L82dotU9CcKvt18pOqAIj3XjC8zF2eeefW6"
    "w+PGgIJyVSBBx9Afc7iZHNc9ECJeyXObZJ2KkxK141i6Mr7IxFMRGlASYR6NUno5C7FKnIx4LrhFHoOgE4a5Y7oRzMVtwQ/t"
    "9CKDXtjRNjRelRHHy5wnxdTwxwg03jdPG6SHePj69E+XV4UrpqUievJO4ecykExxTbd5Etn0yZgZ7PAtR/uABsqaM66+N4+z"
    "ci1RCrZwt0A0glPvtzQPPCcYLOnyQweUaOYtCDcEkem4UzODL+EmoaQ+xujIcjdQFU200gtHpJlbnBJ754QhVRO41C5oCypv"
    "0Ktoei1SEW0rUnn0Ers9CrnTEiKygIUGK/GnxfwITr3uSRFcGqydJM8s8kSAWWwby2uZ+HO7n4bJ/OwTmX00eoSC+y+7Ofgb"
    "OathbZy0nF7uDn9dXFKSLF/qpO6N+iNZ9rAgBmSBEsGGMMXm5wQQehZooefu83jzIUPr00ObSN0G4PE+APSeXrxWfIMjs26i"
    "7xToi83D5ic1eDEm6rCB2M5AfCB3nCeekr8Y8lM4S/g/0jPJO5Y6XZusQJYf56nPkGrefCDEapCCBysaw6rlJJ/j2l4OP+AQ"
    "cj4TNuxpyqBap4anbyhj6hLp9Vx6fSVxcruvzwC9CGGFsFtDf4shUxlHf8YguxNWR9W8LVTpAOx6zK2AjZN1dXb97WVHJY4/"
    "xF03UEaIBYmwlyaiCnU9GVFHO9B0spc7Sx09mngZkrsbK0S8mfMBDQjVOLFCr12CzBA5hySCyzwVp9QfqGjzSTd5sbSAR6cw"
    "Hf5M1KScBYyLx53+gZHAZmBT0PLSfQ42fpCr8vsj4ccD2PE4rP/cL1BDDoh2te6UHWH6CYac+BG77B8+bdv+7Zay1PnFH8+u"
    "rk+nJIRZsUZHAlvYfTe40hT+bRFrqcwwLUbMghfX5MhImB/+TQZ00rb+ZB/rL4xjUmT1DQtIooJW2gVkYjtKxE0WeYytLC0z"
    "rl6RcAl/GIRfSCcTYaQmgx/Fv6wRACjvgYLNp0TSWpqWmhgVM9AXSRvPCE8OAxVOvCnQhghfZ9hFphfmyrwLA7OwPGZ2Btri"
    "yUw5/HOw2nxIjS5T/LjcsgTSEmGDUa+1bjPg+Xk4Q7FZzDDiBCMJkY8OH1Z53FTWec7eWHIw2sm1BqPWVNtQfW0Xuew/mHXK"
    "ZIHJnOCA/A6jRWDGg1TSt1g/PeOHLC3s9XtPJFLe5zTj2FnJDXwMg0PffOI0SudMheDXyZLGv4X4+p3jXr2SJoOdFcq5w0cE"
    "uSJ1jcM8jzsceL374J2DJBwL/aYdclmXgE+gB/HR7ZMZy8o1SLzirO+9Ve+L75Lf+f4nvUdYzUnvUVLzttRacD6XUzlhjOPC"
    "km0qocdblLJDRfX4c3wyYAQKsgiBG38hWiNit9HQ3zMOSRX0xou5zamDT1lqKkJLZc1XviG4cDWKP/Q0G142wVmmU8WW7pk4"
    "e8Xy93EP2vguqRcVkX/ld06NugOpExbTPqghUig/s5+lgJOEniz3q/wSj3apTyLrIk9lMVH+UHhMd3BUw005qFh4KqJI/2Mp"
    "BXVH45rvuAaT9fUDzKN+J5/Ra9g97u9CF9t3RDZfhH57efHq7Or0VSAwDMmbj2uioYahITCjDguTqcwuV7F59m/GMBGrBZ7B"
    "nuCZ6Zn1BCazCWgxEvNY/7WdmLFaU9XzxkxaNe8X3osfHCr91zwWAowEAZl9In5SOnUEgRG1ufnoi1/gIx2peckdAnwYseez"
    "Jd0AizRXMnIwDyzSbykBYBmR4DNXJjvHV6txEplDbj1enNTQ9CctIrLWxX0UoXIUNs36Rlkfr1WM4ornMoq9OK4BrlFEmKpD"
    "LCgM/hDdnkmJ4MWY8Wv8or8zgLF9G4HX5xeqNJuUA6r8MXFLyzQ7ZQFqZnUdi4Pxi6OROuTlVDn8C5FqEbOOSCzSvrpziZvb"
    "pWVEaEW42OLo6Jvaog1abRZIK2WrrOMtNY5PtrbrPCT9zFnCSF/YY7knNgvryLK/BwxVFUTrno89V1QstCcl7i1H2MLZ2Zaj"
    "LAaNmh61Nm+H5nmC9N1WXYjXLY0RzDohANN1idAzw9rxGuc0zcx/yvXj1env/QckFCLcg5rbEOciN/cBfR8737gbP3dhlOr+"
    "rNc2+WB/k8eaifmNBrFq2Pqf/0AIE5MthP2C4NoEFC7KYT9FH2TmeXFVCNWFmkvRuVA15+fngmpKRETyQZhldYS0THy9e47O"
    "L69dFxT/+WaHbODOfWmrgmWlYeq6aRsSf868hWO/0TF8h25ZeR7rO5Oh558P/Cps9BuSbPlEGk667uAvLRAM9wYBz8oNPJZd"
    "bmOgHhDkRC2RJaCblPln0B3JTc2KVEJRZ2HquS5vYCLW9Oe5v18urmKlSujPZ8v+bz6zIuYbdnuhKDfGXw8yHUPfL6xaW7GN"
    "BSZ2/lqyDgi5CpdYmjEcpUVw6Eqm3jjhvjxdAAGdPYy+EAKwZbYMO/KPZOWrXFIFZPE7ADDa3wukbu1gQrBZxNqFW4TIUkPC"
    "N3QFhlfsaxiaakCq5ovGkm+ECRpgwCFlPMwb7DWUwD3B3FXu3OXsq/lr64aSTCpnEsveyu2/WpbBnkVnHwN8OlTWDtWhlIEf"
    "VJybhZOq7bPCyEi6fNmDaOJjlozZE3ou5JJByrD+qryKZVkLS2eVpkSlm58WrLYVPl9HuulnTHmlzdwSHdLNBwVIOPqSPGOF"
    "YMv3VM6m0M0XgplXIxsbmmRDqRL+2FYq26hW/++kCUgd3bWYCNpYucOBjlt4NEub6ZtbHW9V9s7ufdlyJRcxgOSEzgAIKx4+"
    "MBYETdQgNFG+PLLdcaqktMUbHz4n8jc+ww6n575YdZO3PkPJwRpD6Q3cruF94GrzMdQIO/7+ddDhrGPl1HELKm+vzr4+vz5V"
    "r0/fXJ/61yaBn4idX15eXZ39/vzKk2fRi6Q6bmYQfkwaM3G81bNUinZE8Psc3usWPysisx8qtoDgVSab5ZdKg3mlheaHun4E"
    "L9uaFf8lCpW/7kMDrTmspipSsoO/CMzcOzuzYsc/H0wekWHymAiTxh4y2/yJ3KH83cbrYF+8WvgzEFG5/LoRhqNJcHYEU7pr"
    "ebQld527gl60df9I5+7Ru4ZL8OtEeipN/jo9sYlQpWLvRG51dOGAqd6einBa+uOueisphiOeXBUYk+LepPmSqw7Uc2Tar88v"
    "zl+eb/7voqteVQyIK7ZERA8BZCnXYa8DATrwcOyw9lkPvuOUP8l57YKpkH2/bomjujChMfz0hqn9KJUZfpTl3FCiCs1bGmq0"
    "ZjCXjWoHYuU89GtTMK/N5PGe97rVb5hgB9oDTOCFxKtuXzn7RtCVRY78RUO5u3qhtQ3t4b7QFkJ0A0KUw6NYCVatinuK+Hhn"
    "WajQatLjS4LNT7VSO4ud9com9LVcOX/NIqEVbIHkiSUqEIK1Tdps8DWfKsldPAm7lMrTIqWehgI8LMrQwOht+eySr0beaUKD"
    "2Zaf2JO2epeOPFLwImWG1z614d52IibP4qTXJo2n315eq/OL6+9e42CcXXx7dh2OBuK+9YkeReYRkfdW4ryLIO9LNMVyv/4M"
    "yMwVdr0mCszbeQ3WVOpcokNoq12oSYt/WyM4fqfnQWjvzIOYbaC28TXaF18gQKlmBhqxZmLbtNNuPpI/QucznBL1EkpQevOj"
    "v4TB4QHziNqUMZNgCU2tdfAEspBkoBbEao6dmCnGy13POidx3MpO4SnhApzSD3nE21mhX0xCYrh3Wb2qNsKqSJQWZsYCW7ik"
    "IpK8a4o2nzJ2aoHn9XdnV6fq1Zl6dfk1HOtlQA45Lri6EfTEm09L60uyOB9B8gYjLQ+C5xpa6AOp9oJJKzLaz8DqURbYVImE"
    "Mzn4O1XFD4V+C1jVdCdcstR66CBB3gsukCCbkkHSu7loY6090DfeF32pi2/4onmRMmFsMs1sDXDwzZ28Sg15jC86GclooAPn"
    "CziIdEsTCGHd+/LK8Jh052To667G01dfuujuSHQanns46fU7TFcdQXvLJ7Th0bcUYyEb0qBw3wO4RCG5m5aL8jG4/1MelR8X"
    "b3iHx0ds6I/6/IFV+GLeV26ZkTn4wsSExzRFVdOnBaHjYCp5kH/HSQLFcstasgO4Ei0V6XYdf9Jr1fGvLt90+TJQqG6MDcAp"
    "8nEClRnym0AfuGJDB0GXaWv/UgyH61qttb9mAvGpJ+6whDym9FqMjNyFmyQzmx/Nr3fBzIt8WlQTRWYTOfmXt5j40kLLj6D5"
    "X0r8Q9R8MQAA"
)

if not RUTA_EVAL.exists():
    RUTA_EVAL.parent.mkdir(parents=True, exist_ok=True)
    RUTA_EVAL.write_bytes(gzip.decompress(base64.b64decode(_BLOB_EVAL)))
    print(f"Eval set escrito en {RUTA_EVAL} (copia embebida).")
else:
    print(f"Se usará el eval set existente en {RUTA_EVAL}.")

eval_set = [json.loads(l) for l in RUTA_EVAL.read_text(encoding="utf-8").splitlines()]

sha_real = hashlib.sha256(RUTA_EVAL.read_bytes()).hexdigest()
print("SHA-256:", sha_real[:16], "… coincide:", sha_real == SHA_EVAL)
print()

from collections import Counter
print(f"Casos: {len(eval_set)}  ->  {dict(Counter(c['bloque'] for c in eval_set))}")
print()
for c in eval_set:
    print(f"  {c['id']:8s} {c['bloque']:13s} {c['subtipo']:26s} {c['input'][:52]}...")

Se usará el eval set existente en data/eval_set_m2.jsonl.
SHA-256: ba9c4f3b2d80d687 … coincide: True

Casos: 21  ->  {'calculo': 12, 'conocimiento': 4, 'adversarial': 5}

  dif-01   calculo       multipaso_encadenado       Un taller compra 3 cajas de tornillos a 18500 pesos ...
  dif-02   calculo       informacion_distractora    En un salón hay 32 estudiantes: 18 son mujeres y 14 ...
  dif-03   calculo       conversion_unidades        Un tanque tiene una capacidad de 2.5 metros cúbicos....
  dif-04   calculo       redondeo_contextual        Una empresa debe transportar 1250 cajas. Cada camión...
  dif-05   calculo       division_decimal           Un lote de 7.5 kilogramos de café se empaca en bolsa...
  dif-06   calculo       porcentaje_encadenado      Una tienda ofrece 20% de descuento y, sobre el preci...
  dif-07   calculo       fraccion_del_resto         Ana leyó 2/5 de un libro el lunes y 1/3 de lo que qu...
  dif-08   calculo       proporcionalidad_inversa   Si 6 obreros construy

In [6]:
# Un caso completo, para ver la estructura.
print(json.dumps(eval_set[5], ensure_ascii=False, indent=2))

{
  "id": "dif-06",
  "bloque": "calculo",
  "subtipo": "porcentaje_encadenado",
  "input": "Una tienda ofrece 20% de descuento y, sobre el precio ya rebajado, un 10% adicional por pago en efectivo. Si el precio original es 200000 pesos, ¿cuánto se paga al final?",
  "esperado": "Paso 1: Aplicamos el primer descuento: 200000 × 0.80 = 160000.\nPaso 2: El segundo descuento se aplica sobre el precio ya rebajado: 160000 × 0.90 = 144000.\nRespuesta final: 144000 pesos",
  "criterio": "Los descuentos sucesivos NO se suman. Responder 140000 (equivalente a un 30% único) es el error clásico.",
  "evaluacion": {
    "tipo": "numerico",
    "valor": "144000"
  }
}


In [7]:
# GUARDA · ¿es este el mismo eval set con el que se midió M3?
#
# Las corridas del 22-10 y del 22-10 (tarde) imprimieron las dos `SHA coincide:
# False`: en Drive había una copia distinta de `data/eval_set_m2.jsonl` —la de
# antes de corregir el criterio de `adv-02`— así que su 15/21 NO era comparable
# con el 13/21 de M3. No se compara contra un número lo que no se midió con la
# misma vara.
#
# Ahora la celda lo CORRIGE sola, sin borrar nada: aparta la copia que no cuadra
# con un nombre que dice por qué, escribe la copia embebida y verificada, y
# recarga el eval set. La guarda se mantiene: si tras eso el hash sigue sin
# cuadrar, la comparación queda marcada como no válida.
import shutil

if sha_real != SHA_EVAL:
    descartada = RUTA_EVAL.with_name(f"eval_set_m2.NO_COINCIDE-{sha_real[:12]}.jsonl")
    if not descartada.exists():
        shutil.copy2(RUTA_EVAL, descartada)      # se conserva como evidencia
    RUTA_EVAL.write_bytes(gzip.decompress(base64.b64decode(_BLOB_EVAL)))
    eval_set = [json.loads(l) for l in RUTA_EVAL.read_text(encoding="utf-8").splitlines()]
    sha_real = hashlib.sha256(RUTA_EVAL.read_bytes()).hexdigest()
    print("La copia de Drive no coincidía con la del repositorio. Se corrigió:")
    print(f"  la anterior se conservó en {descartada.name}")
    print(f"  ahora SHA-256: {sha_real[:16]}…  coincide: {sha_real == SHA_EVAL}")
    print(f"  casos recargados: {len(eval_set)}")
    print()

EVAL_M2_COMPARABLE = (sha_real == SHA_EVAL)
if EVAL_M2_COMPARABLE:
    print("El eval set de M2 es el mismo con el que se midió M3: la regresión es")
    print("comparable, y la fila de regresión de la tabla de delta es válida.")
else:
    print("=" * 78)
    print("ATENCIÓN · el eval set de M2 sigue sin coincidir con el del repositorio.")
    print("=" * 78)
    print(f"  esperado (embebido): {SHA_EVAL[:16]}…")
    print(f"  encontrado en disco: {sha_real[:16]}…")
    print()
    print("La sección 8 se ejecutará igual, pero su cifra queda marcada como NO")
    print("comparable y no se pone al lado de la de M3.")

El eval set de M2 es el mismo con el que se midió M3: la regresión es
comparable, y la fila de regresión de la tabla de delta es válida.


### 3.3 · El eval set de M4

`data/eval_set_m4_vision.jsonl`, generado y validado por
`scripts/eval_set_m4_fuente.py`. Se escribió **antes** de tener las fotos y antes
de escribir el pipeline: el criterio de cada caso está fijado de antemano, que es
la única forma de que medir signifique algo.

| Bloque | n | Qué mide |
|---|---|---|
| **localización** | 4 | Nombrar el paso donde está el error |
| **compensado** | 2 | Un paso mal con el resultado final **bien**: lo que un corrector de respuestas no ve |
| **sin error** | 3 | **No** inventar un error donde no lo hay |
| **adversarial** | 3 | Abstenerse: hoja ilegible, hoja que no es matemáticas, dato que no está escrito |

Cada caso trae `evaluacion` con las **claves aceptadas** y los **textos
prohibidos**, y un `criterio_s13` (`exacto` / `valvula`) que lo liga a la
plantilla de S13. La calificación la hace el mismo `evaluar_caso` de M2 y M3.

Hay además 3 casos de **desarrollo** (`dev-*`) que no se reportan: existen para
mirar sin gastar los de evaluación, y no se fotografían.

> **`vis-12` reutiliza la hoja de `vis-07`.** La foto es la misma y lo que cambia
> es la pregunta: se pide un dato que no está escrito en la hoja. Por eso son
> **12 casos y 11 fotos**.

In [8]:
import base64, gzip, hashlib, json
from pathlib import Path

_RUTA_EVAL_M4_TODO = Path("data/eval_set_m4_vision.jsonl")
_SHA_EVAL_M4_TODO = "3fb96f72d9dd2e3922c8b951d11fd750546df2694111d59d50e52d169efc0caa"
_BLOB_EVAL_M4_TODO = (
    "H4sIAAAAAAAC/+1azY4buRG+5ykIAQHGWEVWd+tvFCyCge0AAWbXyTjZS7AYUN0cDR022e4fIbax95xzydnHHHwIfAjgywLW"
    "m+RJ8hXZLbWk7pE1k5EReA6DEVlksVj1VbFY7LcdGXWmrLOQ2a/6XqfLOjNlXhWC+pQJuZJveCiNJkqSinmhc060C4EZnMWS"
    "JakJRSRjKXRupuzTz0KzV8XynyzhmWGxYOJVIReGun5DXGTM50Kv1+y9TObUL3ShQ8kjQ6Q/ac5C/pKzPOWC+QOmlu8SGYqs"
    "hwWeFMt3WCtbdbJr/pphWS+wkzK7Di2fgdefO98VKpeJkiGPaRLP3CiWmJSpOptIgBJxVmg+pUWX/wDLHjGrGuxbFvR923Uh"
    "sqQQWc7ZldRcTYmwYtX5sZTgUqSpSSGGj44FVya9tMMvRRamMrebxcTOihqaNBVhSfAsQYBSOCtM2dsO9mKpodE5lC4uuYJZ"
    "OI0MFV8It2mrfb9SBPOnlpNiW/16+TEW6UZP79NH18zI3FCHWr7XgrstpeZazqQ1UrmGt5oZrH4NOj/+ROJghyKV5jLzApJY"
    "/JXTzkDRJueXV8ZuUxdK0WqwUG5HYbudn37xto5M/wsg029BJsu5hhzMKp95w36fQfIUQHrNMiDI6KyIsdRkMN5EqxuEqRHX"
    "WxA9N9TvaMTEzBx3cOI6gvB2mW8sT5q5bgKRflD27kCSKOW6u4j02hFJ85ohORoGd4PkGi7eBiS9HUh6m5CsTWyDon9XKHae"
    "KWbVw1w0SBTXueAVgsgaFGcW4g1RU1J1ajV/I3iDLwDeoA28PM2X78NCGRY6qAwApT5LRGYB7GDn939JG4wAiYLWr+GY0Jnw"
    "Od8CMBS3Gk3Kc1wRM/s9vw+MnqJZYTS3cVjUZkzL8f/529/tSAqzo9qMrUC7lngX1cENcZbmtUTaUUm6PbDXkAs2gB3sADvY"
    "BHZt4v4Y698S2OfQeIS/JJUkQ2ZlgxGyHFZlM+zt12SRCvzoZVwhliWSp0QA1nGKAkb74T74AnAfNML9BfloAsRTbuAPJiwq"
    "FJ3zaNrejMl5wZXtyVPBJkzL5b/MZpIBjMjZKjEwW7h/KhcSYjs859C4KnlRYuG4Td3ay3+ziYvdVQso9wajZoyDUEp7UCqB"
    "ac0A94ajh1TCQmW4Cc/QxInQGSHmvsA5bE0kYFGhDUEMOcW8UPC1WCLwej642myBctRh2ahgqSjAZgVbvktFQxy23RS+C205"
    "A8dziveYNOMUvcGSq7xIeQlHCtNDgHHcEm9H/UqasOBRCvkPA+WoJeiO+l8BJDtPzs9+eDZlK+HK6BojUpyM+49wjkFAxSnC"
    "1tVO5iqVxdnJqP9of+AdHR3Zo0Zkr3MF7CF4PECi6j2ebGe8CluguMmZwRYXFCdp7Lds9NjFSfx3M9E3Kft2sDl+PDkIizS+"
    "EYwl4StBI/Bmk9moEZbQ9mfiElr7DGCON4GZSV2a6r5wOW7E5Tnu/KQMm9qfaajAaDbo9bss6A2R+w56w+0QCzliyLF96r+A"
    "7qpaAg77ku2UmAGyxO0b4kYHvCsW1NIEmx0ElBRQEkAVhUEztMFsF9rlCdeCbprSiO6ScCt044Cy9ZXKYmgLbfN2AERLPV9+"
    "1C1U2ws2tLSF10om18O3OjaauaGEVa1yVbJWRc8qD6vjhG0xAM7qElQkSv8s70Oy7aDZy6DPBY7tHTf7/YZcT55fXDx78sfn"
    "PfY9XIoCXyXXft+ZHNt3Jm3ZSihDJclm0CKpElc09heplh/K5ADLBOzapHwnUVkIXAJkxCNGzrSdspzz2gC6d9MitDJ+kddE"
    "vHQaYDRO3AUES5fO47VU4+BedeEo6SHZDvUor6U0d4fK3IM//c/8aa/3nB7be05bc/1UVom4q7HYXNwbsDDG0ePycXaK1oFZ"
    "vqpz3k3xKx8ypQ/BTC7rtwXtU+tAo/IeYH3KpwwsaLkIBCwEm+X7/deBm71qFLTcCIIHr7qbV+3JCg/zr659ESk4Kcnubq/D"
    "ef1Nh+PRQqQZTyVG3pPLef1GlztRxDtDjnttXtrrsHtu6jLa8zzlV5IqOrFZ0AHzmmqRQl8ZKvM82nC2w6C9QaxBu+y/LbTh"
    "0zyhSlRpOgbHjEyFZDreMjlTgikhKnzTFBVy/YbjilXrV9i8WXVIJeY0syTyhGOVqkUOtWGc9YKFbqTkQs+JzpmEMtMYW11+"
    "0GuJFuIeD4mzpz88u3hxdvG7s/OptXKjdaENLJ8sP2SwWNc+u3z/3Fa17eMLKabHnoqZsHmOxh5TO6hUutOx4l3qkHqBzfPU"
    "qgMS7/cP7/j+0fzCegKcMZfPYU+4l0I3ZDlSNuko5trYPW5bGpRcxMt3uQzNg6fcp6fUWYiXsJCEOqjXmm1NXhuEZ8dyL2vH"
    "GdUYG1B0okR4LbrAp+6y60IsTNZlHKB486jHfluI1I6OTCy1BKAjcrYIJnOOxq+X71mp05vAt9/Z/M92tk8//4EcivYLgwMz"
    "UI87uK9ERskUlUfE2gpNfua3nkMbJ9C460orFHFKCWy+hq1GPDdVrHGZCDwd891CD8523GOprJkM3b+gZx8OYA0LEnqXubXz"
    "pKLIJb2JkXHtQbUCR4+d8zUuQh7PJJ9arEhX0W/DCGGsPLfWrrRW3fZx5bZxowtFYnH0T4Lcmm13KCVn6erd2psMq1fg+vO0"
    "o2o2cKNv/BZoWjLBTcjV4NZNeg7xW1+gLantBfqm6rPf+gI9Htz5Bfr/8okOFt/GnH/ES7tbcP8zhj+kyq43HrZUggtN5SuR"
    "0ZUJd2RO31LQXZl+Ttez3cdkVYtqv/6wpfrrDw+u/vrDluqvIzzcqw+9Vx8S4j8D2MExH+jciq3F3LJ0UxajFP0cN1SfkKqU"
    "5Z7dEtSaZutQK5b0+Y9NadCpqk9GbL2JvlYLWgq2/mSjuHRQXPVbHvX8yVcbUf8Lif3Lz9krAAA="
)
if not _RUTA_EVAL_M4_TODO.exists():
    _RUTA_EVAL_M4_TODO.parent.mkdir(parents=True, exist_ok=True)
    _RUTA_EVAL_M4_TODO.write_bytes(gzip.decompress(base64.b64decode(_BLOB_EVAL_M4_TODO)))
    print(f"Escrito {_RUTA_EVAL_M4_TODO} (copia embebida).")
eval_m4_todo = [json.loads(l) for l in _RUTA_EVAL_M4_TODO.read_text(encoding="utf-8").splitlines()]
_sha = hashlib.sha256(_RUTA_EVAL_M4_TODO.read_bytes()).hexdigest()
print(f"data/eval_set_m4_vision.jsonl: {len(eval_m4_todo)} registros | SHA coincide: {_sha == _SHA_EVAL_M4_TODO}")

# Se separan los dos splits. `dev` existe para mirar y afinar; `eval` solo se
# ejecuta, y sus 12 casos son los que se reportan.
eval_m4 = [c for c in eval_m4_todo if c["split"] == "eval"]
dev_m4 = [c for c in eval_m4_todo if c["split"] == "dev"]
CASOS_M4 = {c["id"]: c for c in eval_m4_todo}

# `archivo_de` resuelve el alias de `vis-12` -> `vis-07.jpg` y viene de vision.py.
from collections import Counter
print("Casos de evaluación:", len(eval_m4), "| de desarrollo:", len(dev_m4))
print("Por bloque:", dict(Counter(c["bloque"] for c in eval_m4)))
print("Por criterio:", dict(Counter(c["criterio_s13"] for c in eval_m4)))
print("Archivos de imagen distintos que se necesitan:",
      len({archivo_de(c) for c in eval_m4}))
for c in eval_m4:
    alias = "" if archivo_de(c) == c["imagen"] else f"  (alias de {archivo_de(c)})"
    print(f"  {c['id']:8s} {c['bloque']:13s} {c['criterio_s13']:9s} {c['imagen']}{alias}")

data/eval_set_m4_vision.jsonl: 15 registros | SHA coincide: True
Casos de evaluación: 12 | de desarrollo: 3
Por bloque: {'localizacion': 4, 'compensado': 2, 'sin_error': 3, 'adversarial': 3}
Por criterio: {'exacto': 6, 'valvula': 6}
Archivos de imagen distintos que se necesitan: 11
  vis-01   localizacion  exacto    vis-01.jpg
  vis-02   localizacion  exacto    vis-02.jpg
  vis-03   localizacion  exacto    vis-03.jpg
  vis-04   localizacion  exacto    vis-04.jpg
  vis-05   compensado    exacto    vis-05.jpg
  vis-06   compensado    exacto    vis-06.jpg
  vis-07   sin_error     valvula   vis-07.jpg
  vis-08   sin_error     valvula   vis-08.jpg
  vis-09   sin_error     valvula   vis-09.jpg
  vis-10   adversarial   valvula   vis-10.jpg
  vis-11   adversarial   valvula   vis-11.jpg
  vis-12   adversarial   valvula   vis-12.jpg  (alias de vis-07.jpg)


### 3.4 · Las imágenes

**11 archivos para 12 casos**: `vis-12` reutiliza la foto de `vis-07` porque lo
que cambia no es la hoja sino la pregunta — se pide un dato que no está escrito en
ella.

Las hojas para escribir a mano están en `m4/eval_set/HOJAS_PARA_ESCRIBIR.md`: cada
una dice qué copiar y, en los casos con error, **qué error cometer a propósito**.
Las fotos van en `m4/eval_set/imagenes/` con exactamente los nombres del eval set.

El emparejamiento es por **nombre base**: el eval set dice `vis-10.jpg` y un
archivo `vis-10.jpeg` o `vis-10.png` lo satisface igual. Si hay **dos** archivos
con el mismo nombre base, la celda se detiene en vez de elegir uno: si las fotos
fueran distintas, la corrida mediría una hoja y el informe hablaría de la otra.

Si falta alguna, el notebook sigue y marca esos casos como `sin_imagen`. **No
entran en ninguna media**: una foto que no se subió no es un fallo del sistema, y
contarla como cero inventaría un resultado.

In [9]:
from pathlib import Path

from PIL import Image, ImageOps

DIR_IMAGENES = Path("m4/eval_set/imagenes")
LADO_MAYOR = 1024          # límite del lado mayor, en píxeles

# Por qué se redimensiona: Qwen2-VL convierte la imagen en tokens visuales, y una
# foto de celular a resolución completa llena la T4. Peor: haría que el coste de
# cada caso dependiera de cómo salió la foto, con lo que la columna de latencia
# dejaría de ser comparable entre casos.
#
# Por qué `exif_transpose`: las fotos de celular guardan la orientación en los
# metadatos. Sin corregirla, el bloque adversarial mediría rotación en vez de
# legibilidad, que es otra cosa.


def cargar_imagen(ruta, lado_mayor=LADO_MAYOR):
    im = ImageOps.exif_transpose(Image.open(ruta)).convert("RGB")
    if max(im.size) > lado_mayor:
        escala = lado_mayor / max(im.size)
        im = im.resize((int(im.width * escala), int(im.height * escala)), Image.LANCZOS)
    return im


DIR_IMAGENES.mkdir(parents=True, exist_ok=True)

# El emparejamiento es por NOMBRE BASE, no por nombre exacto: `vis-10.jpeg`
# satisface el caso que el eval set declara como `vis-10.jpg`. En la corrida del
# 22-10 el emparejamiento exacto dejó ese caso sin evaluar, que era el único
# adversarial de hoja ilegible.
#
# Si hay dos archivos con el mismo nombre base, `inventario` LANZA y la corrida
# se detiene: elegir uno por orden alfabético mediría una hoja y el informe
# hablaría de la otra, sin que nada avisara.
presentes = [p.name for p in DIR_IMAGENES.iterdir() if not p.name.startswith(".")]
PEDIDAS, FALTAN, SOBRAN = inventario(eval_m4, presentes)   # puede lanzar ImagenAmbigua

print(f"{len(eval_m4)} casos  |  {len(PEDIDAS)} fotos distintas  |  {len(presentes)} archivos en la carpeta\n")
for logico, info in sorted(PEDIDAS.items()):
    marca = "FALTA" if info["archivo"] is None else " ok  "
    real = "" if info["archivo"] in (None, logico) else f"  [archivo: {info['archivo']}]"
    print(f"  [{marca}] {logico:14s} -> {', '.join(info['casos'])}{real}")
if SOBRAN:
    print("\nArchivos que el eval set no pide (se ignoran):", sorted(SOBRAN))

if FALTAN:
    print(f"\nFaltan {len(FALTAN)} fotos de {len(PEDIDAS)}.")
    print("El notebook sigue, pero esos casos quedan marcados `sin_imagen` y NO entran")
    print("en ninguna media: una foto que no se subió no es un fallo del sistema, y")
    print("contarla como cero sería inventar un resultado.")
    print("\nQué escribir en cada hoja está en m4/eval_set/HOJAS_PARA_ESCRIBIR.md.")
    print(f"Súbanlas a {DIR_IMAGENES} con el nombre base del eval set (.jpg, .jpeg o .png).")
else:
    print(f"\nEstán las {len(PEDIDAS)} fotos: el scorecard de esta corrida será completo.")

12 casos  |  11 fotos distintas  |  11 archivos en la carpeta

  [ ok  ] vis-01.jpg     -> vis-01
  [ ok  ] vis-02.jpg     -> vis-02
  [ ok  ] vis-03.jpg     -> vis-03
  [ ok  ] vis-04.jpg     -> vis-04
  [ ok  ] vis-05.jpg     -> vis-05
  [ ok  ] vis-06.jpg     -> vis-06
  [ ok  ] vis-07.jpg     -> vis-07, vis-12
  [ ok  ] vis-08.jpg     -> vis-08
  [ ok  ] vis-09.jpg     -> vis-09
  [ ok  ] vis-10.jpg     -> vis-10  [archivo: vis-10.jpeg]
  [ ok  ] vis-11.jpg     -> vis-11

Están las 11 fotos: el scorecard de esta corrida será completo.


---
## 4 · Métricas y criterio de acierto

Las mismas funciones de M2 y M3, leídas del mismo sitio. El criterio de cada caso
viene declarado en el eval set, escrito antes de ejecutar.

In [10]:
import re
import unicodedata
from fractions import Fraction

MARCADOR_RESPUESTA = "Respuesta final:"

_PAT_RESPUESTA = re.compile(r"Respuesta\s+final\s*:\s*(.+)", re.IGNORECASE)
_PAT_PASO1     = re.compile(r"Paso\s*1\s*:", re.IGNORECASE)
_PAT_CATEGORIA = re.compile(r"Categor[ií]a\s*:\s*([a-zA-Z_]+)", re.IGNORECASE)
# La alternativa de fracción va primero: en "3/5" queremos capturar la fracción
# completa, no el "3" suelto.
_PAT_VALOR = re.compile(r"-?\d+(?:\.\d+)?\s*/\s*-?\d+(?:\.\d+)?|-?\d+(?:\.\d+)?")


def _sin_miles(texto):
    """Quita la coma como separador de miles. El corpus usa el punto como
    separador decimal y nunca la coma, así que la conversión no es ambigua."""
    return texto.replace(",", "")


def a_float(valor):
    if valor is None:
        return None
    try:
        return float(Fraction(valor)) if "/" in valor else float(valor)
    except (ValueError, ZeroDivisionError):
        return None


def valor_predicho(texto):
    """Extrae la respuesta del modelo en forma canónica.

    Prioridad 1: el número que sigue al marcador 'Respuesta final:'.
    Prioridad 2: el último número del texto.

    El segundo caso importa para que la comparación sea JUSTA: un modelo sin
    fine-tuning no conoce nuestro formato, y penalizarlo por eso mediría
    obediencia al formato, no capacidad matemática. Con el fallback medimos lo
    segundo; el apego al formato se mide aparte con `formato_valido`.
    """
    m = _PAT_RESPUESTA.search(texto)
    if m:
        linea = m.group(1).splitlines()[0]
        v = _PAT_VALOR.search(_sin_miles(linea))
        if v:
            return v.group(0).replace(" ", "")
    todos = _PAT_VALOR.findall(_sin_miles(texto))
    return todos[-1].replace(" ", "") if todos else None


def respuesta_correcta(generado, valor_oro, tol=1e-6):
    a = a_float(valor_predicho(generado))
    b = a_float(valor_oro)
    if a is None or b is None:
        return False
    return abs(a - b) <= tol * max(1.0, abs(b))


def formato_valido(texto):
    """¿El modelo produjo la estructura que le enseñamos?"""
    return bool(_PAT_PASO1.search(texto)) and bool(_PAT_RESPUESTA.search(texto))


def categoria_predicha(texto):
    m = _PAT_CATEGORIA.search(texto)
    return m.group(1).lower() if m else None


def _tokens(texto):
    t = unicodedata.normalize("NFKD", texto.lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    return re.findall(r"[a-z0-9]+|[^\sa-z0-9]", t)


def _lcs(a, b):
    """Longitud de la subsecuencia común más larga (programación dinámica)."""
    previa = [0] * (len(b) + 1)
    for x in a:
        actual = [0]
        for j, y in enumerate(b):
            actual.append(previa[j] + 1 if x == y else max(previa[j + 1], actual[j]))
        previa = actual
    return previa[-1]


def rouge_l(generado, referencia):
    """ROUGE-L (F1 sobre la subsecuencia común más larga).

    Se implementa a mano en lugar de usar `evaluate` para que el notebook no
    dependa de descargas en tiempo de ejecución y el número sea exactamente
    reproducible.
    """
    p, r = _tokens(generado), _tokens(referencia)
    if not p or not r:
        return 0.0
    l = _lcs(p, r)
    if l == 0:
        return 0.0
    prec, rec = l / len(p), l / len(r)
    return 2 * prec * rec / (prec + rec)


def evaluar_generacion(generados, registros):
    """Métricas de generación sobre un conjunto de ejemplos.

    exactitud       : ¿la respuesta final es numéricamente correcta?  <- la que importa
    formato_valido  : ¿respetó la estructura Paso N / Respuesta final?
    rouge_l         : ¿se parece el procedimiento al de referencia?
    long_media      : longitud media en palabras (detecta divagación)
    """
    assert len(generados) == len(registros)
    n = len(generados)
    correctas = [respuesta_correcta(g, r["valor"]) for g, r in zip(generados, registros)]
    formatos  = [formato_valido(g) for g in generados]
    rouges    = [rouge_l(g, r["salida"]) for g, r in zip(generados, registros)]
    return {
        "n": n,
        "exactitud": sum(correctas) / n,
        "formato_valido": sum(formatos) / n,
        "rouge_l": sum(rouges) / n,
        "long_media": sum(len(g.split()) for g in generados) / n,
        "_correctas": correctas,
    }


def tabla_metricas(antes, despues, titulo="Baseline vs Fine-tuned"):
    """Imprime la comparación en el formato que usaremos en el informe."""
    filas = [
        ("Exactitud de la respuesta", "exactitud", "{:.1%}"),
        ("Formato válido",            "formato_valido", "{:.1%}"),
        ("ROUGE-L del procedimiento", "rouge_l", "{:.3f}"),
        ("Longitud media (palabras)", "long_media", "{:.1f}"),
    ]
    ancho = 30
    print(titulo)
    print("=" * 68)
    print(f"{'Métrica':{ancho}s} {'Baseline':>12s} {'Fine-tuned':>12s} {'Δ':>10s}")
    print("-" * 68)
    for etiqueta, clave, fmt in filas:
        a, d = antes[clave], despues[clave]
        print(f"{etiqueta:{ancho}s} {fmt.format(a):>12s} {fmt.format(d):>12s} "
              f"{d - a:>+10.3f}")
    print("=" * 68)

### Dimensión 1 · Métricas clásicas

Dos métricas automáticas, y la segunda existe para demostrar por qué **no
basta**.

**1a · Exactitud de la respuesta final** — la métrica del dominio, heredada de
M1: se extrae el número que sigue a `Respuesta final:` y se compara
numéricamente con el esperado. Es objetiva y no admite discusión.

**1b · Similitud por embeddings** — la métrica clásica del lab de S05/S06.
La incluimos porque la entrega la pide y porque es útil para juzgar la
*explicación*, pero la sección siguiente muestra que en matemáticas es
peligrosa si se usa sola.

In [11]:
from sentence_transformers import SentenceTransformer
import numpy as np

# El MISMO modelo de embeddings de S05, S06 y S07 (multilingüe, pequeño).
st = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


def sim_embeddings(a, b):
    ea, eb = st.encode([a, b])
    return float(np.dot(ea, eb) / (np.linalg.norm(ea) * np.linalg.norm(eb)))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

#### Por qué la similitud sola no sirve para un tutor de matemáticas

Esta demostración es el equivalente, en nuestro dominio, del Lab A de S05
—donde BLEU premiaba la respuesta equivocada—. Aquí el que falla es el
embedding: dos respuestas idénticas salvo por el número obtienen una similitud
altísima, aunque una esté bien y la otra mal.

Es la razón por la que la Dimensión 3 usa un criterio numérico y no el
`sim >= 0.60` de la plantilla de clase.

In [12]:
referencia = ("Paso 1: Aplicamos el primer descuento: 200000 × 0.80 = 160000.\n"
              "Paso 2: El segundo se aplica sobre el precio ya rebajado: 160000 × 0.90 = 144000.\n"
              "Respuesta final: 144000 pesos")

candidatos = {
    "correcta (misma redacción)": referencia,
    "CORRECTA pero parafraseada": ("Primero quitamos el 20 por ciento, quedando 160000 pesos. "
                                   "Después restamos el 10 por ciento de esa cantidad. "
                                   "El precio final es de 144000 pesos."),
    "INCORRECTA (sumó los %)":    ("Paso 1: Sumamos los descuentos: 20% + 10% = 30%.\n"
                                   "Paso 2: 200000 × 0.70 = 140000.\n"
                                   "Respuesta final: 140000 pesos"),
}

print(f"{'candidato':<30} {'similitud':>10} {'¿correcta?':>12}")
print("-" * 54)
for nombre, texto in candidatos.items():
    s = sim_embeddings(texto, referencia)
    ok = respuesta_correcta(texto, "144000")
    print(f"{nombre:<30} {s:>10.3f} {str(ok):>12}")
print("-" * 54)
print("La respuesta INCORRECTA obtiene una similitud altísima: comparte")
print("estructura, vocabulario y casi todos los números. Con el umbral de 0.60")
print("del lab de clase, contaría como acierto. Por eso el criterio de dominio")
print("tiene que mirar el número, no el parecido del texto.")

candidato                       similitud   ¿correcta?
------------------------------------------------------
correcta (misma redacción)          1.000         True
CORRECTA pero parafraseada          0.642         True
INCORRECTA (sumó los %)             0.910        False
------------------------------------------------------
La respuesta INCORRECTA obtiene una similitud altísima: comparte
estructura, vocabulario y casi todos los números. Con el umbral de 0.60
del lab de clase, contaría como acierto. Por eso el criterio de dominio
tiene que mirar el número, no el parecido del texto.


### Dimensión 3 · Aciertos de dominio — el criterio, caso por caso

Aquí nos apartamos deliberadamente de la plantilla del laboratorio de clase, y
conviene justificarlo porque es una decisión de diseño, no un descuido.

El lab define el acierto como `similitud >= 0.60 **o** juez >= 4`. Esa regla
funciona en dominios abiertos, pero **en matemáticas cuenta como acierto una
respuesta con el número equivocado**:

```
Referencia : "Respuesta final: 144000 pesos"
Respuesta  : "Respuesta final: 140000 pesos"     <- INCORRECTA
similitud de embeddings ≈ 0.99  ->  la regla del lab la daría por buena
```

Nuestro dominio tiene **verdad objetiva**, así que el criterio debe usarla.
Cada caso del eval set trae su propia regla en el campo `evaluacion`, versionada
junto al eval set:

| Tipo | Cómo se decide el acierto |
|---|---|
| `numerico` | El valor tras `Respuesta final:` coincide numéricamente con el esperado (tolerancia 1e-6; acepta fracciones). |
| `contiene_alguna` | La respuesta menciona alguna de las claves esperadas (p. ej. "no es primo"). |
| `abstencion` | La respuesta reconoce el límite de su alcance. |
| `prohibido` | Lista de textos que invalidan el caso: si aparecen, el sistema cayó en la trampa. |

Además del acierto registramos dos señales más, porque **fallar y mentir no son
lo mismo**:

- **`abstuvo`** — admitió no tener la información. No es un acierto, pero es el
  comportamiento correcto cuando el dato no está.
- **`alucino`** — ni acertó ni se abstuvo: afirmó algo incorrecto con seguridad.
  Es el fallo grave, y el que más importa reportar en un tutor.

In [13]:
import re
import unicodedata


def _norm(texto):
    """Minúsculas y sin tildes: para comparar claves sin depender de la ortografía."""
    t = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in t if not unicodedata.combining(c))


# Marcadores de abstención honesta. Se detectan por separado del acierto porque
# "no lo sé" NO es una respuesta correcta, pero tampoco es una alucinación: es
# el comportamiento deseable cuando el sistema no tiene el dato.
#
# Cuidado con los marcadores genéricos: "falta" a secas produce falsos
# positivos constantes en este dominio ("faltan 1600 litros para llenarlo",
# "le hace falta recorrer 40 km"). Un falso positivo de abstención enmascara
# una alucinación, así que todos los marcadores son frases específicas.
MARCADORES_ABSTENCION = [
    "no tengo esa informacion", "no tengo informacion", "no dispongo",
    "no tengo acceso", "no puedo confirmar", "no aparece en", "no se especifica",
    "no se indica", "no se menciona", "no se proporciona", "no cuento con",
    "no esta en mis fuentes", "no tengo suficiente", "no lo se",
    "no puedo responder", "fuera de mi alcance", "no puedo ayudar",
    "no corresponde", "necesito mas datos", "necesito conocer",
    "falta un dato", "faltan datos", "falta informacion", "falta la distancia",
    "no es posible determinar", "no se puede determinar",
]


def se_abstuvo(respuesta):
    r = _norm(respuesta)
    return any(m in r for m in MARCADORES_ABSTENCION)


def evaluar_caso(caso, respuesta):
    """Dimensión 3: ¿esta respuesta cumple el criterio de ESTE caso?

    Por qué no usamos la regla genérica del lab de clase
    (`sim >= 0.60 or juez >= 4`): en matemáticas esa regla cuenta como acierto
    una respuesta con el número equivocado. "El resultado es 140000" y "el
    resultado es 144000" tienen similitud de embeddings ~0.99 y un juez pequeño
    les da la misma nota — pero una está mal. El criterio tiene que ser el del
    dominio, y aquí el dominio tiene verdad objetiva.

    Devuelve un dict con tres señales independientes:
      acierto  -> cumplió el criterio estricto del caso
      abstuvo  -> admitió no tener la información
      alucino  -> ni acertó ni se abstuvo (afirmó algo incorrecto con seguridad)
    """
    ev = caso["evaluacion"]
    r = _norm(respuesta)

    # 1) Texto prohibido: caer en la trampa invalida el caso de inmediato.
    prohibidos = [k for k in ev.get("prohibido", []) if _norm(k) in r]
    if prohibidos:
        return {"acierto": False, "abstuvo": False, "alucino": True,
                "motivo": f"cayó en la trampa: {prohibidos[0]!r}"}

    abstuvo = se_abstuvo(respuesta)
    tipo = ev["tipo"]

    if tipo.startswith("numerico"):
        acierto = respuesta_correcta(respuesta, ev["valor"])
        motivo = "número correcto" if acierto else f"esperaba {ev['valor']}, dio {valor_predicho(respuesta)}"
    elif tipo.startswith("contiene_alguna"):
        encontradas = [k for k in ev["claves"] if _norm(k) in r]
        acierto = bool(encontradas)
        motivo = f"contiene {encontradas[0]!r}" if acierto else "no menciona ninguna clave esperada"
    elif tipo == "abstencion":
        acierto = abstuvo or any(_norm(k) in r for k in ev.get("claves", []))
        motivo = "reconoció el límite" if acierto else "respondió como si estuviera en su dominio"
    else:
        raise ValueError(f"tipo de evaluación desconocido: {tipo}")

    # Alucinación = afirmó algo concreto y equivocado sin admitir que no sabía.
    alucino = (not acierto) and (not abstuvo)
    return {"acierto": acierto, "abstuvo": abstuvo, "alucino": alucino, "motivo": motivo}

---
## 5 · Etapa 1 · `ver()` — la foto se convierte en pasos

El VLM **solo transcribe**, y se le pide explícitamente que **no corrija**: un
modelo instruido tiende a "arreglar" la cuenta mal al copiarla, y entonces el
error que había que encontrar desaparece antes de llegar al tutor.

In [14]:
import json
import time

import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

VLM_ID = "Qwen/Qwen2-VL-2B-Instruct"
MAX_PIXELS = 1024 * 1024          # tope de tokens visuales por imagen

vlm_proc = AutoProcessor.from_pretrained(VLM_ID, max_pixels=MAX_PIXELS)
vlm = Qwen2VLForConditionalGeneration.from_pretrained(
    VLM_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None).eval()
print("VLM cargado:", VLM_ID)
if DEVICE == "cuda":
    print(f"GPU ocupada: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# La instrucción (INSTRUCCION_VER) y el parser tolerante (json_de) vienen del
# módulo `vision.py` de la celda 3.1, donde están probados.
print("\nInstrucción de lectura:\n", INSTRUCCION_VER)


@torch.no_grad()
def ver(imagen, instruccion=INSTRUCCION_VER, max_new_tokens=320, muestrear=False):
    """imagen -> (texto crudo, dict o None, segundos). La etapa 1 del pipeline.

    Con `muestrear=True` genera con temperatura en vez de greedy: es la segunda
    opinión que usa `leer_dos_veces`.
    """
    t0 = time.perf_counter()
    msgs = [{"role": "user", "content": [{"type": "image"},
                                         {"type": "text", "text": instruccion}]}]
    prompt = vlm_proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ent = vlm_proc(text=[prompt], images=[imagen], return_tensors="pt").to(vlm.device)
    kwargs = (dict(do_sample=True, temperature=0.8, top_p=0.95) if muestrear
              else dict(do_sample=False))
    out = vlm.generate(**ent, max_new_tokens=max_new_tokens,
                       return_dict_in_generate=True, output_scores=True, **kwargs)
    nuevos = out.sequences[:, ent["input_ids"].shape[1]:]
    texto = vlm_proc.batch_decode(nuevos, skip_special_tokens=True)[0].strip()

    # Confianza del modelo en los tokens que son NÚMEROS. Un dígito leído de una
    # cifra nítida sale con más probabilidad que uno rellenado de memoria, así que
    # esto es una señal sobre la lectura que no depende de mirar la imagen.
    confianza = None
    try:
        scores = vlm.compute_transition_scores(out.sequences, out.scores,
                                               normalize_logits=True)[0]
        tok = vlm_proc.tokenizer
        de_numeros = [float(s) for s, i in zip(scores, nuevos[0])
                      if any(c.isdigit() for c in tok.decode([int(i)]))]
        confianza = sum(de_numeros) / len(de_numeros) if de_numeros else None
    except Exception as e:                      # nunca tumbar la lectura por esto
        print(f"    (sin confianza de tokens: {type(e).__name__})")

    return texto, json_de(texto), time.perf_counter() - t0, confianza


# Una pregunta distinta, en una llamada aparte: no se le pide transcribir, solo
# juzgar si la hoja se lee. Separar las dos preguntas es el punto — en la corrida
# del 24-10 la transcripción venía arrastrada por el prior del ejercicio y el
# modelo rellenó los números; preguntado sin transcribir, el prior no tiene dónde
# agarrarse igual.
INSTRUCCION_LEGIBILIDAD = (
    "Mira esta foto de una hoja escrita a mano. NO la transcribas. Juzga solo si "
    "se puede leer con claridad cada cifra y cada renglón.\n"
    "Responde SOLO un JSON: {\"legible\": true|false, \"motivo\": \"...\"}\n"
    "Pon `legible: false` si la foto está movida, desenfocada, con reflejos o "
    "demasiado lejos para distinguir los números con seguridad."
)


@torch.no_grad()
def preguntar_legibilidad(imagen):
    """-> (legible: bool|None, motivo). None si la respuesta no se pudo parsear."""
    msgs = [{"role": "user", "content": [{"type": "image"},
                                         {"type": "text", "text": INSTRUCCION_LEGIBILIDAD}]}]
    prompt = vlm_proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ent = vlm_proc(text=[prompt], images=[imagen], return_tensors="pt").to(vlm.device)
    out = vlm.generate(**ent, max_new_tokens=80, do_sample=False)
    texto = vlm_proc.batch_decode(out[:, ent["input_ids"].shape[1]:],
                                  skip_special_tokens=True)[0].strip()
    js = json_de(texto) or {}
    legible = js.get("legible")
    return (bool(legible) if isinstance(legible, bool) else None,
            str(js.get("motivo", ""))[:200])


def leer_dos_veces(imagen, archivo):
    """La salvaguarda contra la lectura inventada.

    Se lee la foto dos veces: una greedy (la que se usa) y otra con muestreo. Si
    las dos no coinciden en el número de pasos y en los números transcritos, la
    lectura no es de fiar y el tutor recibirá "la hoja NO se pudo leer" en vez de
    un procedimiento inventado.

    Por qué así y no con una métrica de nitidez de la imagen: un umbral de
    desenfoque habría que calibrarlo, y calibrarlo con estas 11 fotos sería
    ajustarlo al eval set. Que dos muestreos del mismo modelo sobre la misma foto
    discrepen es una señal del propio modelo —está inventando— y no necesita
    ninguna constante elegida a mano.
    """
    crudo, js, seg, confianza = ver(imagen)
    lectura = {"crudo": crudo, "json": js, "segundos": round(seg, 2),
               "ok": js is not None, "archivo": archivo,
               "confianza_numeros": confianza}

    # Señal 2 · la misma pregunta, otra vez, con muestreo.
    _, js2, seg2, _ = ver(imagen, muestrear=True)
    lectura["consistente"] = lecturas_consistentes(lectura, {"json": js2})
    lectura["json_segunda"] = js2

    # Señal 3 · otra pregunta: ¿se lee? Sin transcribir.
    legible, motivo = preguntar_legibilidad(imagen)
    lectura["legible_declarada"] = legible
    lectura["motivo_legibilidad"] = motivo

    lectura["segundos_verificacion"] = round(seg2, 2)
    return lectura

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

VLM cargado: Qwen/Qwen2-VL-2B-Instruct
GPU ocupada: 4.9 GB

Instrucción de lectura:
 Transcribe EXACTAMENTE lo que está escrito a mano en esta hoja. No corrijas ningún error: si una cuenta está mal, cópiala mal, tal como aparece.
Devuelve SOLO un JSON con esta forma:
{"enunciado": "...", "pasos": ["...", "..."], "respuesta_final": "..."}
Un elemento de `pasos` por cada línea escrita.
Si la hoja no se puede leer, o no contiene un procedimiento matemático, devuelve {"ilegible": true, "motivo": "..."}.


### 5.1 · Calibración de la detección de hojas ilegibles

Con un conjunto **aparte del eval set**: 6 hojas sintéticas × 3 versiones
(nítida, borrosa leve, borrosa fuerte). Mide si la salvaguarda separa, sin
convertir `vis-10` en dato de ajuste.

El límite es grande y está dicho en la celda: es texto **impreso**. Un buen
resultado aquí no autoriza a afirmar que funciona con letra manuscrita.

In [15]:
# CALIBRACIÓN de la detección de hojas ilegibles · con datos que NO son el eval set.
#
# Por qué aquí y no mirando `vis-10`: la detección tiene un umbral (la confianza
# en los tokens numéricos, marcada con mediana − k·MAD). Ajustar ese umbral
# mirando `vis-10` convertiría un caso de evaluación en conjunto de calibración, y
# entonces el 0/3 adversarial dejaría de significar nada.
#
# El conjunto lo genera `scripts/calibracion_legibilidad_fuente.py`: 6 hojas × 3
# versiones (nítida, borrosa leve, borrosa fuerte). La leve es la que importa: es
# el análogo de `vis-10`, donde el VLM alcanza a leer parte y rellena el resto.
#
# LÍMITE, y es grande: son hojas con texto IMPRESO. Un buen resultado aquí NO
# autoriza a decir que la salvaguarda funciona con letra manuscrita. La validación
# de verdad necesita fotos de las hojas dev-* escritas a mano, nítidas y
# desenfocadas, y mientras no existan la salvaguarda está sin validar.
DIR_CALIBRACION = Path("m4/eval_set/calibracion")
MANIFIESTO = DIR_CALIBRACION / "manifiesto.json"

if not MANIFIESTO.exists():
    print("No está el conjunto de calibración. Genérenlo con")
    print("  python scripts/calibracion_legibilidad_fuente.py")
    print("y súbanlo a Drive. Sin él, la salvaguarda queda sin medir.")
else:
    pares = json.loads(MANIFIESTO.read_text(encoding="utf-8"))["pares"]
    cal = {}
    for r in pares:
        cal[r["archivo"]] = leer_dos_veces(cargar_imagen(DIR_CALIBRACION / r["archivo"]),
                                           r["archivo"])
        cal[r["archivo"]]["etiqueta"] = r["etiqueta"]

    # El criterio de atípicos se calcula DENTRO del conjunto de calibración.
    conf = {k: v["confianza_numeros"] for k, v in cal.items()
            if v.get("confianza_numeros") is not None}
    at = atipicos_por_mad(conf)
    for k in cal:
        cal[k]["confianza_atipica"] = k in at

    print(f"{'archivo':28s}{'etiqueta':17s}{'conf.núm':>10s}{'¿se lee?':>10s}   detectada")
    filas_cal = []
    for k in sorted(cal):
        v = cal[k]
        no_fiable, _ = lectura_no_fiable(v)
        c = v.get("confianza_numeros")
        filas_cal.append({"archivo": k, "etiqueta": v["etiqueta"],
                          "legible": v["etiqueta"] == "legible",
                          "confianza": c, "atipica": v["confianza_atipica"],
                          "legible_declarada": v.get("legible_declarada"),
                          "consistente": v.get("consistente"),
                          "detectada_no_fiable": no_fiable})
        print(f"{k:28s}{v['etiqueta']:17s}"
              f"{(f'{c:.3f}' if c is not None else '—'):>10s}"
              f"{({True: 'sí', False: 'NO', None: '?'}[v.get('legible_declarada')]):>10s}"
              f"   {'NO FIABLE' if no_fiable else 'ok'}")

    CAL = pd.DataFrame(filas_cal)
    CAL.to_csv("m4/calibracion_legibilidad.csv", index=False)

    ileg = CAL[~CAL.legible]
    leg = CAL[CAL.legible]
    print()
    print("=" * 78)
    print("QUÉ TAN BIEN SEPARA · sobre calibración, nunca sobre el eval set")
    print("=" * 78)
    print(f"  ilegibles detectadas   : {int(ileg.detectada_no_fiable.sum())}/{len(ileg)}"
          "   (cuantas más, mejor)")
    for nivel in ("ilegible_leve", "ilegible_fuerte"):
        g = CAL[CAL.etiqueta == nivel]
        if len(g):
            print(f"      {nivel:16s}: {int(g.detectada_no_fiable.sum())}/{len(g)}")
    print(f"  legibles marcadas mal  : {int(leg.detectada_no_fiable.sum())}/{len(leg)}"
          "   (cuantas menos, mejor)")
    print("=" * 78)
    print()
    print("Cómo leerlo:")
    print("  · Si NO caza las `ilegible_fuerte`, la salvaguarda no funciona en absoluto.")
    print("  · Si caza las fuertes pero no las leves, cazará lo evidente y dejará pasar")
    print("    justo el caso que importa, que es el de `vis-10`.")
    print("  · Si marca legibles, el coste es real: a una hoja buena se le dirá que no")
    print("    se pudo leer y su caso fallará por la salvaguarda.")
    print()
    print("Y el límite que no se puede saltar: esto es texto IMPRESO. Con letra")
    print("manuscrita el problema es más difícil incluso en nítido. Mientras no haya")
    print("fotos a mano de las hojas dev-*, la salvaguarda sigue SIN VALIDAR y así hay")
    print("que reportarla.")

No está el conjunto de calibración. Genérenlo con
  python scripts/calibracion_legibilidad_fuente.py
y súbanlo a Drive. Sin él, la salvaguarda queda sin medir.


### 5.2 · Puerta piloto · tres imágenes

Antes de gastar la corrida completa. El riesgo que puede costar la entrega es que
un modelo de 2B no lea letra manuscrita; más vale descubrirlo en tres fotos que a
los cuarenta minutos. Las tres están elegidas para que cada una pueda fallar de
una forma distinta, y la celda dice qué hacer en cada caso.

In [16]:
# PUERTA PILOTO · tres imágenes antes de gastar la corrida completa.
#
# El riesgo que puede costar la entrega es que un VLM de 2B no lea letra
# manuscrita. Más vale descubrirlo en tres fotos que a los cuarenta minutos.
#
# Las tres están elegidas para que cada una pueda fallar de forma distinta:
PILOTO = {
    "vis-01": "letra manuscrita con una cuenta mal: ¿copia el error o lo 'arregla'?",
    "vis-07": "procedimiento correcto: ¿no inventa un error al transcribir?",
    "vis-10": "foto movida a propósito: ¿dice que no puede leerla?",
}

lecturas_piloto = {}
for cid, que_prueba in PILOTO.items():
    caso = CASOS_M4[cid]
    logico = archivo_de(caso)
    real = PEDIDAS[logico]["archivo"]
    print(f"[{cid}] {que_prueba}")
    if real is None:
        print(f"      falta la foto de {logico}\n")
        continue
    lectura = leer_dos_veces(cargar_imagen(DIR_IMAGENES / real), real)
    js, seg = lectura["json"], lectura["segundos"]
    lecturas_piloto[logico] = lectura
    print(f"      archivo: {real} | {seg:.1f}s | json parseado: {js is not None}"
          f" | dos lecturas coinciden: {lectura['consistente']}")
    no_fiable, motivo = lectura_no_fiable(lectura)
    if no_fiable:
        print(f"      LECTURA NO FIABLE -> {motivo}")
        print("      el tutor recibirá 'la hoja NO se pudo leer', no un procedimiento")

    # Se imprimen los pasos YA NUMERADOS, que es exactamente lo que verá el tutor.
    # La numeración incluye la respuesta final como último paso, igual que el eval
    # set: si aquí no coincide, el paso que hay que señalar no existe para el tutor.
    if js and js.get("ilegible"):
        print(f"      ilegible: {js.get('motivo')}")
    elif js:
        for i, paso in enumerate(pasos_numerados(lectura), 1):
            print(f"      Paso {i}: {paso}")
    else:
        print("      crudo:", crudo[:200].replace("\n", " | "))

    esperados = caso["pasos"]
    if esperados:
        vistos = len(pasos_numerados(lectura))
        print(f"      la hoja tiene {len(esperados)} pasos; el tutor verá {vistos}"
              + ("  <-- NO COINCIDE" if vistos != len(esperados) else ""))
        print(f"      diagnóstico esperado: {diagnostico_esperado(caso)!r}")
        print(f"      ¿transcripción fiel?: {lectura_fiel(caso, lectura)}")
    else:
        print(f"      diagnóstico esperado: {diagnostico_esperado(caso)!r} (caso adversarial)")
    print()

print("DECISIÓN: ¿los pasos transcritos coinciden con lo que dice la hoja, y su")
print("          numeración con la que cuenta el eval set?")
print("  Si sí  -> sigan a la transcripción completa.")
print("  Si no  -> paren aquí. Repliegues, en este orden:")
print("     1. subir el contraste y el tamaño en el preproceso;")
print("     2. un OCR clásico (pytesseract) y dejar al VLM solo estructurando el texto;")
print("     3. declarar la lectura como el cuello de botella y medirlo, que también es")
print("        un resultado. Lo que NO sirve es correr las 11 y enterarse al final.")
print("\nSi aparece 'NO COINCIDE', paren igualmente: ese caso sería inacertable por")
print("construcción y contarlo como fallo del tutor sería un error de medición.")

[vis-01] letra manuscrita con una cuenta mal: ¿copia el error o lo 'arregla'?


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


      archivo: vis-01.jpg | 7.4s | json parseado: True | dos lecturas coinciden: True
      Paso 1: Multiplicamos las cajas por lápices de cada una: 24x13
      Paso 2: 24x13 = 302
      Paso 3: Respuesta final: 302 lápices
      la hoja tiene 3 pasos; el tutor verá 3
      diagnóstico esperado: 'paso 2'
      ¿transcripción fiel?: True

[vis-07] procedimiento correcto: ¿no inventa un error al transcribir?
      archivo: vis-07.jpg | 5.1s | json parseado: True | dos lecturas coinciden: True
      Paso 1: Sumamos las 3 notas: 4.0 + 3.5 + 4.5 = 12
      Paso 2: Dividimos entre 3: 12 / 3 = 4
      Paso 3: Respuesta final: 4.0
      la hoja tiene 3 pasos; el tutor verá 3
      diagnóstico esperado: 'sin error'
      ¿transcripción fiel?: True

[vis-10] foto movida a propósito: ¿dice que no puede leerla?
      archivo: vis-10.jpeg | 4.7s | json parseado: True | dos lecturas coinciden: True
      Paso 1: Multiplicamos las cajas por la cantidad de piezas de cada caja. 24 x 13 = 306
      Paso

### 5.3 · Transcripción completa

Se guarda en disco **antes** de soltar el VLM: si Colab se desconecta no se pierde
nada, y la fase de diagnóstico se puede repetir sin volver a leer las fotos.

In [17]:
# Transcripción completa, con caché en disco ANTES de soltar el VLM.
#
# Si Colab se desconecta después de esta celda no se pierde nada, y el diagnóstico
# se puede reejecutar o afinar sin volver a pasar el VLM por las fotos.
Path("m4").mkdir(exist_ok=True)
RUTA_LECTURA = Path("m4/lecturas_m4.json")
lecturas = json.loads(RUTA_LECTURA.read_text(encoding="utf-8")) if RUTA_LECTURA.exists() else {}

if len(FALTAN) == len(PEDIDAS):
    raise SystemExit("No hay ninguna foto en m4/eval_set/imagenes: no hay nada que leer. "
                     "Súbanlas y vuelvan a ejecutar desde la celda 3.4.")

# Las lecturas se guardan con el nombre LÓGICO del eval set (`vis-10.jpg`) y
# registran aparte el archivo real que se leyó (`vis-10.jpeg`). Así el archivo de
# lecturas no cambia de forma si mañana la foto llega con otra extensión.
lecturas.update(lecturas_piloto)        # el piloto ya leyó tres: no se repiten

# La caché se reusa solo si trae la verificación de consistencia: una lectura
# guardada antes de la salvaguarda no se puede dar por fiable, así que se relee.
for logico, info in sorted(PEDIDAS.items()):
    if info["archivo"] is None:
        continue
    previa = lecturas.get(logico)
    if previa is not None and "consistente" in previa:
        continue
    lecturas[logico] = leer_dos_veces(cargar_imagen(DIR_IMAGENES / info["archivo"]),
                                      info["archivo"])
    l = lecturas[logico]
    marca = "json ok" if l["ok"] else "SIN json"
    aviso = "" if l["consistente"] else "  <-- LAS DOS LECTURAS NO COINCIDEN"
    print(f"  {info['archivo']:14s} {l['segundos']:5.1f}s  {marca}{aviso}")

# Señal 4 · confianza atípica en los números, RELATIVA al resto del conjunto.
# El criterio es mediana − 3·MAD, la convención robusta de siempre. No hay ningún
# umbral calibrado con estas fotos, y no debe calibrarse con ellas: `vis-10` es un
# caso de evaluación, no de ajuste.
confianzas = {k: v["confianza_numeros"] for k, v in lecturas.items()
              if v.get("confianza_numeros") is not None}
atipicas = atipicos_por_mad(confianzas)
for k in lecturas:
    lecturas[k]["confianza_atipica"] = k in atipicas

RUTA_LECTURA.write_text(json.dumps(lecturas, ensure_ascii=False, indent=2), encoding="utf-8")

print("\nLAS CUATRO SEÑALES DE FIABILIDAD, foto por foto")
print(f"{'foto':14s}{'conf.núm':>10s}{'atípica':>9s}{'2ª lectura':>12s}"
      f"{'¿se lee?':>10s}   veredicto")
for k in sorted(lecturas):
    v = lecturas[k]
    c = v.get("confianza_numeros")
    no_fiable, motivo = lectura_no_fiable(v)
    print(f"{k:14s}{(f'{c:.3f}' if c is not None else '—'):>10s}"
          f"{('sí' if v.get('confianza_atipica') else 'no'):>9s}"
          f"{('coincide' if v.get('consistente') else 'DISCREPA'):>12s}"
          f"{({True: 'sí', False: 'NO', None: '?'}[v.get('legible_declarada')]):>10s}"
          f"   {'NO FIABLE' if no_fiable else 'ok'}")

print("\nLa columna de confianza se imprime para que la vean, no para ajustar nada:")
print("si quieren saber si separa las fotos ilegibles, hace falta un conjunto de")
print("calibración aparte (ver el README de la corrida 03). Mover el criterio hasta")
print("que una foto del eval set caiga del lado deseado la convierte en ajuste.")

no_fiables = [k for k, v in lecturas.items() if lectura_no_fiable(v)[0]]
if no_fiables:
    print(f"\nLecturas NO fiables: {', '.join(sorted(no_fiables))}")
    print("En esos casos el tutor recibirá 'la hoja NO se pudo leer' en lugar de una")
    print("transcripción. No se acredita ningún caso por esto: solo se evita juzgar")
    print("al tutor sobre un procedimiento que el VLM se inventó.")
else:
    print("\nNinguna lectura se marcó como no fiable.")
    print("Si alguna de las fotos era ilegible a propósito, esto es un FALSO NEGATIVO")
    print("de la salvaguarda y hay que decirlo: ese caso no mide al tutor.")

RUTA_LECTURA.write_text(json.dumps(lecturas, ensure_ascii=False, indent=2), encoding="utf-8")
leidas = sum(1 for v in lecturas.values() if v["ok"])
print(f"\n{len(lecturas)} imágenes transcritas, {leidas} con JSON válido -> {RUTA_LECTURA}")
print(f"Latencia media de lectura: {np.mean([v['segundos'] for v in lecturas.values()]):.1f} s")


LAS CUATRO SEÑALES DE FIABILIDAD, foto por foto
foto            conf.núm  atípica  2ª lectura  ¿se lee?   veredicto
vis-01.jpg        -0.004       no    coincide        sí   ok
vis-02.jpg             —       no    coincide         ?   ok
vis-03.jpg             —       no    coincide         ?   ok
vis-04.jpg             —       no    coincide         ?   ok
vis-05.jpg             —       no    coincide         ?   ok
vis-06.jpg             —       no    coincide         ?   ok
vis-07.jpg        -0.000       no    coincide        sí   ok
vis-08.jpg             —       no    coincide         ?   ok
vis-09.jpg             —       no    coincide         ?   ok
vis-10.jpg        -0.101       no    coincide        sí   ok
vis-11.jpg             —       no    coincide         ?   ok

La columna de confianza se imprime para que la vean, no para ajustar nada:
si quieren saber si separa las fotos ilegibles, hace falta un conjunto de
calibración aparte (ver el README de la corrida 03). Mover el 

In [18]:
# Liberar la GPU antes de cargar el tutor. La transcripción ya está en disco, así
# que el VLM no vuelve a hacer falta en esta corrida.
import gc

del vlm, vlm_proc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU tras liberar el VLM: {torch.cuda.memory_allocated() / 1e9:.1f} GB "
          f"de {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
else:
    print("Sin GPU: nada que liberar.")

GPU tras liberar el VLM: 0.5 GB de 16 GB


---
## 6 · Etapa 2 · El sistema de M3, sin reescribir

Corpus, chunking y retrieval idénticos a S07/S08; el agente con herramientas,
idéntico a S10. El código sale de `scripts/agente/*.py`, donde está probado con
`pytest`: lo que se prueba es exactamente lo que se ejecuta.

Lo único nuevo es que la pregunta llega acompañada de lo que se leyó en la foto,
y que **se le exige una conclusión explícita**.

### El formato de salida, y por qué se cambió

En la corrida del 22-10 (archivada en `m4/corridas/corrida-01_2026-10-22/`) el
tutor no revisaba: **reescribía** el procedimiento en el formato `Paso 1: …
Respuesta final: …` con el que se entrenó la LoRA de M1. Ante la hoja de `vis-02`
respondió el ejercicio bien resuelto, sin decir nunca qué paso estaba mal.

Así que ahora la última línea tiene que ser una de estas cinco:

| Conclusión | Cuándo | Bloque del eval set |
|---|---|---|
| `Diagnóstico: Paso N` | un paso está mal | `localizacion`, `compensado` |
| `Diagnóstico: sin error` | todos los pasos están bien | `sin_error` |
| `Diagnóstico: ilegible` | la hoja no se puede leer | `adversarial` (`vis-10`) |
| `Diagnóstico: fuera de dominio` | la hoja no es matemáticas | `adversarial` (`vis-11`) |
| `Diagnóstico: dato no disponible` | se pide algo que no está escrito | `adversarial` (`vis-12`) |

**El criterio lee solo esa línea.** La explicación se conserva y se mide aparte,
nunca como acierto: era lo que invalidaba la corrida anterior.

In [19]:
corpus = [
    {"id": "siee", "fuente": "Sistema Institucional de Evaluación (SIEE) 2026 — capítulo 3",
     "texto": (
        "La valoración del periodo académico en el área de matemáticas se compone de cuatro "
        "elementos con la siguiente ponderación: el examen final del periodo equivale al 40 por "
        "ciento de la nota, los talleres al 25 por ciento, los quices al 20 por ciento y el "
        "trabajo en clase al 15 por ciento. "
        "La escala institucional va de 1.0 a 5.0 y la nota mínima aprobatoria es 3.0. "
        "Cada estudiante dispone de dos oportunidades de recuperación por periodo; la nota "
        "máxima que puede obtenerse en una recuperación es 3.5. "
        "La inasistencia no justificada a un examen se califica con 1.0, salvo que el acudiente "
        "presente excusa dentro de los tres días hábiles siguientes.")},

    {"id": "plan_area", "fuente": "Plan de área de matemáticas 2026 — secuencia por grados",
     "texto": (
        "La secuencia del área organiza los contenidos así. En grado sexto se trabajan las "
        "operaciones con números naturales, la divisibilidad y una introducción a las fracciones. "
        "En grado séptimo se abordan los números enteros, la proporcionalidad directa e inversa y "
        "los porcentajes. "
        "En grado octavo se introducen las ecuaciones de primer grado con una incógnita, junto con "
        "el lenguaje algebraico y la traducción de enunciados verbales a expresiones matemáticas. "
        "En grado noveno se amplía a sistemas de dos ecuaciones lineales y a la función lineal. "
        "En grado décimo se trabajan la trigonometría y la geometría analítica. "
        "El área recomienda dedicar al menos dos sesiones a la traducción de enunciados verbales "
        "antes de introducir el algoritmo de despeje.")},

    {"id": "protocolo", "fuente": "Protocolo de errores frecuentes en matemáticas — 2026",
     "texto": (
        "Este protocolo recoge los errores más frecuentes detectados en las pruebas diagnósticas y "
        "la estrategia recomendada para cada uno. "
        "Descuentos sucesivos: los estudiantes suman los porcentajes, de modo que un 20 por ciento "
        "seguido de un 10 por ciento lo tratan como un 30 por ciento único. La estrategia "
        "recomendada es trabajar con el factor multiplicativo del precio que queda, es decir 0.80 y "
        "luego 0.90, organizados en una tabla de dos pasos, en lugar de sumar los porcentajes. "
        "Fracción del resto: cuando un enunciado dice 'un tercio de lo que quedaba', los estudiantes "
        "aplican la fracción al total original. Se recomienda pedir que escriban explícitamente "
        "cuánto queda antes de aplicar la segunda fracción. "
        "Proporcionalidad inversa: ante 'más obreros, menos días' aplican regla de tres directa. La "
        "estrategia es calcular primero el trabajo total en días-obrero. "
        "Redondeo contextual: al dividir personas entre vehículos o cajas entre camiones, el "
        "resultado debe redondearse hacia arriba porque no existen fracciones de vehículo.")},

    {"id": "formulario", "fuente": "Formulario de geometría — guía del estudiante",
     "texto": (
        "Áreas. Rectángulo: base por altura. Cuadrado: lado al cuadrado. Triángulo: base por altura "
        "dividido entre dos. Trapecio: la semisuma de las bases multiplicada por la altura. "
        "Círculo: pi por el radio al cuadrado. "
        "Perímetros. Rectángulo: dos por la suma de base y altura. Cuadrado: cuatro por el lado. "
        "Circunferencia: dos por pi por el radio. "
        "Volúmenes. Cubo: arista al cubo. Prisma rectangular: largo por ancho por alto. "
        "Cilindro: pi por el radio al cuadrado por la altura. "
        "Teorema de Pitágoras: en un triángulo rectángulo, el cuadrado de la hipotenusa es igual a "
        "la suma de los cuadrados de los catetos. "
        "Para figuras compuestas se descompone la figura en formas simples y se suman o restan sus "
        "áreas según corresponda.")},

    {"id": "jerarquia", "fuente": "Guía de operaciones — jerarquía y signos",
     "texto": (
        "El orden para resolver una expresión con varias operaciones es: primero los paréntesis, "
        "empezando por los más internos; después las potencias y raíces; luego las multiplicaciones "
        "y divisiones de izquierda a derecha; y por último las sumas y restas, también de izquierda "
        "a derecha. "
        "Un error común es resolver de izquierda a derecha sin respetar la jerarquía. "
        "Reglas de signos: al multiplicar o dividir, signos iguales dan positivo y signos distintos "
        "dan negativo. Restar un número negativo equivale a sumar su valor absoluto. "
        "La división entre cero no está definida: no existe ningún número que multiplicado por cero "
        "dé un resultado distinto de cero.")},

    {"id": "fracciones", "fuente": "Guía de fracciones, decimales y porcentajes",
     "texto": (
        "Para sumar o restar fracciones con distinto denominador se busca el mínimo común múltiplo y "
        "se convierten ambas al denominador común. Un error frecuente es sumar numeradores y "
        "denominadores por separado. "
        "Para multiplicar fracciones se multiplican numeradores entre sí y denominadores entre sí. "
        "Para dividir se multiplica por la fracción inversa. "
        "Calcular un porcentaje de una cantidad equivale a multiplicar por el porcentaje dividido "
        "entre cien. Para hallar el precio tras un aumento del quince por ciento se multiplica por "
        "1.15; para recuperar el precio anterior a partir del actual se DIVIDE entre 1.15, no se "
        "resta el quince por ciento. "
        "Una división entre un número menor que uno da un resultado mayor que el dividendo.")},
]

print(f"Documentos: {len(corpus)}")
print("Caracteres por documento:", [len(d["texto"]) for d in corpus])

Documentos: 6
Caracteres por documento: [648, 746, 1029, 718, 649, 723]


In [20]:
CHUNK_SIZE = 400
OVERLAP    = 60


def partir_en_chunks(texto, size=CHUNK_SIZE, overlap=OVERLAP):
    chunks, inicio = [], 0
    while inicio < len(texto):
        chunks.append(texto[inicio:inicio + size])
        inicio += size - overlap
    return chunks


chunks, metadatos, ids_chunks = [], [], []
for doc in corpus:
    for j, ch in enumerate(partir_en_chunks(doc["texto"])):
        chunks.append(ch)
        metadatos.append({"fuente": doc["fuente"], "doc_id": doc["id"]})
        ids_chunks.append(f"{doc['id']}_chunk{j}")

print(f"{len(corpus)} documentos -> {len(chunks)} chunks")
print(f"Caracteres por chunk: min={min(len(c) for c in chunks)} max={max(len(c) for c in chunks)}")
print("\nEjemplo de chunk. Miren DÓNDE cortó: ¿partió una idea por la mitad?\n")
print(repr(chunks[2]))

6 documentos -> 17 chunks
Caracteres por chunk: min=9 max=400

Ejemplo de chunk. Miren DÓNDE cortó: ¿partió una idea por la mitad?

'La secuencia del área organiza los contenidos así. En grado sexto se trabajan las operaciones con números naturales, la divisibilidad y una introducción a las fracciones. En grado séptimo se abordan los números enteros, la proporcionalidad directa e inversa y los porcentajes. En grado octavo se introducen las ecuaciones de primer grado con una incógnita, junto con el lenguaje algebraico y la tradu'


In [21]:
import chromadb

cliente = chromadb.Client()
try:
    cliente.delete_collection("tutor_m4")
except Exception:
    pass
coleccion = cliente.create_collection("tutor_m4", metadata={"hnsw:space": "cosine"})
coleccion.add(ids=ids_chunks, documents=chunks, metadatas=metadatos,
              embeddings=st.encode(chunks, show_progress_bar=False).tolist())
POS = {cid: i for i, cid in enumerate(ids_chunks)}


def buscar_densa(consulta, k=3):
    r = coleccion.query(query_embeddings=st.encode([consulta]).tolist(),
                        n_results=min(k, len(chunks)))
    return [POS[cid] for cid in r["ids"][0]]


print("Índice:", coleccion.count(), "chunks (mismo corpus y mismo chunking que S07/S08)")

Índice: 17 chunks (mismo corpus y mismo chunking que S07/S08)


In [22]:
"""
Tool Registry: herramientas tipadas, con esquema generado y ejecución segura.

Decisión de diseño: el esquema JSON de cada herramienta NO se escribe a mano.
Se deriva de la firma de la función (tipos) y de su docstring (descripciones),
con el mismo formato que espera la plantilla de chat de Qwen2.5 para tool
calling (`{"type": "function", "function": {...}}`). Así la descripción que ve
el modelo y el código que se ejecuta no pueden divergir, y añadir una
herramienta es escribir UNA función decorada: el núcleo del agente no cambia.

El registro separa tres responsabilidades que el agente nunca mezcla:
  esquemas()  -> lo que se le MUESTRA al modelo
  validar()   -> convertir los argumentos que PROPONE el modelo a los tipos reales
  ejecutar()  -> correr la función y devolver un resultado tipado (nunca lanza)
"""

from __future__ import annotations

import inspect
import json
import re
import time
import typing
from dataclasses import asdict, dataclass, field
from fractions import Fraction
from typing import Any, Callable, Literal, get_args, get_origin


class ErrorHerramienta(Exception):
    """Error esperado de dominio (p. ej. división entre cero). Su mensaje se
    devuelve al modelo como observación, para que lo explique o corrija."""


# --------------------------------------------------------------------------
# Tipos de datos del protocolo agente <-> herramientas
# --------------------------------------------------------------------------

@dataclass
class LlamadaHerramienta:
    nombre: str
    argumentos: dict
    formato: str = "nativo"        # nativo (<tool_call>) | recuperado (JSON suelto)


@dataclass
class ResultadoHerramienta:
    herramienta: str
    argumentos: dict
    ok: bool
    resultado: str | None = None   # texto legible para el modelo
    valor: str | None = None       # forma canónica ("73/72", "9.316") para métricas
    error: str | None = None
    tipo_error: str | None = None  # desconocida | argumentos | dominio | interno
    ms: float = 0.0
    extra: dict = field(default_factory=dict)

    def como_observacion(self) -> str:
        if self.ok:
            d = {"ok": True, "resultado": self.resultado}
            if self.valor is not None and self.valor != self.resultado:
                d["valor"] = self.valor
        else:
            d = {"ok": False, "error": self.error}
        return json.dumps(d, ensure_ascii=False)

    def a_dict(self) -> dict:
        return asdict(self)


@dataclass
class Herramienta:
    nombre: str
    descripcion: str
    familia: str
    funcion: Callable
    parametros: dict
    requeridos: list
    comodin_de: tuple = ()          # familias que esta herramienta también puede cubrir

    def esquema(self) -> dict:
        return {"type": "function", "function": {
            "name": self.nombre,
            "description": self.descripcion,
            "parameters": {"type": "object", "properties": self.parametros,
                           "required": self.requeridos},
        }}


# --------------------------------------------------------------------------
# Esquema a partir de tipos y docstring
# --------------------------------------------------------------------------

_TIPOS_JSON = {int: "integer", float: "number", str: "string", bool: "boolean"}


def _esquema_tipo(anotacion) -> dict:
    origen = get_origin(anotacion)
    if origen is Literal:
        return {"type": "string", "enum": list(get_args(anotacion))}
    if origen in (list, typing.List):
        (interno,) = get_args(anotacion) or (float,)
        return {"type": "array", "items": _esquema_tipo(interno)}
    if anotacion in _TIPOS_JSON:
        return {"type": _TIPOS_JSON[anotacion]}
    raise TypeError(f"tipo no soportado en una herramienta: {anotacion!r}")


def _parsear_docstring(doc: str) -> tuple[str, dict]:
    """Formato Google: descripción, y luego una sección 'Args:' con 'nombre: texto'."""
    doc = inspect.cleandoc(doc or "")
    partes = re.split(r"^\s*Args:\s*$", doc, maxsplit=1, flags=re.MULTILINE)
    descripcion = " ".join(partes[0].split())
    args: dict = {}
    if len(partes) > 1:
        actual = None
        for linea in partes[1].splitlines():
            m = re.match(r"^\s*(\w+)\s*:\s*(.*)$", linea)
            if m:
                actual = m.group(1)
                args[actual] = m.group(2).strip()
            elif actual and linea.strip():
                args[actual] += " " + linea.strip()
    return descripcion, args


# --------------------------------------------------------------------------
# Conversión de argumentos: lo que escribe un LLM pequeño no siempre es JSON limpio
# --------------------------------------------------------------------------

def a_fraccion(valor: Any) -> Fraction:
    """Número exacto a partir de lo que proponga el modelo: 12, 2.35, "2,35",
    "18.500"?, "3/4", "1 1/2"... Se trabaja con Fraction para no introducir
    errores de coma flotante en una herramienta cuyo propósito es no equivocarse."""
    if isinstance(valor, bool):
        raise ValueError("un booleano no es un número")
    if isinstance(valor, int):
        return Fraction(valor)
    if isinstance(valor, float):
        return Fraction(repr(valor))
    if isinstance(valor, Fraction):
        return valor
    if not isinstance(valor, str):
        raise ValueError(f"no es un número: {valor!r}")
    t = valor.strip().replace("−", "-").replace(" ", "").replace("%", "")
    if re.fullmatch(r"-?\d{1,3}(,\d{3})+(\.\d+)?", t):      # 18,500 -> miles
        t = t.replace(",", "")
    elif re.fullmatch(r"-?\d+,\d+", t):                     # 2,35 -> decimal
        t = t.replace(",", ".")
    m = re.fullmatch(r"(-?\d+)\s*\+?\s*(\d+)/(\d+)", valor.strip())  # "1 1/2"
    if m and " " in valor.strip():
        entero, num, den = map(int, m.groups())
        signo = -1 if entero < 0 else 1
        return Fraction(entero) + signo * Fraction(num, den)
    try:
        return Fraction(t)
    except (ValueError, ZeroDivisionError) as e:
        raise ValueError(f"no es un número: {valor!r}") from e


def _convertir(valor: Any, anotacion, nombre: str):
    origen = get_origin(anotacion)
    if origen is Literal:
        opciones = get_args(anotacion)
        v = str(valor).strip().lower()
        if v not in opciones:
            raise ValueError(f"'{nombre}' debe ser uno de {list(opciones)}, no {valor!r}")
        return v
    if origen in (list, typing.List):
        if isinstance(valor, str):
            try:
                valor = json.loads(valor)
            except json.JSONDecodeError:
                valor = [p for p in re.split(r"[;\s]+|,(?=\s)", valor.strip("[] ")) if p]
        if not isinstance(valor, (list, tuple)) or not valor:
            raise ValueError(f"'{nombre}' debe ser una lista no vacía")
        (interno,) = get_args(anotacion) or (float,)
        return [_convertir(v, interno, nombre) for v in valor]
    if anotacion is float:
        return a_fraccion(valor)
    if anotacion is int:
        f = a_fraccion(valor)
        if f.denominator != 1:
            raise ValueError(f"'{nombre}' debe ser entero, no {valor!r}")
        return int(f)
    if anotacion is str:
        if isinstance(valor, (dict, list)):
            raise ValueError(f"'{nombre}' debe ser texto")
        return str(valor)
    if anotacion is bool:
        return bool(valor)
    return valor


# --------------------------------------------------------------------------
# El registro
# --------------------------------------------------------------------------

class RegistroHerramientas:
    def __init__(self):
        self._herramientas: dict[str, Herramienta] = {}

    # ---- alta de herramientas -------------------------------------------
    def herramienta(self, familia: str, comodin_de: tuple = (), nombre: str | None = None):
        """Decorador: `@registro.herramienta(familia="aritmetica")`."""
        def decorar(fn: Callable) -> Callable:
            self.registrar(fn, familia=familia, comodin_de=comodin_de, nombre=nombre)
            return fn
        return decorar

    def registrar(self, fn: Callable, familia: str, comodin_de: tuple = (),
                  nombre: str | None = None) -> Herramienta:
        nombre = nombre or fn.__name__
        if nombre in self._herramientas:
            raise ValueError(f"ya existe una herramienta llamada {nombre!r}")
        descripcion, docs_args = _parsear_docstring(fn.__doc__)
        if not descripcion:
            raise ValueError(f"la herramienta {nombre!r} necesita una descripción (docstring)")
        hints = typing.get_type_hints(fn, include_extras=False)
        parametros, requeridos = {}, []
        for p in inspect.signature(fn).parameters.values():
            if p.name not in hints:
                raise TypeError(f"{nombre}: el parámetro {p.name!r} no tiene tipo")
            esquema = _esquema_tipo(hints[p.name])
            if p.name not in docs_args:
                raise ValueError(f"{nombre}: falta describir {p.name!r} en 'Args:'")
            esquema["description"] = docs_args[p.name]
            if p.default is inspect.Parameter.empty:
                requeridos.append(p.name)
            else:
                esquema["default"] = p.default
            parametros[p.name] = esquema
        h = Herramienta(nombre, descripcion, familia, fn, parametros, requeridos, tuple(comodin_de))
        self._herramientas[nombre] = h
        return h

    # ---- consulta ----------------------------------------------------------
    def __contains__(self, nombre: str) -> bool:
        return nombre in self._herramientas

    def __len__(self) -> int:
        return len(self._herramientas)

    def obtener(self, nombre: str) -> Herramienta:
        return self._herramientas[nombre]

    def nombres(self) -> list[str]:
        return list(self._herramientas)

    def esquemas(self) -> list[dict]:
        return [h.esquema() for h in self._herramientas.values()]

    def familia_de(self, nombre: str) -> str | None:
        h = self._herramientas.get(nombre)
        return h.familia if h else None

    def subconjunto(self, incluir=None, excluir=()) -> "RegistroHerramientas":
        """Vista con parte de las herramientas (p. ej. C4 sin `buscar_documentos`)."""
        nuevo = RegistroHerramientas()
        for n, h in self._herramientas.items():
            if (incluir is None or n in incluir) and n not in excluir:
                nuevo._herramientas[n] = h
        return nuevo

    # ---- validación y ejecución -------------------------------------------
    def validar(self, llamada: LlamadaHerramienta) -> dict:
        """Devuelve los argumentos convertidos o lanza ValueError con un mensaje
        pensado para que el MODELO lo lea y corrija la llamada."""
        h = self._herramientas[llamada.nombre]
        hints = typing.get_type_hints(h.funcion)
        args = llamada.argumentos if isinstance(llamada.argumentos, dict) else {}
        desconocidos = set(args) - set(h.parametros)
        if desconocidos:
            raise ValueError(f"argumentos desconocidos para {h.nombre}: {sorted(desconocidos)}; "
                             f"acepta {list(h.parametros)}")
        faltan = [r for r in h.requeridos if r not in args]
        if faltan:
            raise ValueError(f"faltan argumentos para {h.nombre}: {faltan}")
        return {k: _convertir(v, hints[k], k) for k, v in args.items()}

    def ejecutar(self, llamada: LlamadaHerramienta) -> ResultadoHerramienta:
        t0 = time.perf_counter()
        base = dict(herramienta=llamada.nombre, argumentos=llamada.argumentos)

        def fin(**kw):
            return ResultadoHerramienta(**base, **kw, ms=(time.perf_counter() - t0) * 1000)

        if llamada.nombre not in self._herramientas:
            return fin(ok=False, tipo_error="desconocida",
                       error=f"no existe la herramienta {llamada.nombre!r}. Disponibles: {self.nombres()}")
        try:
            args = self.validar(llamada)
        except (ValueError, TypeError) as e:
            return fin(ok=False, tipo_error="argumentos", error=str(e))
        try:
            salida = self._herramientas[llamada.nombre].funcion(**args)
        except ErrorHerramienta as e:
            return fin(ok=False, tipo_error="dominio", error=str(e))
        except Exception as e:  # noqa: BLE001 — una herramienta nunca tumba al agente
            return fin(ok=False, tipo_error="interno", error=f"{type(e).__name__}: {e}")

        if isinstance(salida, dict):
            return fin(ok=True, resultado=str(salida.get("resultado")), valor=salida.get("valor"),
                       extra={k: v for k, v in salida.items() if k not in ("resultado", "valor")})
        return fin(ok=True, resultado=str(salida), valor=str(salida))

In [23]:
"""
Herramientas matemáticas del tutor.

Todas trabajan con aritmética EXACTA (Fraction / SymPy racional): una
calculadora que devuelve 0.30000000000000004 no le ahorra errores a nadie.

Cada herramienta devuelve {"resultado": texto para el modelo, "valor": forma
canónica para las métricas}. Los errores de dominio (división entre cero, raíz
par de un negativo) se lanzan como ErrorHerramienta: el agente los recibe como
observación y debe EXPLICARLOS, que es justo lo que pide el caso adv-02.

Familias (usadas para medir si el agente eligió bien):
  aritmetica, fracciones, potencias_raices, porcentajes, algebra, estadistica
  (+ documentos, que registra el notebook con el retriever de S08)

Para añadir una herramienta: escribir una función tipada con docstring y
decorarla con `@REGISTRO.herramienta(familia=...)`. Nada más cambia.
"""

from __future__ import annotations

import math
import re
from fractions import Fraction
from typing import Literal


REGISTRO = RegistroHerramientas()

FAMILIAS_NUMERICAS = ("aritmetica", "fracciones", "potencias_raices", "porcentajes", "estadistica")


# --------------------------------------------------------------------------
# Formato de salida
# --------------------------------------------------------------------------

def _decimal(q: Fraction, max_decimales: int = 6) -> str:
    """Decimal legible: exacto si termina en <= max_decimales, redondeado si no."""
    if q.denominator == 1:
        return str(q.numerator)
    texto = f"{float(round(q, max_decimales)):.{max_decimales}f}".rstrip("0").rstrip(".")
    return "0" if texto in ("-0", "") else texto


def _es_decimal_exacto(q: Fraction, max_decimales: int = 6) -> bool:
    d = q.denominator
    for p in (2, 5):
        while d % p == 0:
            d //= p
    return d == 1 and round(q, max_decimales) == q


def formatear(q: Fraction, preferir_fraccion: bool = False) -> dict:
    """{"resultado", "valor"} para un racional."""
    if q.denominator == 1:
        v = str(q.numerator)
        return {"resultado": v, "valor": v}
    frac = f"{q.numerator}/{q.denominator}"
    if preferir_fraccion or not _es_decimal_exacto(q):
        return {"resultado": f"{frac} (≈ {_decimal(q)})", "valor": frac}
    dec = _decimal(q)
    return {"resultado": dec, "valor": dec}


# --------------------------------------------------------------------------
# Parser seguro de expresiones (SymPy, nunca eval de Python)
# --------------------------------------------------------------------------

_PERMITIDOS = re.compile(r"^[0-9a-z_+\-*/().,\s=]*$")
_IDENTIFICADORES_OK = {"x", "y", "z", "a", "b", "c", "n", "m", "t", "sqrt", "pi"}
_SUPERINDICES = {"²": "**2", "³": "**3", "⁴": "**4", "⁵": "**5"}


def normalizar_expresion(texto: str, x_es_por: bool = False) -> str:
    t = texto.strip().lower()
    t = t.replace("×", "*").replace("·", "*").replace("÷", "/").replace("−", "-").replace("–", "-")
    t = t.replace("^", "**").replace(":", "/")
    for s, r in _SUPERINDICES.items():
        t = t.replace(s, r)
    t = re.sub(r"√\s*\(", "sqrt(", t)
    t = re.sub(r"√\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", t)
    t = re.sub(r"ra[ií]z\s+(?:cuadrada\s+)?de\s+(\d+(?:\.\d+)?)", r"sqrt(\1)", t)
    if x_es_por:                                   # "3847 x 296": aquí x es "por"
        t = re.sub(r"(?<=[\d)])\s*x\s*(?=[\d(])", "*", t)
    t = re.sub(r"(\d),(\d{3})(?!\d)", r"\1\2", t)  # 18,500 -> 18500
    t = re.sub(r"(\d),(\d)", r"\1.\2", t)          # 2,35   -> 2.35
    return t


def _decimales_a_racional(t: str) -> str:
    return re.sub(r"\d+\.\d+", lambda m: f"({Fraction(m.group())})", t)


def parsear(texto: str, x_es_por: bool = False):
    import sympy
    from sympy.parsing.sympy_parser import (
        convert_xor, implicit_multiplication_application, parse_expr, standard_transformations,
    )
    t = normalizar_expresion(texto, x_es_por=x_es_por)
    if not _PERMITIDOS.match(t) or "__" in t:
        raise ErrorHerramienta(f"la expresión contiene símbolos no permitidos: {texto!r}")
    desconocidos = set(re.findall(r"[a-z_]+", t)) - _IDENTIFICADORES_OK
    if desconocidos:
        raise ErrorHerramienta(f"nombres no permitidos en la expresión: {sorted(desconocidos)}")
    locales = {n: sympy.Symbol(n) for n in _IDENTIFICADORES_OK - {"sqrt", "pi"}}
    locales.update(sqrt=sympy.sqrt, pi=sympy.pi)
    transformaciones = standard_transformations + (implicit_multiplication_application, convert_xor)
    try:
        return parse_expr(_decimales_a_racional(t), local_dict=locales,
                          transformations=transformaciones, evaluate=True)
    except Exception as e:  # noqa: BLE001
        raise ErrorHerramienta(f"no pude interpretar la expresión {texto!r}: {e}") from e


def _sympy_a_salida(expr) -> dict:
    import sympy
    if expr.has(sympy.zoo, sympy.nan) or expr == sympy.oo:
        raise ErrorHerramienta("la operación no está definida (hay una división entre cero)")
    if expr.is_Rational:
        return formatear(Fraction(int(expr.p), int(expr.q)))
    if expr.is_number:
        if not expr.is_real:
            raise ErrorHerramienta("el resultado no es un número real")
        aprox = float(sympy.N(expr, 15))
        texto = f"{aprox:.6f}".rstrip("0").rstrip(".")
        return {"resultado": f"{sympy.sstr(expr)} (≈ {texto})", "valor": texto}
    texto = sympy.sstr(expr)
    return {"resultado": texto, "valor": texto}


# --------------------------------------------------------------------------
# ARITMÉTICA
# --------------------------------------------------------------------------

@REGISTRO.herramienta(familia="aritmetica")
def sumar(numeros: list[float]) -> dict:
    """Suma dos o más números (enteros o decimales) de forma exacta.

    Args:
        numeros: lista de números a sumar, por ejemplo [245, 378.5].
    """
    return formatear(sum(numeros, Fraction(0)))


@REGISTRO.herramienta(familia="aritmetica")
def restar(minuendo: float, sustraendo: float) -> dict:
    """Resta dos números de forma exacta: minuendo menos sustraendo.

    Args:
        minuendo: número del que se resta.
        sustraendo: número que se resta.
    """
    return formatear(minuendo - sustraendo)


@REGISTRO.herramienta(familia="aritmetica")
def multiplicar(numeros: list[float]) -> dict:
    """Multiplica dos o más números de forma exacta.

    Args:
        numeros: lista de factores, por ejemplo [3847, 296].
    """
    producto = Fraction(1)
    for n in numeros:
        producto *= n
    return formatear(producto)


@REGISTRO.herramienta(familia="aritmetica")
def dividir(dividendo: float, divisor: float,
            redondeo: Literal["ninguno", "arriba", "abajo"] = "ninguno") -> dict:
    """Divide dos números. Permite redondear el cociente a entero hacia arriba
    (p. ej. cuántos vehículos hacen falta) o hacia abajo (cuántos grupos completos).

    Args:
        dividendo: número que se divide.
        divisor: número entre el que se divide. No puede ser cero.
        redondeo: 'ninguno' (cociente exacto), 'arriba' o 'abajo' (a entero).
    """
    if divisor == 0:
        raise ErrorHerramienta("la división entre cero no está definida: ningún número "
                               "multiplicado por 0 da el dividendo")
    q = dividendo / divisor
    if redondeo == "arriba":
        return {**formatear(Fraction(math.ceil(q))), "cociente_exacto": _decimal(q)}
    if redondeo == "abajo":
        return {**formatear(Fraction(math.floor(q))), "cociente_exacto": _decimal(q)}
    return formatear(q)


@REGISTRO.herramienta(familia="aritmetica", comodin_de=FAMILIAS_NUMERICAS)
def evaluar_expresion(expresion: str) -> dict:
    """Evalúa una expresión aritmética completa respetando la jerarquía de
    operaciones. Acepta +, -, ×, ÷, paréntesis, potencias (^) y raíces (sqrt o √).

    Args:
        expresion: la expresión, por ejemplo '2.35 × 4.8 - 1.964' o '(5/8)*(4/7)'.
    """
    expr = parsear(expresion, x_es_por=True)
    if expr.free_symbols:
        raise ErrorHerramienta("la expresión tiene incógnitas: usa resolver_ecuacion u operar_expresion")
    return _sympy_a_salida(expr)


# --------------------------------------------------------------------------
# FRACCIONES
# --------------------------------------------------------------------------

@REGISTRO.herramienta(familia="fracciones")
def operar_fracciones(fraccion_a: str, fraccion_b: str,
                      operacion: Literal["suma", "resta", "multiplicacion", "division"]) -> dict:
    """Suma, resta, multiplica o divide dos fracciones y simplifica el resultado.

    Args:
        fraccion_a: primera fracción, por ejemplo '17/24' (también acepta enteros).
        fraccion_b: segunda fracción, por ejemplo '11/36'.
        operacion: 'suma', 'resta', 'multiplicacion' o 'division'.
    """
    a, b = a_fraccion(fraccion_a), a_fraccion(fraccion_b)
    if operacion == "division" and b == 0:
        raise ErrorHerramienta("no se puede dividir entre una fracción igual a cero")
    q = {"suma": a + b, "resta": a - b, "multiplicacion": a * b,
         "division": a / b if b else None}[operacion]
    return formatear(q, preferir_fraccion=True)


@REGISTRO.herramienta(familia="fracciones")
def simplificar_fraccion(fraccion: str) -> dict:
    """Lleva una fracción a su mínima expresión e indica el máximo común divisor usado.

    Args:
        fraccion: la fracción, por ejemplo '1386/2310'.
    """
    m = re.fullmatch(r"\s*(-?\d+)\s*/\s*(-?\d+)\s*", str(fraccion))
    if not m:
        raise ErrorHerramienta(f"'{fraccion}' no tiene la forma numerador/denominador con enteros")
    num, den = int(m.group(1)), int(m.group(2))
    if den == 0:
        raise ErrorHerramienta("el denominador no puede ser cero")
    mcd = math.gcd(num, den)
    return {**formatear(Fraction(num, den), preferir_fraccion=True), "mcd": mcd}


# --------------------------------------------------------------------------
# POTENCIAS Y RAÍCES
# --------------------------------------------------------------------------

@REGISTRO.herramienta(familia="potencias_raices")
def potencia(base: float, exponente: float) -> dict:
    """Eleva una base a un exponente (entero o fraccionario).

    Args:
        base: la base, por ejemplo 15.
        exponente: el exponente, por ejemplo 3.
    """
    import sympy
    if base == 0 and exponente < 0:
        raise ErrorHerramienta("cero elevado a un exponente negativo no está definido")
    if exponente.denominator == 1:
        return formatear(base ** exponente.numerator)
    return _sympy_a_salida(sympy.Rational(base.numerator, base.denominator)
                           ** sympy.Rational(exponente.numerator, exponente.denominator))


@REGISTRO.herramienta(familia="potencias_raices")
def raiz(radicando: float, indice: int = 2) -> dict:
    """Calcula la raíz n-ésima de un número; exacta si existe, aproximada si no.

    Args:
        radicando: el número dentro de la raíz, por ejemplo 7056.
        indice: 2 para raíz cuadrada, 3 para cúbica, etc.
    """
    import sympy
    if indice < 1:
        raise ErrorHerramienta("el índice de la raíz debe ser un entero positivo")
    if radicando < 0 and indice % 2 == 0:
        raise ErrorHerramienta("la raíz de índice par de un número negativo no es un número real")
    r = sympy.Rational(radicando.numerator, radicando.denominator)
    expr = -sympy.root(-r, indice) if r < 0 else sympy.root(r, indice)
    return _sympy_a_salida(sympy.nsimplify(expr))


# --------------------------------------------------------------------------
# PORCENTAJES
# --------------------------------------------------------------------------

@REGISTRO.herramienta(familia="porcentajes")
def porcentaje_de(porcentaje: float, cantidad: float) -> dict:
    """Calcula el porcentaje de una cantidad (p. ej. el 18% de 2450000).

    Args:
        porcentaje: el porcentaje como número, por ejemplo 18 para 18%.
        cantidad: la cantidad sobre la que se calcula.
    """
    return formatear(porcentaje / 100 * cantidad)


@REGISTRO.herramienta(familia="porcentajes")
def aplicar_porcentaje(cantidad: float, porcentaje: float,
                       operacion: Literal["aumento", "descuento"]) -> dict:
    """Aplica un aumento o un descuento porcentual a una cantidad y devuelve el
    valor final. Para descuentos sucesivos, llamarla una vez por descuento.

    Args:
        cantidad: el valor inicial.
        porcentaje: el porcentaje como número, por ejemplo 20 para 20%.
        operacion: 'aumento' o 'descuento'.
    """
    factor = 1 + porcentaje / 100 if operacion == "aumento" else 1 - porcentaje / 100
    return {**formatear(cantidad * factor), "factor": _decimal(factor)}


@REGISTRO.herramienta(familia="porcentajes")
def valor_antes_de_porcentaje(valor_final: float, porcentaje: float,
                              operacion: Literal["aumento", "descuento"]) -> dict:
    """Recupera el valor ORIGINAL a partir del valor después de un aumento o un
    descuento (porcentaje inverso): divide entre el factor, no resta el porcentaje.

    Args:
        valor_final: el valor después del cambio, por ejemplo 92000.
        porcentaje: el porcentaje aplicado, por ejemplo 15.
        operacion: 'aumento' o 'descuento'.
    """
    factor = 1 + porcentaje / 100 if operacion == "aumento" else 1 - porcentaje / 100
    if factor == 0:
        raise ErrorHerramienta("un descuento del 100% no permite recuperar el valor original")
    return {**formatear(valor_final / factor), "factor": _decimal(factor)}


# --------------------------------------------------------------------------
# ESTADÍSTICA
# --------------------------------------------------------------------------

@REGISTRO.herramienta(familia="estadistica")
def promedio_ponderado(valores: list[float], pesos: list[float]) -> dict:
    """Calcula el promedio ponderado de unos valores con sus pesos. Los pesos
    pueden ser porcentajes (30, 70) o fracciones (0.3, 0.7).

    Args:
        valores: las notas o valores, por ejemplo [4.2, 3.5].
        pesos: el peso de cada valor, en el mismo orden, por ejemplo [30, 70].
    """
    if len(valores) != len(pesos):
        raise ErrorHerramienta("debe haber un peso por cada valor")
    total = sum(pesos, Fraction(0))
    if total == 0:
        raise ErrorHerramienta("los pesos no pueden sumar cero")
    return formatear(sum((v * p for v, p in zip(valores, pesos)), Fraction(0)) / total)


# --------------------------------------------------------------------------
# ÁLGEBRA
# --------------------------------------------------------------------------

@REGISTRO.herramienta(familia="algebra")
def resolver_ecuacion(ecuacion: str, incognita: str = "x") -> dict:
    """Resuelve una ecuación con una incógnita, por ejemplo '7x - 23 = 4x + 58'.

    Args:
        ecuacion: la ecuación con un signo '='.
        incognita: la letra de la incógnita (por defecto 'x').
    """
    import sympy
    if ecuacion.count("=") != 1:
        raise ErrorHerramienta("la ecuación debe tener exactamente un signo '='")
    if incognita not in _IDENTIFICADORES_OK - {"sqrt", "pi"}:
        raise ErrorHerramienta(f"incógnita no válida: {incognita!r}")
    izq, der = ecuacion.split("=")
    x = sympy.Symbol(incognita)
    soluciones = sympy.solve(sympy.Eq(parsear(izq), parsear(der)), x)
    if not soluciones:
        raise ErrorHerramienta("la ecuación no tiene solución (o se cumple para todo valor)")
    salidas = [_sympy_a_salida(s) for s in soluciones]
    if len(salidas) == 1:
        return {"resultado": f"{incognita} = {salidas[0]['resultado']}", "valor": salidas[0]["valor"]}
    return {"resultado": ", ".join(f"{incognita} = {s['resultado']}" for s in salidas),
            "valor": ";".join(s["valor"] for s in salidas)}


@REGISTRO.herramienta(familia="algebra")
def operar_expresion(expresion: str,
                     operacion: Literal["simplificar", "expandir", "factorizar"]) -> dict:
    """Simplifica, expande o factoriza una expresión algebraica, por ejemplo
    expandir '(2x + 3)(x - 5)'.

    Args:
        expresion: la expresión algebraica.
        operacion: 'simplificar', 'expandir' o 'factorizar'.
    """
    import sympy
    expr = parsear(expresion)
    resultado = {"simplificar": sympy.simplify, "expandir": sympy.expand,
                 "factorizar": sympy.factor}[operacion](expr)
    texto = sympy.sstr(resultado).replace("**", "^").replace("*", "")
    return {"resultado": texto, "valor": texto}

In [24]:
"""
Núcleo del agente: normalizador de consulta, parser de acciones y bucle ReAct.

El núcleo no conoce ninguna herramienta concreta ni ningún modelo concreto:

  llm(mensajes, esquemas) -> texto      cualquier función (Qwen, un simulador de tests)
  RegistroHerramientas                  qué herramientas hay y cómo se ejecutan

Flujo de una pregunta (la arquitectura de la propuesta M3):

  Usuario
    -> Normalizador        (reglas deterministas + reescritura LLM opcional, con salvaguarda de números)
    -> Razonador (LLM)     pensamiento + propuesta de acción, en el formato nativo <tool_call> de Qwen2.5
    -> Parser/Validador    texto -> LlamadaHerramienta (o error de formato que se devuelve al modelo)
    -> Tool Registry       validación de tipos + ejecución segura -> ResultadoHerramienta
    -> Observación         vuelve al modelo como <tool_response>
    -> ... repetir hasta que el modelo responda o se agote el tope de pasos
    -> Verificador         ¿la respuesta final usa algún resultado de las herramientas?
    -> Respuesta final + traza completa (para métricas y RAGAS)
"""

from __future__ import annotations

import json
import re
import time
from dataclasses import asdict, dataclass, field
from typing import Callable



# --------------------------------------------------------------------------
# Utilidades numéricas (independientes de CODIGO_METRICAS)
# --------------------------------------------------------------------------

_PAT_NUM = re.compile(r"-?\d+(?:[.,]\d+)?(?:\s*/\s*\d+)?")


def numeros_en(texto: str) -> list[str]:
    """Números de un texto con punto decimal. Como en todo el proyecto, el punto
    es SIEMPRE decimal (9.316) y la coma solo es de miles si le siguen 3 cifras."""
    t, previo = texto or "", None
    while t != previo:                      # 1,138,712 necesita dos pasadas
        previo, t = t, re.sub(r"(\d),(\d{3})(?!\d)", r"\1\2", t)
    return [re.sub(r"\s+", "", n).replace(",", ".") for n in _PAT_NUM.findall(t)]


def a_numero(texto: str | None) -> float | None:
    if texto is None:
        return None
    try:
        if "/" in texto:
            a, b = texto.split("/")
            return float(a) / float(b)
        return float(texto)
    except (ValueError, ZeroDivisionError):
        return None


def mismo_numero(a, b, tol: float = 1e-6) -> bool:
    x, y = a_numero(str(a)) if a is not None else None, a_numero(str(b)) if b is not None else None
    return x is not None and y is not None and abs(x - y) <= tol * max(1.0, abs(y))


_PAT_RESPUESTA = re.compile(r"Respuesta\s+final\s*:\s*(.+)", re.IGNORECASE)


def valor_final(texto: str) -> str | None:
    m = _PAT_RESPUESTA.search(texto or "")
    fuente = m.group(1).splitlines()[0] if m else (texto or "")
    nums = numeros_en(fuente)
    if not nums:
        return None
    return nums[0] if m else nums[-1]


# --------------------------------------------------------------------------
# Normalizador de la consulta (preguntas mal escritas)
# --------------------------------------------------------------------------
# Decisión: dos capas.
#  1) REGLAS deterministas y auditables para lo frecuente y seguro: espacios,
#     signos repetidos, abreviaturas de chat y operadores escritos en palabras
#     ENTRE números ("17/24 mas 11/36", "3847 x 296").
#  2) REESCRITURA con el LLM (opcional, solo en la arquitectura agéntica) para
#     faltas de ortografía que ninguna regla cubre ("bale", "evaluasion").
# Salvaguarda común: si la versión corregida no conserva EXACTAMENTE los mismos
# números, se descarta. Una corrección ortográfica que cambia un dato es peor
# que la pregunta mal escrita.

_ABREVIATURAS = [
    (r"\bq\b", "que"), (r"\bxq\b", "porque"), (r"\bpq\b", "porque"), (r"\bpa\b", "para"),
    # "x" solo significa "por" delante de palabras típicas; "halla x si..." debe quedar igual.
    (r"\bx\b(?=\s+(?:pagar|persona|personas|cada|dia|día|mes|periodo|período|ciento|favor|hora|kilo)\b)", "por"),
    (r"\bai\b", "hay"), (r"\bk\b", "que"),
    (r"\btmb\b", "también"), (r"\bdl\b", "del"),
]
_OPERADORES_ENTRE_NUMEROS = [
    (r"(?<=\d)\s+(?:mas|más)\s+(?=\d)", " + "),
    (r"(?<=\d)\s+menos\s+(?=\d)", " - "),
    (r"(?<=\d)\s*[x×*]\s*(?=\d)", " × "),
    (r"(?<=\d)\s+por\s+(?=\d)", " × "),
    (r"(?<=\d)\s+(?:dividido\s+(?:en|entre|por)|entre)\s+(?=\d)", " ÷ "),
]


@dataclass
class ConsultaNormalizada:
    original: str
    normalizada: str
    cambios: list = field(default_factory=list)
    reescrita_llm: bool = False
    descartada: str | None = None      # motivo si la reescritura LLM se rechazó


def parece_informal(texto: str) -> bool:
    """Disparador de la reescritura LLM: sin tildes, sin eñes y sin '¿'. Una
    pregunta bien escrita en español casi siempre tiene alguno de los tres; así
    no se gasta una generación (ni se arriesga el sentido) en las que no lo necesitan."""
    return not re.search(r"[¿áéíóúñÁÉÍÓÚÑ]", texto)


class Normalizador:
    def __init__(self, reescribir: Callable[[str], str] | None = None,
                 condicion: Callable[[str], bool] | None = parece_informal):
        self.reescribir = reescribir
        self.condicion = condicion

    @staticmethod
    def _mismos_numeros(a: str, b: str) -> bool:
        return sorted(numeros_en(a)) == sorted(numeros_en(b))

    def reglas(self, texto: str) -> tuple[str, list]:
        cambios, t = [], " ".join(texto.split())
        for patron, reemplazo in _OPERADORES_ENTRE_NUMEROS + _ABREVIATURAS:
            nuevo = re.sub(patron, reemplazo, t, flags=re.IGNORECASE)
            if nuevo != t:
                cambios.append(patron)
                t = nuevo
        t2 = re.sub(r"([?!.])\1+", r"\1", t)
        if t2 != t:
            cambios.append("signos repetidos")
        t = t2
        if "?" in t and "¿" not in t:
            t = "¿" + t[0].upper() + t[1:] if t else t
        elif t:
            t = t[0].upper() + t[1:]
        return t, cambios

    def __call__(self, texto: str) -> ConsultaNormalizada:
        t, cambios = self.reglas(texto)
        if not self._mismos_numeros(texto, t):     # las reglas nunca deben tocar números
            return ConsultaNormalizada(texto, texto, [], descartada="las reglas alteraban números")
        res = ConsultaNormalizada(texto, t, cambios)
        if self.reescribir is not None and (self.condicion is None or self.condicion(texto)):
            candidata = (self.reescribir(t) or "").strip().strip('"').splitlines()
            candidata = candidata[0].strip() if candidata else ""
            if not candidata:
                res.descartada = "reescritura vacía"
            elif not self._mismos_numeros(t, candidata):
                res.descartada = f"la reescritura cambió los números: {candidata!r}"
            elif not (0.5 <= len(candidata) / max(1, len(t)) <= 2.0):
                res.descartada = "la reescritura cambió demasiado la longitud"
            else:
                res.normalizada, res.reescrita_llm = candidata, candidata != t
        return res


# --------------------------------------------------------------------------
# Parser: texto del modelo -> pensamiento + llamadas | respuesta
# --------------------------------------------------------------------------

@dataclass
class SalidaModelo:
    texto: str
    pensamiento: str = ""
    llamadas: list = field(default_factory=list)
    respuesta: str | None = None
    error_formato: str | None = None


_PAT_TOOL_CALL = re.compile(r"<tool_call>\s*(.*?)\s*(?:</tool_call>|$)", re.DOTALL)


def _json_de(texto: str):
    """Primer objeto JSON balanceado del texto (tolera texto alrededor)."""
    inicio = texto.find("{")
    while inicio != -1:
        nivel, en_cadena, escape = 0, False, False
        for i in range(inicio, len(texto)):
            c = texto[i]
            if en_cadena:
                if escape:
                    escape = False
                elif c == "\\":
                    escape = True
                elif c == '"':
                    en_cadena = False
                continue
            if c == '"':
                en_cadena = True
            elif c == "{":
                nivel += 1
            elif c == "}":
                nivel -= 1
                if nivel == 0:
                    try:
                        return json.loads(texto[inicio:i + 1])
                    except json.JSONDecodeError:
                        break
        inicio = texto.find("{", inicio + 1)
    return None


def _a_llamada(obj, formato: str) -> LlamadaHerramienta | None:
    if not isinstance(obj, dict):
        return None
    nombre = obj.get("name") or obj.get("tool") or obj.get("nombre")
    args = obj.get("arguments", obj.get("args", obj.get("argumentos", {})))
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except json.JSONDecodeError:
            args = {}
    if not isinstance(nombre, str):
        return None
    return LlamadaHerramienta(nombre=nombre, argumentos=args if isinstance(args, dict) else {},
                              formato=formato)


def analizar_salida(texto: str) -> SalidaModelo:
    s = SalidaModelo(texto=texto)
    bloques = _PAT_TOOL_CALL.findall(texto)
    if "<tool_call>" in texto:
        s.pensamiento = texto.split("<tool_call>")[0].strip()
        for b in bloques:
            llamada = _a_llamada(_json_de(b), "nativo")
            if llamada is None:
                s.error_formato = f"no pude leer la llamada a herramienta: {b[:120]!r}"
            else:
                s.llamadas.append(llamada)
        if not s.llamadas and s.error_formato is None:
            s.error_formato = "etiqueta <tool_call> vacía"
    else:
        # Recuperación: el modelo escribió el JSON sin las etiquetas.
        obj = _json_de(texto)
        llamada = _a_llamada(obj, "recuperado") if obj else None
        if llamada is not None and ("arguments" in obj or "args" in obj):
            s.pensamiento = texto[:texto.find("{")].strip()
            s.llamadas.append(llamada)
        else:
            s.respuesta = texto.strip()
    s.pensamiento = re.sub(r"^\s*(Pensamiento|Thought)\s*:\s*", "", s.pensamiento, flags=re.IGNORECASE)
    return s


# --------------------------------------------------------------------------
# Traza
# --------------------------------------------------------------------------

@dataclass
class EventoTraza:
    paso: int
    tipo: str        # normalizacion | contexto | pensamiento | llamada | observacion |
                     # error_formato | verificacion | respuesta | tope_pasos
    datos: dict


@dataclass
class RespuestaAgente:
    pregunta: str
    respuesta: str
    pregunta_normalizada: str
    traza: list = field(default_factory=list)
    contextos: list = field(default_factory=list)       # chunks usados (fijos o buscados)
    observaciones: list = field(default_factory=list)   # ResultadoHerramienta como dict
    pasos_llm: int = 0
    terminacion: str = "respuesta"                      # respuesta | tope_pasos
    segundos: float = 0.0

    @property
    def llamadas(self) -> list:
        return self.observaciones

    def a_dict(self) -> dict:
        d = asdict(self)
        d["traza"] = [asdict(e) if not isinstance(e, dict) else e for e in self.traza]
        return d


# --------------------------------------------------------------------------
# El agente
# --------------------------------------------------------------------------

class AgenteTutor:
    """Una sola clase configurable para las configuraciones C2–C5.

    contexto_fijo : función pregunta -> [chunks] que se INYECTA siempre (RAG clásico).
                    None = el agente decide si buscar (vía herramienta `buscar_documentos`).
    registro      : herramientas disponibles; None = sin herramientas.
    max_rondas    : cuántas veces puede pedir herramientas (1 = function calling de una
                    ronda, C3; >1 = ReAct, C4/C5).
    verificar     : si la respuesta final no coincide con ningún resultado numérico
                    observado, se le pide UNA revisión.
    normalizador  : Normalizador o None.
    few_shot      : lista de conversaciones de ejemplo (mensajes) que van antes de la pregunta.
    """

    def __init__(self, llm: Callable, system: str, registro: RegistroHerramientas | None = None,
                 contexto_fijo: Callable | None = None, max_rondas: int = 1, max_pasos: int = 6,
                 verificar: bool = False, normalizador: Normalizador | None = None,
                 few_shot: list | None = None, nombre: str = "agente",
                 extraer_contextos: Callable | None = None):
        self.llm, self.system, self.registro = llm, system, registro
        self.contexto_fijo, self.max_rondas, self.max_pasos = contexto_fijo, max_rondas, max_pasos
        self.verificar, self.normalizador, self.few_shot = verificar, normalizador, few_shot or []
        self.nombre = nombre
        # Cómo sacar chunks de una observación de `buscar_documentos` (para RAGAS).
        self.extraer_contextos = extraer_contextos

    # ------------------------------------------------------------------
    def _mensajes_iniciales(self, pregunta: str, contextos: list) -> list:
        msgs = [{"role": "system", "content": self.system}]
        for conversacion in self.few_shot:
            msgs.extend(conversacion)
        if contextos:
            bloque = "\n\n".join(contextos)
            msgs.append({"role": "user", "content": f"Contexto:\n{bloque}\n\nPregunta: {pregunta}"})
        else:
            msgs.append({"role": "user", "content": pregunta})
        return msgs

    @staticmethod
    def _msg_llamadas(pensamiento: str, llamadas: list) -> dict:
        return {"role": "assistant", "content": pensamiento,
                "tool_calls": [{"type": "function",
                                "function": {"name": l.nombre, "arguments": l.argumentos}}
                               for l in llamadas]}

    # ------------------------------------------------------------------
    def __call__(self, pregunta: str) -> RespuestaAgente:
        t0 = time.perf_counter()
        traza: list = []
        paso = 0

        norm = self.normalizador(pregunta) if self.normalizador else None
        q = norm.normalizada if norm else pregunta
        if norm:
            traza.append(EventoTraza(0, "normalizacion", asdict(norm)))

        contextos = list(self.contexto_fijo(q)) if self.contexto_fijo else []
        if contextos:
            traza.append(EventoTraza(0, "contexto", {"chunks": contextos}))

        msgs = self._mensajes_iniciales(q, contextos)
        esquemas = self.registro.esquemas() if self.registro is not None and len(self.registro) else None
        observaciones: list = []
        rondas, revisado, pasos_llm = 0, False, 0

        while pasos_llm < self.max_pasos:
            paso += 1
            herramientas_activas = esquemas if (esquemas and rondas < self.max_rondas) else None
            texto = self.llm(msgs, herramientas_activas)
            pasos_llm += 1
            salida = analizar_salida(texto)

            if salida.llamadas and herramientas_activas:
                if salida.pensamiento:
                    traza.append(EventoTraza(paso, "pensamiento", {"texto": salida.pensamiento}))
                msgs.append(self._msg_llamadas(salida.pensamiento, salida.llamadas))
                for llamada in salida.llamadas:
                    traza.append(EventoTraza(paso, "llamada", asdict(llamada)))
                    res = self.registro.ejecutar(llamada)
                    d = res.a_dict()
                    d["formato"] = llamada.formato
                    observaciones.append(d)
                    traza.append(EventoTraza(paso, "observacion", d))
                    if res.ok and self.extraer_contextos and llamada.nombre == "buscar_documentos":
                        contextos.extend(self.extraer_contextos(res))
                    msgs.append({"role": "tool", "name": llamada.nombre,
                                 "content": res.como_observacion()})
                rondas += 1
                continue

            if salida.llamadas and not herramientas_activas:
                # Pidió herramientas cuando ya no le quedan rondas: se le pide responder.
                traza.append(EventoTraza(paso, "error_formato",
                                         {"motivo": "llamada sin rondas disponibles", "texto": texto}))
                msgs.append({"role": "assistant", "content": texto})
                msgs.append({"role": "user", "content": "Ya no puedes usar más herramientas. "
                             "Responde con lo que tienes y termina con 'Respuesta final:'."})
                continue

            if salida.error_formato:
                traza.append(EventoTraza(paso, "error_formato", {"motivo": salida.error_formato, "texto": texto}))
                msgs.append({"role": "assistant", "content": texto})
                msgs.append({"role": "user", "content": f"Error: {salida.error_formato}. Escribe la llamada "
                             "como JSON válido dentro de <tool_call></tool_call> o responde directamente."})
                continue

            respuesta = salida.respuesta or ""
            if self.verificar and not revisado:
                problema = self._verificar(respuesta, observaciones)
                if problema:
                    revisado = True
                    traza.append(EventoTraza(paso, "verificacion", {"problema": problema, "respuesta": respuesta}))
                    msgs.append({"role": "assistant", "content": respuesta})
                    msgs.append({"role": "user", "content": f"Verificación: {problema} Revisa tu respuesta "
                                 "usando los resultados de las herramientas y vuelve a escribirla completa, "
                                 "terminando con 'Respuesta final:'."})
                    continue
            traza.append(EventoTraza(paso, "respuesta", {"texto": respuesta}))
            return RespuestaAgente(pregunta, respuesta, q, traza, contextos, observaciones,
                                   pasos_llm, "respuesta", time.perf_counter() - t0)

        # Tope de pasos: una última generación SIN herramientas.
        traza.append(EventoTraza(paso, "tope_pasos", {"max_pasos": self.max_pasos}))
        msgs.append({"role": "user", "content": "Se acabaron los pasos. Responde ya con lo que tienes "
                     "y termina con 'Respuesta final:'."})
        respuesta = analizar_salida(self.llm(msgs, None)).respuesta or ""
        pasos_llm += 1
        traza.append(EventoTraza(paso + 1, "respuesta", {"texto": respuesta}))
        return RespuestaAgente(pregunta, respuesta, q, traza, contextos, observaciones,
                               pasos_llm, "tope_pasos", time.perf_counter() - t0)

    # ------------------------------------------------------------------
    @staticmethod
    def _verificar(respuesta: str, observaciones: list) -> str | None:
        """Consistencia respuesta <-> herramientas. Solo interviene si hubo
        resultados numéricos y la respuesta final no usa NINGUNO de ellos."""
        valores = [o.get("valor") for o in observaciones if o.get("ok") and o.get("valor")]
        valores = [v for v in valores if a_numero(v) is not None]
        if not valores:
            return None
        v = valor_final(respuesta)
        if v is None:
            return "tu respuesta no tiene un resultado numérico final."
        if any(mismo_numero(v, x) for x in valores):
            return None
        return (f"tu respuesta final ({v}) no coincide con ningún resultado obtenido con las "
                f"herramientas ({', '.join(valores[-3:])}).")

In [25]:
"""
El RAG de S08 convertido en una herramienta más (S10: "el retrieval también es
una herramienta").

No se reimplementa nada: la función recibe la búsqueda que ya existe
(`buscar_con_rerank` de S08) y la envuelve como `buscar_documentos`. El formato
del resultado es el mismo "[Fuente: ...]" del prompt de S07, así el modelo ve
los pasajes igual que cuando el contexto se inyectaba fijo.
"""

from __future__ import annotations

from typing import Callable



def formatear_pasajes(pasajes: list[tuple[str, str]]) -> str:
    """[(fuente, texto)] -> bloque de texto con el formato de S07."""
    return "\n\n".join(f"[Fuente: {fuente}]\n{texto}" for fuente, texto in pasajes)


def registro_con_documentos(base: RegistroHerramientas, buscar: Callable[[str, int], list[int]],
                            chunks: list[str], metadatos: list[dict], ids_chunks: list[str],
                            k: int = 3) -> RegistroHerramientas:
    """Copia de `base` + `buscar_documentos`. `base` no se modifica."""
    registro = base.subconjunto()

    @registro.herramienta(familia="documentos")
    def buscar_documentos(consulta: str) -> dict:
        """Busca en los documentos oficiales del colegio (sistema institucional de
        evaluación, plan de área de matemáticas, protocolos y guías) y devuelve los
        pasajes más relevantes con su fuente.

        Args:
            consulta: qué información buscar, por ejemplo 'porcentaje del examen final'.
        """
        idxs = buscar(consulta, k)
        return {"resultado": formatear_pasajes([(metadatos[i]["fuente"], chunks[i]) for i in idxs]),
                "valor": None,
                "chunks": [chunks[i] for i in idxs],
                "ids": [ids_chunks[i] for i in idxs]}

    return registro


def chunks_de_resultado(resultado) -> list[str]:
    """Para `AgenteTutor(extraer_contextos=...)`: los chunks que devolvió la búsqueda."""
    return list(resultado.extra.get("chunks", []))

In [26]:
"""
Prompts de sistema de las configuraciones con herramientas (C3, C4, C5).

Se versionan aquí, junto al núcleo, porque son parte de la arquitectura: las
trayectorias de entrenamiento (F6) usan EXACTAMENTE el mismo prompt que el
agente evaluado, o el fine-tuning aprendería un comportamiento distinto del
que se mide.

Cada configuración añade UNA idea sobre la anterior:
  C3  herramientas de cálculo + "no calcules de memoria"
  C4  + formato ReAct explícito: Pensamiento antes de cada acción, varios pasos
  C5  + el retrieval es una herramienta más + planificación de fuentes
"""

_BASE_TUTOR = (
    "Eres un tutor de matemáticas para estudiantes de colegio.\n"
    "- No hagas cuentas de memoria: si el problema requiere una operación (sumar, restar, "
    "multiplicar, dividir, fracciones, potencias, raíces, porcentajes, promedios o ecuaciones), "
    "llama a la herramienta adecuada y usa exactamente su resultado.\n"
    "- Si una herramienta devuelve un error (por ejemplo, una división entre cero), explícale al "
    "estudiante por qué la operación no tiene resultado.\n"
    "- Si la pregunta no es de matemáticas, o le faltan datos para resolverla, dilo con claridad "
    "sin usar herramientas.\n"
    "- Al terminar, explica el procedimiento paso a paso (Paso 1:, Paso 2:, ...) y termina con "
    "una línea 'Respuesta final: <resultado>'."
)

_CONTEXTO_FIJO = (
    "\n- Si recibes un Contexto con documentos del colegio y la pregunta pide un dato institucional, "
    "usa solo ese contexto y menciona la fuente. Si el dato no aparece, responde: "
    "\"No tengo esa información en mis fuentes.\""
)

_REACT = (
    "\n- Trabaja por pasos. Antes de cada herramienta escribe una línea 'Pensamiento:' con qué "
    "necesitas y por qué. Después de cada resultado decide si necesitas otra herramienta o si ya "
    "puedes responder."
)

_AGENTE = (
    "\n- Para cualquier dato del colegio (sistema de evaluación, porcentajes de la nota, notas "
    "mínimas, recuperaciones, plan de área, protocolos) usa la herramienta buscar_documentos: "
    "nunca inventes un dato institucional. Si después de buscar el dato no aparece, responde: "
    "\"No tengo esa información en mis fuentes.\"\n"
    "- Para un problema que solo requiere calcular, no busques documentos.\n"
    "- En tu primer Pensamiento planifica qué datos necesitas y de dónde sale cada uno: del "
    "enunciado, de los documentos o de un cálculo."
)

SYSTEM_C3_HERRAMIENTAS = _BASE_TUTOR + _CONTEXTO_FIJO
SYSTEM_C4_REACT = _BASE_TUTOR + _CONTEXTO_FIJO + _REACT
SYSTEM_C5_AGENTE = _BASE_TUTOR + _REACT + _AGENTE

SYSTEM_NORMALIZADOR = (
    "Corrige la ortografía y la puntuación de la pregunta de un estudiante. NO cambies ningún "
    "número, no la respondas y no añadas información. Escribe solo la pregunta corregida en una línea."
)

In [27]:
import json, time
from contextlib import nullcontext
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_BASE    = "Qwen/Qwen2.5-1.5B-Instruct"
DIR_LORA_M1    = "adaptadores/qwen-lora"          # M1: formato "Paso N / Respuesta final"
DIR_LORA_AGENT = "adaptadores/qwen-lora-agente"   # S10: entrenado con trayectorias (opcional)

if not Path(DIR_LORA_M1).exists():
    raise FileNotFoundError(f"No se encuentra {DIR_LORA_M1}. Ejecuten S04_Lab_Fine_tuning_Qwen.ipynb "
                            "o monten el Drive donde quedó guardado.")

tok = AutoTokenizer.from_pretrained(MODELO_BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

_base = AutoModelForCausalLM.from_pretrained(MODELO_BASE, torch_dtype=torch.float16).to(DEVICE)
modelo = PeftModel.from_pretrained(_base, DIR_LORA_M1, adapter_name="m1").eval()
ADAPTADORES = ["m1"]
if Path(DIR_LORA_AGENT).exists():
    modelo.load_adapter(DIR_LORA_AGENT, adapter_name="agente")
    ADAPTADORES.append("agente")
print("Adaptadores cargados:", ADAPTADORES, "(+ None = modelo base)")


class ContadorTokens:
    """Acumula tokens y llamadas al LLM de la pregunta en curso."""
    def __init__(self):
        self.reset()

    def reset(self):
        self.llamadas, self.entrada, self.salida = 0, 0, 0


CONTADOR = ContadorTokens()
_CACHE_LLM = {}


@torch.no_grad()
def generar_chat(mensajes, esquemas=None, adaptador="m1", max_new_tokens=220, muestreo=None):
    """Una generación. `adaptador`: "m1", "agente" o None (modelo base).
    `muestreo`: None (greedy) o (temperatura, semilla) para autoconsistencia."""
    clave = (json.dumps(mensajes, ensure_ascii=False, sort_keys=True, default=str),
             json.dumps(esquemas, sort_keys=True) if esquemas else None, adaptador, max_new_tokens, muestreo)
    prompt = tok.apply_chat_template(mensajes, tools=esquemas, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt", add_special_tokens=False).to(modelo.device)
    CONTADOR.llamadas += 1
    CONTADOR.entrada += ids["input_ids"].shape[1]
    if clave in _CACHE_LLM:
        texto, n_out = _CACHE_LLM[clave]
        CONTADOR.salida += n_out
        return texto
    if adaptador is None:
        contexto = modelo.disable_adapter()
    else:
        modelo.set_adapter(adaptador)
        contexto = nullcontext()
    kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
    if muestreo is None:
        kwargs["do_sample"] = False
    else:
        temperatura, semilla = muestreo
        torch.manual_seed(semilla)
        kwargs.update(do_sample=True, temperature=temperatura, top_p=0.95)
    with contexto:
        out = modelo.generate(**ids, **kwargs)
    nuevos = out[0][ids["input_ids"].shape[1]:]
    texto = tok.decode(nuevos, skip_special_tokens=True).strip()
    _CACHE_LLM[clave] = (texto, len(nuevos))
    CONTADOR.salida += len(nuevos)
    return texto


def crear_llm(adaptador="m1", max_new_tokens=220, muestreo=None):
    """La firma que espera el núcleo del agente: llm(mensajes, esquemas) -> texto."""
    def llm(mensajes, esquemas):
        return generar_chat(mensajes, esquemas, adaptador=adaptador,
                            max_new_tokens=max_new_tokens, muestreo=muestreo)
    return llm

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Adaptadores cargados: ['m1', 'agente'] (+ None = modelo base)


In [28]:
# `responder_m3`: el sistema que se entregó en M3, sin reescribirlo.
#
# Se usa el adaptador entrenado con trayectorias de herramientas si está en
# Drive; si no, el de M1. La diferencia cambia QUÉ sistema se está midiendo, así
# que queda declarada en el scorecard y en resultados_m4.json.
ADAPTADOR_TUTOR = "agente" if "agente" in ADAPTADORES else "m1"

# Aciertos de M3 sobre el eval set de M2, leídos de
# M3_rag_agentes/resultados/scorecard_agente_v2.csv (la versión con el criterio de
# adv-02 corregido). Son la referencia de la fila de regresión.
REFERENCIA_M3 = {
    "agente": {"nombre": "C5-FT · entrenado", "aciertos": 13, "segundos": 14.1},
    "m1":     {"nombre": "C1 · +LoRA",        "aciertos": 14, "segundos": 5.5},
}[ADAPTADOR_TUTOR]

print(f"Adaptador del tutor: {ADAPTADOR_TUTOR}")
print(f"Referencia de M3   : {REFERENCIA_M3['nombre']} con {REFERENCIA_M3['aciertos']}/21")
if ADAPTADOR_TUTOR == "m1":
    print("  Atención: no se encontró adaptadores/qwen-lora-agente, así que el agente no")
    print("  usará herramientas como en S10. Es otro sistema, y hay que decirlo al leer.")

REGISTRO_M4 = registro_con_documentos(REGISTRO, buscar_densa, chunks, metadatos, ids_chunks, k=3)

agente_m3 = AgenteTutor(
    crear_llm(ADAPTADOR_TUTOR, 320), SYSTEM_C5_AGENTE, registro=REGISTRO_M4,
    max_rondas=6, max_pasos=8, verificar=True,
    extraer_contextos=chunks_de_resultado, nombre="tutor M3")

# La vía visual usa el adaptador de diagnóstico si está; la de texto sigue con el
# de S10. Son dos vías con dos adaptadores a propósito:
#
#   · la regresión de M2 tiene que medir el MISMO sistema que midió M3, o la fila
#     deja de ser una regresión;
#   · la tarea visual necesita un modelo que sepa dictaminar, que es justo lo que
#     el de S10 no sabe.
#
# El precio es que la fila de regresión ya no describe el modelo que responde las
# fotos. Queda dicho aquí y en el artefacto.
ADAPTADOR_DIAGNOSTICO = "diagnostico" if "diagnostico" in ADAPTADORES else ADAPTADOR_TUTOR
if ADAPTADOR_DIAGNOSTICO == ADAPTADOR_TUTOR:
    print("\nNo está adaptadores/qwen-lora-diagnostico: la vía visual usará "
          f"{ADAPTADOR_TUTOR!r}.")
    print("Es el sistema de la corrida del 24-10, que no emitía el veredicto en 7 de 12.")
else:
    print(f"\nVía visual  -> {ADAPTADOR_DIAGNOSTICO!r} (entrenado para dictaminar)")
    print(f"Vía de texto -> {ADAPTADOR_TUTOR!r} (el de S10, para que la regresión siga "
          "midiendo lo mismo)")

agente_diagnostico = AgenteTutor(
    crear_llm(ADAPTADOR_DIAGNOSTICO, 320), SYSTEM_C5_AGENTE, registro=REGISTRO_M4,
    max_rondas=6, max_pasos=8, verificar=True,
    extraer_contextos=chunks_de_resultado, nombre="tutor diagnóstico")

TRAZAS_M4 = {}


def responder_sin_imagen(pregunta):
    """La vía de texto: el sistema de M3, sin tocar. Es la de la regresión."""
    salida = agente_m3(pregunta)
    TRAZAS_M4[pregunta[:160]] = salida
    return salida.respuesta


def responder_diagnostico(pregunta):
    """La vía visual: el adaptador entrenado para emitir un veredicto."""
    salida = agente_diagnostico(pregunta)
    TRAZAS_M4[pregunta[:160]] = salida
    return salida.respuesta


print(f"\n{len(REGISTRO_M4)} herramientas disponibles para el tutor.")

Adaptador del tutor: agente
Referencia de M3   : C5-FT · entrenado con 13/21

No está adaptadores/qwen-lora-diagnostico: la vía visual usará 'agente'.
Es el sistema de la corrida del 24-10, que no emitía el veredicto en 7 de 12.

16 herramientas disponibles para el tutor.


In [29]:
# El enchufe, que es la pieza que M4 evalúa.
#
# Patrón INPUT: la foto entra, el VLM la convierte en texto y el sistema de M3 lo
# consume como contexto. El sistema de M3 no se modifica: se le antepone `ver()`.
#
# `INSTRUCCION_REVISION` y `pasos_como_texto` vienen del módulo `vision.py`.


REINTENTOS = {}        # qué casos necesitaron una segunda pasada


def sistema_multimodal(pregunta, lectura=None, clave=None):
    """pregunta (+ lo que se leyó de la foto) -> respuesta del tutor.

    Sin `lectura` es exactamente el sistema de M3: la misma función responde los
    21 casos de la regresión, que no llevan imagen. Que sea la MISMA función es lo
    que hace legítima esa fila.

    Con `lectura`, además del prompt nuevo hay **un** reintento acotado: si la
    respuesta no trae una conclusión válida, se le devuelve su propia explicación
    y se le pide solo la línea. Queda registrado en `REINTENTOS`, porque contestar
    en dos pasadas no es lo mismo que contestar en una.
    """
    if lectura is None:
        return responder_sin_imagen(pregunta)
    respuesta = responder_diagnostico(prompt_revision(pregunta, lectura))
    respuesta, reintento = asegurar_diagnostico(respuesta, responder_diagnostico)
    REINTENTOS[clave or pregunta[:60]] = reintento
    return respuesta


print("sistema_multimodal() listo.")
print("  con imagen -> ver() + M3 + un reintento si falta la conclusión")
print("  sin imagen -> M3 tal cual")
print("\nEl prompt completo que recibe el tutor (con una hoja de ejemplo):\n")
print("-" * 78)
print(prompt_revision("Revisa mi procedimiento: ¿en qué paso me equivoqué?",
                      {"ok": True, "consistente": True,
                       "json": {"enunciado": "Una caja trae 24 lápices...",
                                "pasos": ["24 × 13.", "24 × 13 = 302."],
                                "respuesta_final": "302"}}))
print("-" * 78)
print("\nLo ÚLTIMO que lee antes de generar es la exigencia del formato. En la")
print("corrida del 22-10 lo último era la respuesta final del estudiante, y ocho de")
print("once respuestas terminaron copiando esa etiqueta en vez de concluir.")

sistema_multimodal() listo.
  con imagen -> ver() + M3 + un reintento si falta la conclusión
  sin imagen -> M3 tal cual

El prompt completo que recibe el tutor (con una hoja de ejemplo):

------------------------------------------------------------------------------
Eres un tutor de matemáticas. Un estudiante te muestra el procedimiento que escribió a mano, ya transcrito. Tu trabajo NO es resolver el ejercicio ni reescribirlo: es revisarlo y dar un veredicto.
Reglas:
- Señala UN solo paso, el primero que esté mal. Si los siguientes arrastran ese error, no los cuentes aparte.
- Si el procedimiento está correcto, dilo. No inventes un error para tener algo que decir.
- No cambies los números del estudiante por los correctos: importa dónde se equivocó, no cuál era el resultado bueno.
- La respuesta final escrita cuenta como un paso más, el último.
- NO escribas 'Respuesta final': eso es lo que escribió el estudiante, no tu veredicto.

La última línea de tu respuesta tiene que verse exacta

---
## 7 · Evaluación de los casos con imagen

### 7.1 · Ensayo de formato sobre los casos de desarrollo

Primero los 3 casos `dev-*`, con transcripción sintética. Mide **solo si el
formato se sostiene**, y no entra en ningún scorecard: es la forma de decidir
sobre el prompt sin convertir los 12 casos de evaluación en conjunto de
entrenamiento.

In [30]:
# ENSAYO sobre los 3 casos de DESARROLLO · antes de tocar los 12 de evaluación.
#
# Para qué: la pregunta "¿el tutor emite la línea `Diagnóstico:`?" es de formato,
# y responderla mirando los 12 casos de evaluación es el camino directo al
# sobreajuste: se acaba retocando el prompt hasta que esos 12 salgan bien.
#
# Los casos `dev-*` existen para esto. No tienen foto, así que la transcripción se
# construye a partir de los pasos que el propio eval set declara: no se mide la
# lectura, se mide **si el formato se sostiene**. Son 3 casos: es una señal, no una
# medición, y por eso no entra en ningún scorecard.
def lectura_sintetica(caso):
    """La transcripción que el VLM habría producido si hubiera leído perfecto."""
    return {"ok": True, "consistente": True, "crudo": "(sintética)", "segundos": 0.0,
            "json": {"enunciado": caso["enunciado"], "pasos": list(caso["pasos"]),
                     "respuesta_final": caso["valor_final_escrito"]}}


print("Ensayo de formato sobre los casos de desarrollo (no entra en el scorecard)\n")
conclusiones = []
for caso in dev_m4:
    respuesta = sistema_multimodal(caso["pregunta"], lectura_sintetica(caso),
                                   clave=caso["id"])
    ev = calificar_m4(caso, respuesta)
    conclusiones.append((caso["id"], ev["conclusion"]))
    print(f"  {caso['id']:8s} {caso['bloque']:13s} esperaba {ev['esperado']!r:18s} "
          f"concluyó {str(ev['conclusion'])!r:18s} "
          f"{'(reintento)' if REINTENTOS.get(caso['id']) else ''}")

PUERTA_DEV = revisar_puerta_dev(conclusiones)
print(f"\nConclusión válida en {PUERTA_DEV['validos']}/{PUERTA_DEV['total']} "
      "casos de desarrollo.\n")

# Y ahora la puerta DETIENE. En la corrida del 24-10 esto era un `print`: dio 0/3,
# dijo "PAREN AQUÍ" y la ejecución siguió hasta evaluar los 12 casos.
exigir_puerta_dev(PUERTA_DEV, permitir_sin_puerta=globals().get("PERMITIR_SIN_PUERTA", False))
print("El formato se sostiene en desarrollo. Sigan a los 12 casos de evaluación.")

Ensayo de formato sobre los casos de desarrollo (no entra en el scorecard)

  dev-01   localizacion  esperaba 'paso 2'           concluyó 'paso 2'           
  dev-02   sin_error     esperaba 'sin error'        concluyó 'paso 2'           
  dev-03   compensado    esperaba 'paso 2'           concluyó 'paso 2'           

Conclusión válida en 3/3 casos de desarrollo.

El formato se sostiene en desarrollo. Sigan a los 12 casos de evaluación.


### 7.2 · Los 12 casos de evaluación

In [31]:
# Cómo se obtiene el resultado de cada caso.
#
# El tutor emite una última línea `Diagnóstico: ...` y **es esa línea la que se
# califica**. La explicación se conserva y se mide aparte.
#
# Por qué: en la corrida del 22-10 el criterio buscaba los textos prohibidos como
# subcadena de TODA la respuesta. Como la respuesta empieza repasando el
# procedimiento ("Paso 1: Correcto. Paso 2: Incorrecto..."), los doce casos
# fallaron con "cayó en la trampa: 'paso 1'". `vis-05` llegó a concluir el paso
# correcto y se contó como fallo igualmente. Eso medía la redacción, no el
# criterio del tutor.
#
# Se siguen registrando tres señales independientes, como en M2 y M3, más dos
# sobre la explicación que son diagnóstico y nunca deciden el acierto.

# La puerta, otra vez y aquí: así la parada no depende de cómo el entorno trate
# una excepción en otra celda. Si alguien ejecuta esta celda directamente sin
# haber pasado el ensayo, se detiene igual.
exigir_puerta_dev(globals().get("PUERTA_DEV"),
                  permitir_sin_puerta=globals().get("PERMITIR_SIN_PUERTA", False))

filas = []
for caso in eval_m4:
    logico = archivo_de(caso)
    if PEDIDAS[logico]["archivo"] is None:
        filas.append({"id": caso["id"], "bloque": caso["bloque"],
                      "criterio": caso["criterio_s13"], "estado": "sin_imagen",
                      "esperado": diagnostico_esperado(caso),
                      "acierto": None, "motivo": f"falta la foto de {logico}"})
        print(f"  {caso['id']:8s} {caso['bloque']:13s} ---  sin imagen")
        continue

    lectura = lecturas.get(logico)
    t0 = time.perf_counter()
    respuesta = sistema_multimodal(caso["pregunta"], lectura, clave=caso["id"])
    seg_tutor = time.perf_counter() - t0

    ev = calificar_m4(caso, respuesta)
    fiel = lectura_fiel(caso, lectura)
    filas.append({
        "id": caso["id"], "bloque": caso["bloque"], "criterio": caso["criterio_s13"],
        "estado": "evaluado",
        "esperado": ev["esperado"], "conclusion": ev["conclusion"],
        "acierto": ev["acierto"], "abstuvo": ev["abstuvo"], "abstuvo_bien": ev["abstuvo_bien"],
        "alucino": ev["alucino"], "sin_conclusion": ev["sin_conclusion"],
        "conclusion_invalida": ev["conclusion_invalida"],
        "reintento": bool(REINTENTOS.get(caso["id"])),
        "lectura_no_fiable": lectura_no_fiable(lectura)[0],
        "motivo": ev["motivo"],
        # La explicación, medida aparte: nunca decide el acierto.
        "explicacion_pasos": " ".join(map(str, ev["explicacion_pasos"])),
        "explicacion_justifica": ev["explicacion_justifica"],
        "lectura_fiel": fiel,
        "fuente_del_fallo": fuente_del_fallo(ev["acierto"], fiel),
        "segundos_lectura": (lectura or {}).get("segundos", 0.0),
        "segundos_tutor": round(seg_tutor, 2),
        "segundos_total": round(seg_tutor + (lectura or {}).get("segundos", 0.0), 2),
        "respuesta": respuesta[:700],
    })
    marca = "OK " if ev["acierto"] else ("abs" if ev["abstuvo"] else "MAL")
    print(f"  {caso['id']:8s} {caso['bloque']:13s} {marca}  {ev['motivo'][:56]}")

M4 = pd.DataFrame(filas)
evaluados = M4[M4.estado == "evaluado"]
fallos = evaluados[~evaluados.acierto.astype(bool)]
print(f"\n{len(evaluados)} de {len(eval_m4)} casos evaluados "
      f"({len(M4) - len(evaluados)} sin imagen, fuera de toda media).")

  vis-01   localizacion  OK   concluyó 'paso 2'
  vis-02   localizacion  MAL  esperaba 'paso 1' y concluyó 'paso 2'
  vis-03   localizacion  MAL  esperaba 'paso 3' y concluyó 'paso 2'
  vis-04   localizacion  OK   concluyó 'paso 2'
  vis-05   compensado    OK   concluyó 'paso 2'
  vis-06   compensado    OK   concluyó 'paso 2'
  vis-07   sin_error     MAL  esperaba 'sin error' y concluyó 'paso 2'
  vis-08   sin_error     MAL  esperaba 'sin error' y concluyó 'paso 2'
  vis-09   sin_error     MAL  esperaba 'sin error' y concluyó 'paso 2'
  vis-10   adversarial   MAL  esperaba 'ilegible' y concluyó 'paso 2'
  vis-11   adversarial   MAL  no emitió la línea 'Diagnóstico:'
  vis-12   adversarial   MAL  esperaba 'dato no disponible' y concluyó 'paso 2'

12 de 12 casos evaluados (0 sin imagen, fuera de toda media).


In [32]:
from pathlib import Path

# Scorecard de M4, por bloque. Nunca un promedio global de los cuatro: miden
# cosas distintas y la media no significaría nada.
QUE_MIDE = {
    "localizacion": "nombrar el paso equivocado",
    "compensado":   "paso mal con el resultado final bien",
    "sin_error":    "NO inventar un error donde no lo hay",
    "adversarial":  "abstenerse: ilegible, fuera de dominio, dato ausente",
}


def bloque(nombre):
    g = evaluados[evaluados.bloque == nombre]
    return len(g), int(g.acierto.sum()), (float(g.segundos_total.mean()) if len(g) else 0.0)


print("=" * 78)
print("SCORECARD M4 · casos CON IMAGEN · tarea nueva: localizar el paso erróneo")
print("Se califica la línea `Diagnóstico:`; la explicación se mide aparte.")
print("=" * 78)
print(f"{'bloque':15s}{'aciertos':>11s}{'seg/caso':>11s}   qué mide")
for b, que in QUE_MIDE.items():
    n, ok, seg = bloque(b)
    if n:
        print(f"{b:15s}{ok:>7d}/{n:<3d}{seg:>11.1f}   {que}")
    else:
        print(f"{b:15s}{'—':>11s}{'—':>11s}   {que}  (sin fotos)")
print("-" * 78)
if len(evaluados):
    print(f"{'TOTAL':15s}{int(evaluados.acierto.sum()):>7d}/{len(evaluados):<3d}"
          f"{evaluados.segundos_total.mean():>11.1f}")
    print(f"{'alucinaciones':15s}{int(evaluados.alucino.sum()):>11d}   "
          "concluyó algo concreto y equivocado")
    print(f"{'abstenciones':15s}{int(evaluados.abstuvo.sum()):>11d}   "
          f"de las cuales acertadas: {int(evaluados.abstuvo_bien.sum())}")
    print(f"{'sin conclusión':15s}{int(evaluados.sin_conclusion.sum()):>11d}   "
          "no emitió la línea `Diagnóstico:`")
    print(f"{'no válida':15s}{int(evaluados.conclusion_invalida.sum()):>11d}   "
          "la emitió en texto libre, fuera de las cinco categorías")
    print(f"{'con reintento':15s}{int(evaluados.reintento.sum()):>11d}   "
          "necesitaron una segunda pasada para concluir")
    print(f"{'lectura dudosa':15s}{int(evaluados.lectura_no_fiable.sum()):>11d}   "
          "el tutor recibió 'no se pudo leer', no un procedimiento")
print("=" * 78)

# El fallo de formato se reporta aparte porque tiene arreglo distinto: si domina,
# el problema es que la LoRA de M1 arrastra al modelo a continuar el procedimiento
# en vez de emitir un veredicto, y se ataca con el prompt o con entrenamiento, no
# con un modelo mayor.
sin_forma = int((evaluados.sin_conclusion | evaluados.conclusion_invalida).sum())
if sin_forma:
    print(f"\nAVISO: {sin_forma} de {len(evaluados)} respuestas no dan una conclusión")
    print("válida ni después del reintento. Mientras eso pase, el total mide en parte el")
    print("formato de salida y no el criterio del tutor: miren una respuesta completa")
    print("antes de leer el scorecard.")

# Un predictor constante acierta por construcción los casos cuya respuesta es la
# que repite siempre. Si eso pasa, el total no es evidencia de capacidad.
conc = evaluados[evaluados.conclusion.notna()].conclusion
if len(conc):
    top, veces = conc.value_counts().index[0], conc.value_counts().iloc[0]
    if veces >= 0.6 * len(conc):
        print(f"\nAVISO: {veces} de {len(conc)} conclusiones son {top!r}.")
        print("El tutor está respondiendo casi siempre lo mismo, así que los aciertos")
        print("coinciden con los casos cuya respuesta es justamente esa. Cuenten cuántos")
        print("acertaría un sistema que contestara siempre eso antes de leer el total:")
        baseline = int((evaluados.esperado == top).sum())
        print(f"  un predictor constante {top!r} acertaría {baseline}/{len(evaluados)}.")

print("\nFUENTE DEL FALLO · solo los casos fallados")
if len(fallos):
    for fuente, n in fallos.fuente_del_fallo.value_counts().items():
        ids = ", ".join(fallos[fallos.fuente_del_fallo == fuente].id)
        print(f"  {fuente:12s} {n:2d}   {ids}")
    print("\n  lectura     -> el VLM no transcribió lo que la hoja decía: el cuello es ver")
    print("  diagnostico -> la transcripción llegó bien y el tutor falló igual")
else:
    print("  (sin fallos)")

print("\nLA EXPLICACIÓN, medida aparte · no decide ningún acierto")
coh = evaluados.explicacion_justifica.astype(bool)
print(f"  argumentan su conclusión : {int(coh.sum())}/{len(evaluados)}")
print("    (la explicación habla del paso que concluyó, o no menciona ninguno)")
incoh = evaluados[~coh]
if len(incoh):
    print("  conclusiones sin argumentar en el texto:")
    for _, f in incoh.iterrows():
        print(f"    {f['id']:8s} concluyó {str(f['conclusion']):16s} "
              f"y la explicación habla de los pasos [{f['explicacion_pasos']}]")

Path("m4").mkdir(exist_ok=True)
M4.to_csv("m4/scorecard_m4.csv", index=False)
print("\nGuardado: m4/scorecard_m4.csv")

SCORECARD M4 · casos CON IMAGEN · tarea nueva: localizar el paso erróneo
Se califica la línea `Diagnóstico:`; la explicación se mide aparte.
bloque            aciertos   seg/caso   qué mide
localizacion         2/4         11.4   nombrar el paso equivocado
compensado           2/2          9.4   paso mal con el resultado final bien
sin_error            0/3         10.9   NO inventar un error donde no lo hay
adversarial          0/3         15.1   abstenerse: ilegible, fuera de dominio, dato ausente
------------------------------------------------------------------------------
TOTAL                4/12        11.8
alucinaciones            7   concluyó algo concreto y equivocado
abstenciones             0   de las cuales acertadas: 0
sin conclusión           1   no emitió la línea `Diagnóstico:`
no válida                0   la emitió en texto libre, fuera de las cinco categorías
con reintento            1   necesitaron una segunda pasada para concluir
lectura dudosa           0   el tuto

---
## 8 · Regresión sobre los 21 casos de M2 · sin imagen

Otra pregunta, no la otra mitad de la anterior. Se corre el **mismo**
`sistema_multimodal`, sin imagen, sobre el eval set que ya midió M1, M2, S07, S08
y S10.

### Dimensión 2 · LLM-as-a-judge

Un LLM lee la respuesta y la califica con una rúbrica 1–5. Da lo que ninguna
métrica automática da —juicio sobre si la explicación *enseña*— a escala de
máquina.

**La rúbrica es el producto, no el código.** Está versionada en la celda de
abajo y adaptada al dominio con dos reglas que no están en la plantilla de
clase:

1. **El resultado pesa más que la forma.** "Un procedimiento elegante con
   resultado equivocado no puede pasar de 3." Sin esa ancla, un juez pequeño
   premia la prosa bien estructurada aunque el número esté mal — que es
   exactamente el modo de fallo de nuestro modelo según M1.
2. **La longitud no es calidad.** Es la mitigación del sesgo de longitud (S06)
   escrita dentro de la propia rúbrica.

Usamos el modo **pointwise** para el scorecard (una nota por respuesta) y el
modo **pairwise** solo para el test de sesgo de posición.

> **Advertencia honesta, y vale para todo el notebook:** el juez es
> `Qwen2.5-1.5B-Instruct`, el mismo modelo base que estamos evaluando. Eso lo
> expone al **sesgo de auto-preferencia** documentado en S06. Lo declaramos
> aquí y lo cuantificamos en la sección de calibración: en nuestro dominio, a
> diferencia de casi cualquier otro, podemos comprobar si el juez tiene razón.

In [33]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

JUEZ_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # si va lento: "Qwen/Qwen2.5-0.5B-Instruct"

juez_tok = AutoTokenizer.from_pretrained(JUEZ_MODEL)
juez_model = AutoModelForCausalLM.from_pretrained(
    JUEZ_MODEL, torch_dtype="auto").to(DEVICE).eval()

# ---------------------------------------------------------------------------
# LA RÚBRICA ES PARTE DE LA ENTREGA. Se versiona aquí, no se improvisa.
# Anclas explícitas por nivel (S06: "ancla cada nivel 1-5 con una descripción
# para que no juzgue a ojo") y control de longitud incorporado en el texto
# (S06: "pide concisión en la rúbrica").
# ---------------------------------------------------------------------------
RUBRICA = """Eres un evaluador de un tutor de matemáticas. Califica la RESPUESTA:

5 = El resultado final es CORRECTO y el procedimiento está bien explicado paso a paso.
4 = El resultado final es CORRECTO, pero la explicación omite un paso o tiene un detalle menor mejorable.
3 = El procedimiento es razonable pero el RESULTADO FINAL ES INCORRECTO, o no hay respuesta final.
2 = Hay errores conceptuales graves en el procedimiento.
1 = Incorrecta, inventada, o no responde lo que se preguntó.

Reglas:
- Lo que más pesa es que el RESULTADO sea correcto. Un procedimiento elegante con
  resultado equivocado no puede pasar de 3.
- La longitud no es calidad: una explicación más larga NO merece mejor nota.
- Si la pregunta no tiene respuesta posible (datos insuficientes, división entre
  cero, premisa falsa), la respuesta correcta es decirlo; inventar un número es 1."""


def _extraer_puntaje(texto):
    m = re.search(r"[1-5]", texto)
    return int(m.group()) if m else 3      # fallback neutro


def _generar_juez(system, user, max_new_tokens=6):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    prompt = juez_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = juez_tok(prompt, return_tensors="pt").to(juez_model.device)
    with torch.no_grad():
        out = juez_model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                                  pad_token_id=juez_tok.eos_token_id)
    return juez_tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)


def juez_puntua(pregunta, respuesta, esperada=None):
    """Modo POINTWISE: una nota de 1 a 5 para una respuesta. Es el que usa el harness."""
    ref = f"\n\nRespuesta de referencia (guía, no literal): {esperada}" if esperada else ""
    user = (f"{RUBRICA}\n\nPregunta: {pregunta}\nRespuesta a evaluar: {respuesta}{ref}\n\n"
            "Responde SOLO con un dígito del 1 al 5. Sin explicación.")
    return _extraer_puntaje(_generar_juez("Eres un evaluador estricto y objetivo.", user))


def juez_compara(pregunta, A, B):
    """Modo PAIRWISE: elige la mejor de dos. Se usa para cazar el sesgo de posición."""
    user = (f"Pregunta: {pregunta}\n\nRespuesta A: {A}\n\nRespuesta B: {B}\n\n"
            "¿Cuál respuesta es mejor? Responde SOLO con la letra A o B.")
    texto = _generar_juez("Eres un evaluador estricto y objetivo.", user, max_new_tokens=3).upper()
    m = re.search(r"[AB]", texto)
    return m.group() if m else "?"


def comparar_robusto(pregunta, X, Y):
    """Mitigación del sesgo de posición: evalúa en los dos órdenes y solo declara
    ganador si el veredicto coincide al invertir. Si se contradice, es empate."""
    v1 = juez_compara(pregunta, X, Y)   # X en la posición A
    v2 = juez_compara(pregunta, Y, X)   # X en la posición B
    if v1 == "A" and v2 == "B":
        return "X"
    if v1 == "B" and v2 == "A":
        return "Y"
    return "empate"

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

---
## El harness: las tres dimensiones en una función

`harness(eval_set, sistema)` recibe el eval set y **cualquier** función
`pregunta -> respuesta`, corre las tres dimensiones sobre cada caso y devuelve
el scorecard.

Que la firma sea tan genérica es deliberado: el mismo harness evaluará el
modelo de M1, el RAG de M3 y lo que venga después. Es la vara fija del
semestre, y solo sirve como vara si no cambia.

El scorecard desglosa los aciertos **por bloque**, porque el promedio global
mezcla tres preguntas distintas: ¿sabe calcular?, ¿conoce los datos de la
institución?, ¿es robusto ante trampas? Un sistema puede ser excelente en una y
pésimo en otra, y un número único lo ocultaría.

In [34]:
import numpy as np


def harness(eval_set, sistema, nombre="sistema", verbose=True):
    """Las tres dimensiones sobre un sistema. `sistema` es cualquier función
    `pregunta -> respuesta`: el modelo de M1, el RAG de M3, lo que sea.

    Devuelve el scorecard: promedios por dimensión + el detalle caso por caso.
    """
    detalle = []
    for i, caso in enumerate(eval_set, 1):
        respuesta = sistema(caso["input"])

        sim = sim_embeddings(respuesta, caso["esperado"])        # Dimensión 1b
        pj = juez_puntua(caso["input"], respuesta, caso["esperado"])  # Dimensión 2
        ev = evaluar_caso(caso, respuesta)                       # Dimensión 3

        detalle.append({
            "id": caso["id"], "bloque": caso["bloque"], "subtipo": caso["subtipo"],
            "input": caso["input"], "respuesta": respuesta,
            "sim": round(sim, 3), "juez": pj,
            "acierto": ev["acierto"], "abstuvo": ev["abstuvo"], "alucino": ev["alucino"],
            "motivo": ev["motivo"],
            "formato_valido": formato_valido(respuesta),
            "palabras": len(respuesta.split()),
        })
        if verbose:
            marca = "OK " if ev["acierto"] else ("abs" if ev["abstuvo"] else "MAL")
            print(f"  [{i:2d}/{len(eval_set)}] {caso['id']:8s} {marca}  juez={pj}  sim={sim:.2f}  {ev['motivo'][:46]}")

    def prom(clave, filas=None):
        filas = filas if filas is not None else detalle
        return float(np.mean([d[clave] for d in filas])) if filas else 0.0

    calculo = [d for d in detalle if d["bloque"] == "calculo"]
    conocim = [d for d in detalle if d["bloque"] == "conocimiento"]
    advers  = [d for d in detalle if d["bloque"] == "adversarial"]

    return {
        "nombre": nombre,
        "n": len(detalle),
        # Dimensión 1
        "exactitud_calculo": prom("acierto", calculo),
        "sim_embeddings": prom("sim"),
        # Dimensión 2
        "juez_promedio": prom("juez"),
        # Dimensión 3
        "aciertos": sum(d["acierto"] for d in detalle),
        "aciertos_calculo": sum(d["acierto"] for d in calculo),
        "aciertos_conocimiento": sum(d["acierto"] for d in conocim),
        "aciertos_adversarial": sum(d["acierto"] for d in advers),
        "n_calculo": len(calculo), "n_conocimiento": len(conocim), "n_adversarial": len(advers),
        # Diagnóstico
        "alucinaciones": sum(d["alucino"] for d in detalle),
        "abstenciones": sum(d["abstuvo"] for d in detalle),
        "formato_valido": prom("formato_valido"),
        "palabras_media": prom("palabras"),
        "detalle": detalle,
    }


def imprimir_scorecard(*scorecards):
    """El scorecard: una tabla con el puntaje por dimensión. Acepta 1 o más
    sistemas para compararlos lado a lado sobre el mismo eval set."""
    nombres = [s["nombre"] for s in scorecards]
    ancho = 40
    total = ancho + 14 * len(scorecards)

    filas = [
        ("DIMENSIÓN 1 · MÉTRICA CLÁSICA", None, None),
        ("  1a · Exactitud en cálculo", lambda s: f"{s['exactitud_calculo']:.1%}", None),
        ("  1b · Similitud embeddings (0-1)", lambda s: f"{s['sim_embeddings']:.3f}", None),
        ("DIMENSIÓN 2 · LLM-AS-A-JUDGE", None, None),
        ("  2 · Nota media (1-5)", lambda s: f"{s['juez_promedio']:.2f}", None),
        ("DIMENSIÓN 3 · ACIERTOS DE DOMINIO", None, None),
        ("  3a · Total", lambda s: f"{s['aciertos']}/{s['n']}", None),
        ("  3b · Bloque cálculo", lambda s: f"{s['aciertos_calculo']}/{s['n_calculo']}", None),
        ("  3c · Bloque conocimiento", lambda s: f"{s['aciertos_conocimiento']}/{s['n_conocimiento']}", None),
        ("  3d · Bloque adversarial", lambda s: f"{s['aciertos_adversarial']}/{s['n_adversarial']}", None),
        ("DIAGNÓSTICO", None, None),
        ("  Alucinaciones (menos es mejor)", lambda s: str(s["alucinaciones"]), None),
        ("  Abstenciones honestas", lambda s: str(s["abstenciones"]), None),
        ("  Formato válido", lambda s: f"{s['formato_valido']:.0%}", None),
        ("  Palabras por respuesta", lambda s: f"{s['palabras_media']:.0f}", None),
    ]

    print("=" * total)
    print(f"{'':<{ancho}}" + "".join(f"{n[:13]:>14}" for n in nombres))
    print("-" * total)
    for etiqueta, fn, _ in filas:
        if fn is None:
            print(etiqueta)
        else:
            print(f"{etiqueta:<{ancho}}" + "".join(f"{fn(s):>14}" for s in scorecards))
    print("=" * total)


def guardar_scorecard(ruta, *scorecards):
    import csv
    nombres = [s["nombre"] for s in scorecards]
    filas = [
        ("exactitud_calculo", lambda s: round(s["exactitud_calculo"], 3)),
        ("sim_embeddings_prom", lambda s: round(s["sim_embeddings"], 3)),
        ("llm_juez_prom", lambda s: round(s["juez_promedio"], 3)),
        ("aciertos_total", lambda s: f"{s['aciertos']}/{s['n']}"),
        ("aciertos_calculo", lambda s: f"{s['aciertos_calculo']}/{s['n_calculo']}"),
        ("aciertos_conocimiento", lambda s: f"{s['aciertos_conocimiento']}/{s['n_conocimiento']}"),
        ("aciertos_adversarial", lambda s: f"{s['aciertos_adversarial']}/{s['n_adversarial']}"),
        ("alucinaciones", lambda s: s["alucinaciones"]),
        ("abstenciones", lambda s: s["abstenciones"]),
        ("formato_valido", lambda s: round(s["formato_valido"], 3)),
        ("palabras_media", lambda s: round(s["palabras_media"], 1)),
    ]
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["dimension"] + nombres)
        for clave, fn in filas:
            w.writerow([clave] + [fn(s) for s in scorecards])
    print(f"Guardado: {ruta}")


def tabla_por_subtipo(*scorecards):
    """Dónde falla el sistema, por tipo de dificultad. Es la tabla accionable:
    dice qué ejemplos añadir al corpus de entrenamiento."""
    nombres = [s["nombre"] for s in scorecards]
    ids = [d["id"] for d in scorecards[0]["detalle"]]
    print(f"{'caso':9s} {'subtipo':28s}" + "".join(f"{n[:12]:>13}" for n in nombres))
    print("-" * (37 + 13 * len(scorecards)))
    for i, cid in enumerate(ids):
        sub = scorecards[0]["detalle"][i]["subtipo"]
        celdas = ""
        for s in scorecards:
            d = s["detalle"][i]
            celdas += f"{('OK' if d['acierto'] else ('abst' if d['abstuvo'] else 'FALLA')):>13}"
        print(f"{cid:9s} {sub:28s}{celdas}")

In [ ]:
# REGRESIÓN sobre los 21 casos de M2 · SIN imagen.
#
# Esto NO es la otra mitad de la medición anterior: es otra pregunta. El eval set
# de M2 mide la tarea de siempre (resolver un problema planteado en texto) y aquí
# se vuelve a correr con el MISMO sistema, solo para comprobar que anteponer
# `ver()` no rompió nada. Un cambio aquí es una regresión; un número de M4 y uno
# de M2 no se restan.
sc_regresion = harness(eval_set, lambda p: sistema_multimodal(p, None),
                       nombre="M4 · sin imagen")
imprimir_scorecard(sc_regresion)

if EVAL_M2_COMPARABLE:
    print("\nMismo eval set de 21 casos, dos corridas del mismo sistema:")
    print(f"  en M3 · {REFERENCIA_M3['nombre']:22s} {REFERENCIA_M3['aciertos']}/21")
    print(f"  ahora · M4                         {sc_regresion['aciertos']}/21")
    print("\nLectura: si coinciden, la visión no tocó lo que ya funcionaba. Si bajó, hay")
    print("que decir en qué caso y por qué ANTES de hablar de lo que se ganó en imagen.")
    print("Con n = 21, una diferencia de un caso es ruido; de tres o más, hay que mirarla.")
else:
    print(f"\nM4 sin imagen: {sc_regresion['aciertos']}/21  —  NO COMPARABLE con M3.")
    print("El eval set en disco no es el que midió M3 (ver la guarda de la sección 3.2).")
    print("Esta cifra sirve para ver si el sistema responde, no para comparar con nada.")

  [ 1/21] dif-01   OK   juez=3  sim=0.83  número correcto
  [ 2/21] dif-02   OK   juez=4  sim=0.89  número correcto
  [ 3/21] dif-03   MAL  juez=3  sim=0.92  esperaba 1600, dio 250000
  [ 4/21] dif-04   OK   juez=3  sim=0.90  número correcto
  [ 5/21] dif-05   OK   juez=3  sim=0.84  número correcto
  [ 6/21] dif-06   OK   juez=3  sim=0.92  número correcto
  [ 7/21] dif-07   MAL  juez=3  sim=0.97  esperaba 60, dio 180
  [ 8/21] dif-08   MAL  juez=3  sim=0.84  esperaba 8, dio 16
  [ 9/21] dif-09   OK   juez=3  sim=0.96  número correcto
  [10/21] dif-10   MAL  juez=3  sim=0.76  esperaba 80000, dio 78200
  [11/21] dif-11   OK   juez=5  sim=0.79  número correcto
  [12/21] dif-12   OK   juez=3  sim=0.87  número correcto
  [13/21] con-01   MAL  juez=3  sim=0.62  esperaba 40, dio 50
  [14/21] con-02   OK   juez=3  sim=0.76  contiene 'octavo'
  [15/21] con-03   OK   juez=3  sim=0.82  contiene '3.0'
  [16/21] con-04   OK   juez=3  sim=0.74  contiene 'factor'
  [17/21] adv-01   OK   juez=3  sim=0

---
## 9 · La tabla de delta

In [ ]:
# La tabla que pide la rúbrica: M3 contra M4, con las dos partes separadas y
# etiquetadas, y la latencia al lado.
adv = evaluados[evaluados.bloque == "adversarial"]
con_img = evaluados[evaluados.bloque != "adversarial"]
peor = (fallos.iloc[0]["id"] if len(fallos) else "sin fallos")

DELTA = pd.DataFrame([
    {"grupo": "casos sin imagen · eval set de M2 (21 casos)",
     "M3": (f"{REFERENCIA_M3['aciertos']}/21" if EVAL_M2_COMPARABLE
            else "no comparable: otro eval set en disco"),
     "M4": f"{sc_regresion['aciertos']}/21",
     "qué significa": ("REGRESIÓN: ¿se rompió lo que ya funcionaba?" if EVAL_M2_COMPARABLE
                       else "NO comparable: el eval set en disco no es el de M3")},
    {"grupo": "casos con imagen · eval set de M4 (tarea nueva)",
     "M3": "no aplica: M3 no recibe imágenes",
     "M4": f"{int(con_img.acierto.sum())}/{len(con_img)}",
     "qué significa": "CAPACIDAD NUEVA: no hay baseline comparable"},
    {"grupo": "adversariales visuales",
     "M3": "no aplica",
     "M4": f"{int(adv.acierto.sum())}/{len(adv)}",
     "qué significa": "¿se negó cuando la hoja era ilegible o no era matemáticas?"},
    {"grupo": "latencia por caso",
     "M3": f"{REFERENCIA_M3['segundos']:.1f} s",
     "M4": f"{evaluados.segundos_total.mean():.1f} s "
           f"({evaluados.segundos_lectura.mean():.1f} s son del VLM)",
     "qué significa": "el precio de ver"},
    {"grupo": "peor caso",
     "M3": "—", "M4": peor,
     "qué significa": "se explica en m4/README.md"},
])
DELTA.to_csv("m4/delta_m3_m4.csv", index=False)
print(DELTA.to_string(index=False))

print("\nLas dos primeras filas son CONJUNTOS DISTINTOS. La primera dice si el sistema")
print("sigue respondiendo igual de bien lo de siempre; la segunda, qué tan bien hace algo")
print("que antes no podía hacer. No se restan, y no existe 'la mejora de M3 a M4' como un")
print("número único.")
print("\nGuardado: m4/delta_m3_m4.csv")

                                          grupo                               M3                         M4                                              qué significa
   casos sin imagen · eval set de M2 (21 casos)                            13/21                      15/21                REGRESIÓN: ¿se rompió lo que ya funcionaba?
casos con imagen · eval set de M4 (tarea nueva) no aplica: M3 no recibe imágenes                        4/9                CAPACIDAD NUEVA: no hay baseline comparable
                         adversariales visuales                        no aplica                        0/3 ¿se negó cuando la hoja era ilegible o no era matemáticas?
                              latencia por caso                           14.1 s 11.8 s (3.9 s son del VLM)                                           el precio de ver
                                      peor caso                                —                     vis-02                                 se explica en m4/README.m

---
## 10 · Artefactos

In [ ]:
json.dump({
    "patron": "input",
    "vlm": VLM_ID,
    "max_pixels": MAX_PIXELS,
    "lado_mayor_px": LADO_MAYOR,
    "adaptador_tutor": ADAPTADOR_TUTOR,
    "adaptador_via_visual": ADAPTADOR_DIAGNOSTICO,
    "referencia_m3": REFERENCIA_M3,
    "n_casos": len(eval_m4),
    "n_imagenes_necesarias": len(PEDIDAS),
    "imagenes_faltantes": sorted(FALTAN),
    "n_evaluados": int(len(evaluados)),
    "visual_por_bloque": {b: {"n": bloque(b)[0], "aciertos": bloque(b)[1]} for b in QUE_MIDE},
    "visual_total": {"n": int(len(evaluados)),
                     "aciertos": int(evaluados.acierto.sum()),
                     "alucinaciones": int(evaluados.alucino.sum()),
                     "abstenciones": int(evaluados.abstuvo.sum())},
    "fuente_del_fallo": (fallos.fuente_del_fallo.value_counts().to_dict() if len(fallos) else {}),
    "eval_m2_comparable_con_m3": bool(EVAL_M2_COMPARABLE),
    "regresion_m2": {k: v for k, v in sc_regresion.items() if k != "detalle"},
    "conclusiones": {str(k): int(v) for k, v in
                     evaluados.conclusion.fillna("sin_conclusion").value_counts().items()},
    "puerta_dev": ("superada" if (globals().get("PUERTA_DEV") or {}).get("ok")
                   else "NO superada"),
    "sin_conclusion": int(evaluados.sin_conclusion.sum()),
    "conclusion_invalida": int(evaluados.conclusion_invalida.sum()),
    "reintentos": int(evaluados.reintento.sum()),
    "lecturas_no_fiables": sorted(evaluados[evaluados.lectura_no_fiable].id),
    "abstuvo_bien": int(evaluados.abstuvo_bien.sum()),
    "explicacion_justifica": int(evaluados.explicacion_justifica.astype(bool).sum()),
    "archivos_leidos": {k: v.get("archivo", k) for k, v in lecturas.items()},
    "latencia_total_s": round(float(evaluados.segundos_total.mean()), 2),
    "latencia_lectura_s": round(float(evaluados.segundos_lectura.mean()), 2),
}, open("m4/resultados_m4.json", "w", encoding="utf-8"),
    ensure_ascii=False, indent=2, default=float)

pd.DataFrame(sc_regresion["detalle"]).to_csv("m4/detalle_regresion_m4.csv", index=False)

for p in ("m4/scorecard_m4.csv", "m4/delta_m3_m4.csv", "m4/detalle_regresion_m4.csv",
          "m4/resultados_m4.json", "m4/lecturas_m4.json"):
    print(" ", p)
print("\nFalta m4/README.md: se escribe con estos números a la vista, no antes.")
print("Si quedó alguna imagen sin subir, el scorecard está incompleto y hay que decirlo.")

  m4/scorecard_m4.csv
  m4/delta_m3_m4.csv
  m4/detalle_regresion_m4.csv
  m4/resultados_m4.json
  m4/lecturas_m4.json

Falta m4/README.md: se escribe con estos números a la vista, no antes.
Si quedó alguna imagen sin subir, el scorecard está incompleto y hay que decirlo.


---
## 11 · Cómo leer esto

**1 · Primero la fuente del fallo, no el total.** Si la mayoría de los fallos son
de `lectura`, el cuello de botella es el VLM y se arregla con preproceso, un OCR
clásico o un modelo mayor, sin tocar el tutor. Si son de `diagnostico`, la
transcripción llegó bien y el límite es el razonamiento del modelo de 1.5B — el
mismo límite que ya salió en M3, ahora con una entrada nueva.

**2 · El bloque `sin_error` antes que el de aciertos.** Un tutor que no encuentra
un error deja al estudiante como estaba; uno que **inventa** un error le dice que
está mal cuando está bien. En un tutor ese es el fallo grave, igual que las
alucinaciones en M2.

**3 · El bloque `compensado` es el que justifica el módulo.** Paso equivocado con
resultado final correcto: un corrector que solo mira el número final los aprueba.
Son 2 casos, así que es una señal, no una medición.

**4 · Las dos primeras filas de la tabla de delta no se restan.** Una dice si el
sistema sigue respondiendo igual de bien lo de siempre; la otra, qué tan bien hace
algo que antes no podía. Con n = 12 y bloques de 2 a 4 casos, ninguna diferencia
pequeña es concluyente, y conviene decirlo antes de que lo pregunten.

**5 · La latencia incluye la pasada del VLM.** Es el precio de ver, y va al lado
de cualquier mejora que se reporte.

**6 · Si faltó alguna foto, el scorecard está incompleto.** Los casos
`sin_imagen` quedan fuera de las medias, y el total hay que leerlo sobre los casos
que sí se evaluaron, no sobre 12.